In [ ]:
from google.colab import auth
auth.authenticate_user()

from googleapiclient.discovery import build

drive_service = build("drive", "v3")

FOLDER_ID = "1jxuW27N8O0P684L22ImILULuKpInYYWu"


def list_folder(folder_id, prefix="", visited=None):
    if visited is None:
        visited = set()

    if folder_id in visited:
        print(prefix + "[Already visited]")
        return

    visited.add(folder_id)

    query = f"'{folder_id}' in parents and trashed = false"

    page_token = None
    all_items = []

    while True:
        response = drive_service.files().list(
            q=query,
            spaces="drive",
            fields="nextPageToken, files(id, name, mimeType, size)",
            pageToken=page_token,
            pageSize=1000,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True
        ).execute()

        all_items.extend(response.get("files", []))
        page_token = response.get("nextPageToken")

        if not page_token:
            break

    all_items.sort(
        key=lambda x: (
            x["mimeType"] != "application/vnd.google-apps.folder",
            x["name"].lower()
        )
    )

    for item in all_items:
        name = item["name"]
        mime_type = item["mimeType"]
        item_id = item["id"]

        if mime_type == "application/vnd.google-apps.folder":
            print(prefix + f"📁 {name}/")
            list_folder(item_id, prefix + "    ", visited)
        else:
            size = int(item.get("size", 0))
            size_mb = size / (1024 * 1024)
            print(prefix + f"📄 {name}  [{size_mb:.2f} MB]")


folder_metadata = drive_service.files().get(
    fileId=FOLDER_ID,
    fields="id, name, mimeType",
    supportsAllDrives=True
).execute()

print(f"Root folder: {folder_metadata['name']}")
print(f"Folder ID: {folder_metadata['id']}")
print()

list_folder(FOLDER_ID)

In [ ]:
# ============================================================
# PITT + WLS FLUENCY PREPROCESSING
# Fresh Google Colab runtime
# ============================================================

!pip -q install xlrd==2.0.1

from google.colab import auth, drive
auth.authenticate_user()
drive.mount("/content/drive", force_remount=False)

import io
import re
import shutil
import unicodedata
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload


# ============================================================
# SETTINGS
# ============================================================

ROOT_FOLDER_ID = "1jxuW27N8O0P684L22ImILULuKpInYYWu"

LOCAL_ROOT = Path("/content/fluency_source")
OUTPUT_DIR = Path("/content/drive/MyDrive/Fluency_Processed")

OVERWRITE_DOWNLOADS = False

FOLDER_MIME = "application/vnd.google-apps.folder"
SHORTCUT_MIME = "application/vnd.google-apps.shortcut"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)


# ============================================================
# GOOGLE DRIVE DOWNLOAD
# ============================================================

drive_service = build("drive", "v3")


def safe_filename(name):
    name = str(name).replace("/", "_").replace("\\", "_").strip()
    return name if name else "unnamed"


def list_drive_children(folder_id):
    page_token = None

    while True:
        response = drive_service.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields=(
                "nextPageToken,"
                "files(id,name,mimeType,size,"
                "shortcutDetails(targetId,targetMimeType))"
            ),
            pageToken=page_token,
            pageSize=1000,
            spaces="drive",
            supportsAllDrives=True,
            includeItemsFromAllDrives=True
        ).execute()

        for item in response.get("files", []):
            yield item

        page_token = response.get("nextPageToken")

        if not page_token:
            break


def download_drive_file(file_id, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.exists() and not OVERWRITE_DOWNLOADS:
        return

    request = drive_service.files().get_media(
        fileId=file_id,
        supportsAllDrives=True
    )

    with io.FileIO(destination, "wb") as handle:
        downloader = MediaIoBaseDownload(handle, request)
        finished = False

        while not finished:
            _, finished = downloader.next_chunk()


def download_drive_tree(folder_id, local_folder):
    local_folder = Path(local_folder)
    local_folder.mkdir(parents=True, exist_ok=True)

    children = sorted(
        list(list_drive_children(folder_id)),
        key=lambda item: item["name"].lower()
    )

    for item in children:
        item_id = item["id"]
        item_name = safe_filename(item["name"])
        mime_type = item["mimeType"]

        if mime_type == SHORTCUT_MIME:
            details = item.get("shortcutDetails", {})
            item_id = details.get("targetId")
            mime_type = details.get("targetMimeType")

            if not item_id:
                print("Skipping broken shortcut:", item_name)
                continue

        if mime_type == FOLDER_MIME:
            download_drive_tree(
                item_id,
                local_folder / item_name
            )
            continue

        extension = Path(item_name).suffix.lower()

        if extension not in {
            ".cha",
            ".xls",
            ".xlsx",
            ".csv",
            ".cdc",
            ".txt"
        }:
            continue

        try:
            download_drive_file(
                item_id,
                local_folder / item_name
            )
        except Exception as error:
            print("Download failed:", item_name, error)


print("Downloading Pitt and WLS source files...")

if OVERWRITE_DOWNLOADS and LOCAL_ROOT.exists():
    shutil.rmtree(LOCAL_ROOT)
    LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

download_drive_tree(
    ROOT_FOLDER_ID,
    LOCAL_ROOT
)

print("Download complete.")
print("Local source folder:", LOCAL_ROOT)
print("Output folder:", OUTPUT_DIR)


# ============================================================
# GENERAL CLEANING
# ============================================================

BULLET_RE = re.compile(r"\x15(\d+)_(\d+)\x15")
CHAT_CODE_RE = re.compile(r"\[[^\]]*\]")
ANGLE_RETRACE_RE = re.compile(r"<([^<>]*)>\s*\[/\]")
EVENT_RE = re.compile(r"&=[A-Za-z_]+")
FRAGMENT_RE = re.compile(r"&[-+][A-Za-z]+")
AT_SUFFIX_RE = re.compile(r"@[\w:]+")
NON_WORD_RE = re.compile(r"[^A-Za-z'\-\s]")
MULTISPACE_RE = re.compile(r"\s+")

FILLERS = {
    "uh",
    "um",
    "er",
    "erm",
    "hm",
    "hmm",
    "mhm",
    "mm",
    "okay",
    "ok",
    "well",
    "yeah",
    "yes",
    "no",
    "and",
    "the",
    "a",
    "an",
    "i",
    "you",
    "it",
    "is",
    "was",
    "are",
    "were",
    "to",
    "of",
    "that",
    "this",
    "let",
    "lets",
    "see",
    "think",
    "know",
    "cant",
    "cannot",
    "dont",
    "do",
    "did",
    "say",
    "said",
    "got",
    "get",
    "have",
    "has",
    "had"
}

TASK_WORDS = {
    "animal",
    "animals",
    "word",
    "words",
    "letter",
    "beginning",
    "starts",
    "start",
    "starting",
    "minute",
    "time"
}


def normalize_token(value):
    value = unicodedata.normalize(
        "NFKC",
        str(value)
    )

    value = value.strip().lower()
    value = value.lstrip("@")
    value = value.replace("’", "'")
    value = re.sub(r"[^a-z'\-]", "", value)
    value = value.strip("-'")

    return value


def clean_chat_text(text):
    text = BULLET_RE.sub(" ", text)
    text = ANGLE_RETRACE_RE.sub(r"\1", text)
    text = CHAT_CODE_RE.sub(" ", text)
    text = EVENT_RE.sub(" ", text)
    text = FRAGMENT_RE.sub(" ", text)
    text = AT_SUFFIX_RE.sub(" ", text)

    text = text.replace("xxx", " ")
    text = text.replace("yyy", " ")
    text = text.replace("(", " ")
    text = text.replace(")", " ")

    text = NON_WORD_RE.sub(" ", text)
    text = MULTISPACE_RE.sub(" ", text).strip()

    return text


def tokenize_text(text):
    tokens = []

    for raw_token in str(text).split():
        token = normalize_token(raw_token)

        if token:
            tokens.append(token)

    return tokens


def extract_candidate_items(text, task):
    tokens = tokenize_text(text)

    blocked = FILLERS | TASK_WORDS

    items = [
        token
        for token in tokens
        if token not in blocked
    ]

    if task == "letter":
        items = [
            token
            for token in items
            if token.startswith("f")
        ]

    return items


def calculate_item_features(items):
    items = [
        normalize_token(item)
        for item in items
    ]

    items = [
        item
        for item in items
        if item
    ]

    counts = Counter(items)
    unique_items = list(dict.fromkeys(items))

    repetitions = sum(
        max(0, count - 1)
        for count in counts.values()
    )

    return {
        "items_text": " ".join(items),
        "total_items": len(items),
        "unique_items": len(unique_items),
        "repetition_count": repetitions,
        "type_token_ratio": (
            len(unique_items) / len(items)
            if items
            else np.nan
        )
    }


# ============================================================
# DIAGNOSIS HANDLING
# ============================================================

def normalize_diagnosis(raw_diagnosis, path):
    cleaned = re.sub(
        r"[^a-z]",
        "",
        str(raw_diagnosis).lower()
    )

    if (
        "probablead" in cleaned
        or "possiblead" in cleaned
        or cleaned == "ad"
        or "alzheimer" in cleaned
    ):
        return "AD", 1

    if "mci" in cleaned:
        return "MCI", np.nan

    if cleaned in {
        "control",
        "healthycontrol",
        "hc",
        "normal"
    }:
        return "Control", 0

    normalized_path = str(path).replace("\\", "/").lower()

    if "/control/" in normalized_path:
        return "Control", 0

    if "/dementia/" in normalized_path:
        return "Other_or_Unknown", np.nan

    return "Other_or_Unknown", np.nan


# ============================================================
# CHAT METADATA
# ============================================================

def parse_chat_id_line(line):
    payload = line.split(":", 1)[1].strip()
    fields = payload.split("|")

    def get_field(index):
        if index < len(fields):
            return fields[index].strip()
        return ""

    return {
        "language": get_field(0),
        "corpus": get_field(1),
        "speaker": get_field(2),
        "age": get_field(3).strip("; "),
        "sex": get_field(4),
        "diagnosis": get_field(5),
        "role": get_field(7),
        "mmse": get_field(8)
    }


# ============================================================
# PITT FLUENCY PARSER
# ============================================================

def parse_pitt_fluency_file(path):
    path = Path(path)

    raw_text = path.read_text(
        encoding="utf-8",
        errors="replace"
    )

    lines = raw_text.splitlines()

    participant_metadata = {}
    media_name = path.stem

    for line in lines:
        if line.startswith("@ID:") and "|PAR|" in line:
            participant_metadata = parse_chat_id_line(line)

        elif line.startswith("@Media:"):
            media_name = (
                line.split(":", 1)[1]
                .split(",", 1)[0]
                .strip()
            )

    source_subject_id = path.stem.split("-", 1)[0]

    visit_match = re.search(
        r"-(\d+)$",
        path.stem
    )

    visit = (
        int(visit_match.group(1))
        if visit_match
        else np.nan
    )

    diagnosis_group, binary_label = normalize_diagnosis(
        participant_metadata.get("diagnosis", ""),
        path
    )

    section_text = {
        "category": [],
        "letter": []
    }

    timed_utterances = {
        "category": [],
        "letter": []
    }

    current_task = None
    pending_participant = None

    def flush_pending():
        nonlocal pending_participant

        if pending_participant is None:
            return

        if current_task not in section_text:
            pending_participant = None
            return

        cleaned_text = clean_chat_text(
            pending_participant["text"]
        )

        if cleaned_text:
            section_text[current_task].append(
                cleaned_text
            )

            items = extract_candidate_items(
                cleaned_text,
                current_task
            )

            if items:
                timed_utterances[current_task].append(
                    (
                        pending_participant["start_ms"],
                        items
                    )
                )

        pending_participant = None

    for line in lines:
        if line.startswith("@G:"):
            flush_pending()

            group_name = (
                line.split(":", 1)[1]
                .strip()
                .lower()
            )

            if "animal" in group_name:
                current_task = "category"

            elif group_name == "f" or group_name.startswith("f "):
                current_task = "letter"

            else:
                current_task = None

            continue

        if line.startswith("*"):
            flush_pending()

            if (
                line.startswith("*PAR:")
                and current_task in section_text
            ):
                payload = line.split(":", 1)[1].strip()

                time_match = BULLET_RE.search(payload)

                start_ms = (
                    int(time_match.group(1))
                    if time_match
                    else None
                )

                pending_participant = {
                    "text": payload,
                    "start_ms": start_ms
                }

            continue

        if (
            pending_participant is not None
            and line.startswith("\t")
        ):
            pending_participant["text"] += (
                " " + line.strip()
            )

    flush_pending()

    rows = []

    for task in ["category", "letter"]:
        if not section_text[task]:
            continue

        response_text = " ".join(
            section_text[task]
        ).strip()

        all_items = []

        for _, utterance_items in timed_utterances[task]:
            all_items.extend(utterance_items)

        features = calculate_item_features(
            all_items
        )

        known_start_times = [
            start_ms
            for start_ms, utterance_items
            in timed_utterances[task]
            if start_ms is not None and utterance_items
        ]

        task_start_ms = (
            min(known_start_times)
            if known_start_times
            else None
        )

        time_bins = [0, 0, 0, 0]

        if task_start_ms is not None:
            for start_ms, utterance_items in timed_utterances[task]:
                if start_ms is None:
                    continue

                relative_seconds = max(
                    0,
                    (start_ms - task_start_ms) / 1000
                )

                bin_index = min(
                    3,
                    int(relative_seconds // 15)
                )

                time_bins[bin_index] += len(
                    utterance_items
                )

        row = {
            "dataset": "Pitt",
            "source_type": "CHAT",
            "task": task,
            "task_prompt": (
                "animals"
                if task == "category"
                else "letter_F"
            ),
            "subject_id": f"Pitt_{source_subject_id}",
            "source_subject_id": source_subject_id,
            "recording_id": (
                f"Pitt_{media_name}_{task}"
            ),
            "source_recording_id": media_name,
            "visit": visit,
            "wave": visit,
            "diagnosis_raw": participant_metadata.get(
                "diagnosis",
                ""
            ),
            "diagnosis_group": diagnosis_group,
            "binary_label": binary_label,
            "age": pd.to_numeric(
                participant_metadata.get("age", ""),
                errors="coerce"
            ),
            "sex": participant_metadata.get(
                "sex",
                ""
            ),
            "mmse": pd.to_numeric(
                participant_metadata.get("mmse", ""),
                errors="coerce"
            ),
            "response_text": response_text,
            "items_text": features["items_text"],
            "total_items": features["total_items"],
            "unique_items": features["unique_items"],
            "repetition_count": features[
                "repetition_count"
            ],
            "type_token_ratio": features[
                "type_token_ratio"
            ],
            "items_0_15s": (
                time_bins[0]
                if task_start_ms is not None
                else np.nan
            ),
            "items_15_30s": (
                time_bins[1]
                if task_start_ms is not None
                else np.nan
            ),
            "items_30_45s": (
                time_bins[2]
                if task_start_ms is not None
                else np.nan
            ),
            "items_45_60s": (
                time_bins[3]
                if task_start_ms is not None
                else np.nan
            ),
            "source_path": str(
                path.relative_to(LOCAL_ROOT)
            )
        }

        rows.append(row)

    return rows


# ============================================================
# WLS SPREADSHEET PARSER
# ============================================================

TIME_BIN_PREFIXES = [
    ("FIR15", "items_0_15s"),
    ("SEC15", "items_15_30s"),
    ("THI15", "items_30_45s"),
    ("FOR15", "items_45_60s")
]


def infer_wls_task(path, columns):
    filename = path.name.lower()
    columns_text = " ".join(
        str(column)
        for column in columns
    ).upper()

    if (
        "cogcat" in filename
        or "FIR15C_" in columns_text
    ):
        return "category", "animals"

    if (
        "coglet" in filename
        or re.search(
            r"\bFIR15_\d+",
            columns_text
        )
    ):
        return "letter", "letter_WLS"

    raise ValueError(
        f"Could not determine task for {path}"
    )


def infer_wls_wave(path):
    match = re.search(
        r"(?:_|-)(03|11)(?:\D|$)",
        path.stem
    )

    if match:
        return 2000 + int(match.group(1))

    return np.nan


def column_number(column_name):
    match = re.search(
        r"_(\d+)$",
        str(column_name)
    )

    if match:
        return int(match.group(1))

    return 999


def parse_wls_spreadsheet(path):
    path = Path(path)

    dataframe = pd.read_excel(path)

    if "idtlkbnk" not in dataframe.columns:
        return []

    task, task_prompt = infer_wls_task(
        path,
        dataframe.columns
    )

    wave = infer_wls_wave(path)

    response_columns = [
        column
        for column in dataframe.columns
        if re.match(
            r"^(FIR15C?|SEC15C?|THI15C?|FOR15C?|FIF15C?)_\d+$",
            str(column)
        )
    ]

    rows = []

    for _, record in dataframe.iterrows():
        raw_subject_id = record.get("idtlkbnk")

        if pd.isna(raw_subject_id):
            continue

        try:
            source_subject_id = str(
                int(raw_subject_id)
            )
        except Exception:
            source_subject_id = str(
                raw_subject_id
            ).strip()

        ordered_items = []
        bin_counts = {}

        for prefix, output_column in TIME_BIN_PREFIXES:
            matching_columns = [
                column
                for column in response_columns
                if str(column).upper().startswith(
                    prefix
                )
            ]

            matching_columns.sort(
                key=column_number
            )

            current_bin_items = []

            for column in matching_columns:
                value = record.get(column)

                if pd.isna(value):
                    continue

                token = normalize_token(value)

                if token:
                    current_bin_items.append(token)
                    ordered_items.append(token)

            bin_counts[output_column] = len(
                current_bin_items
            )

        features = calculate_item_features(
            ordered_items
        )

        wave_text = (
            str(int(wave))
            if pd.notna(wave)
            else path.stem
        )

        row = {
            "dataset": "WLS",
            "source_type": "manual_scoring_spreadsheet",
            "task": task,
            "task_prompt": task_prompt,
            "subject_id": f"WLS_{source_subject_id}",
            "source_subject_id": source_subject_id,
            "recording_id": (
                f"WLS_{source_subject_id}_"
                f"{wave_text}_{task}"
            ),
            "source_recording_id": (
                f"{source_subject_id}_{path.stem}"
            ),
            "visit": np.nan,
            "wave": wave,
            "diagnosis_raw": "Control",
            "diagnosis_group": "Control",
            "binary_label": 0,
            "age": np.nan,
            "sex": "",
            "mmse": np.nan,
            "response_text": features["items_text"],
            "items_text": features["items_text"],
            "total_items": features["total_items"],
            "unique_items": features["unique_items"],
            "repetition_count": features[
                "repetition_count"
            ],
            "type_token_ratio": features[
                "type_token_ratio"
            ],
            "items_0_15s": bin_counts.get(
                "items_0_15s",
                np.nan
            ),
            "items_15_30s": bin_counts.get(
                "items_15_30s",
                np.nan
            ),
            "items_30_45s": bin_counts.get(
                "items_30_45s",
                np.nan
            ),
            "items_45_60s": bin_counts.get(
                "items_45_60s",
                np.nan
            ),
            "source_path": str(
                path.relative_to(LOCAL_ROOT)
            )
        }

        rows.append(row)

    return rows


# ============================================================
# FIND SOURCE FILES
# ============================================================

cha_files = sorted(
    LOCAL_ROOT.rglob("*.cha")
)

excel_files = sorted(
    list(LOCAL_ROOT.rglob("*.xls"))
    + list(LOCAL_ROOT.rglob("*.xlsx"))
)

print()
print("CHAT files found:", len(cha_files))
print("Excel files found:", len(excel_files))


# ============================================================
# PARSE EVERYTHING
# ============================================================

all_rows = []
errors = []

for path in cha_files:
    if "fluency" not in str(path).lower():
        continue

    try:
        rows = parse_pitt_fluency_file(path)
        all_rows.extend(rows)

    except Exception as error:
        errors.append({
            "source_path": str(path),
            "error": repr(error)
        })


for path in excel_files:
    lowercase_name = path.name.lower()

    if (
        "cogcat" not in lowercase_name
        and "coglet" not in lowercase_name
    ):
        continue

    try:
        rows = parse_wls_spreadsheet(path)
        all_rows.extend(rows)

    except Exception as error:
        errors.append({
            "source_path": str(path),
            "error": repr(error)
        })


if not all_rows:
    raise RuntimeError(
        "No fluency samples were created. "
        "Check the Drive folder ID and source files."
    )


# ============================================================
# CREATE DATAFRAME
# ============================================================

data = pd.DataFrame(all_rows)

data["binary_label"] = pd.to_numeric(
    data["binary_label"],
    errors="coerce"
)

data["subject_id"] = data[
    "subject_id"
].astype(str)

data["recording_id"] = data[
    "recording_id"
].astype(str)


# ============================================================
# REMOVE EXACT DUPLICATE RECORDING IDS
# ============================================================

duplicate_mask = data.duplicated(
    subset=["recording_id"],
    keep="first"
)

duplicate_rows = data[
    duplicate_mask
].copy()

data = data[
    ~duplicate_mask
].reset_index(drop=True)


# ============================================================
# COLUMN ORDER
# ============================================================

column_order = [
    "dataset",
    "source_type",
    "task",
    "task_prompt",
    "subject_id",
    "source_subject_id",
    "recording_id",
    "source_recording_id",
    "visit",
    "wave",
    "diagnosis_raw",
    "diagnosis_group",
    "binary_label",
    "age",
    "sex",
    "mmse",
    "response_text",
    "items_text",
    "total_items",
    "unique_items",
    "repetition_count",
    "type_token_ratio",
    "items_0_15s",
    "items_15_30s",
    "items_30_45s",
    "items_45_60s",
    "source_path"
]

data = data[column_order]


# ============================================================
# SEPARATE DATASETS
# ============================================================

category_all = data[
    data["task"] == "category"
].copy()

letter_all = data[
    data["task"] == "letter"
].copy()

category_binary = category_all[
    category_all["binary_label"].isin([0, 1])
].copy()

letter_binary = letter_all[
    letter_all["binary_label"].isin([0, 1])
].copy()


# ============================================================
# VALIDATION CHECKS
# ============================================================

subject_label_counts = (
    data.dropna(subset=["binary_label"])
    .groupby("subject_id")["binary_label"]
    .nunique()
)

conflicting_subjects = subject_label_counts[
    subject_label_counts > 1
].index.tolist()

if conflicting_subjects:
    raise RuntimeError(
        "Subjects with conflicting labels found: "
        + ", ".join(conflicting_subjects[:20])
    )


summary = (
    data.groupby(
        [
            "task",
            "dataset",
            "diagnosis_group",
            "binary_label"
        ],
        dropna=False
    )
    .agg(
        recordings=(
            "recording_id",
            "nunique"
        ),
        subjects=(
            "subject_id",
            "nunique"
        ),
        median_total_items=(
            "total_items",
            "median"
        ),
        median_unique_items=(
            "unique_items",
            "median"
        )
    )
    .reset_index()
)


pitt_multiple_visits = (
    data[
        data["dataset"] == "Pitt"
    ]
    .groupby("subject_id")[
        "source_recording_id"
    ]
    .nunique()
)

wls_multiple_waves = (
    data[
        data["dataset"] == "WLS"
    ]
    .groupby("subject_id")[
        "wave"
    ]
    .nunique()
)

integrity_report = pd.DataFrame({
    "check": [
        "duplicate_recording_ids_removed",
        "subjects_with_conflicting_binary_labels",
        "Pitt_subjects_with_multiple_visits",
        "WLS_subjects_with_multiple_waves"
    ],
    "count": [
        len(duplicate_rows),
        len(conflicting_subjects),
        int((pitt_multiple_visits > 1).sum()),
        int((wls_multiple_waves > 1).sum())
    ]
})


# ============================================================
# SAVE OUTPUTS
# ============================================================

data.to_csv(
    OUTPUT_DIR / "fluency_all.csv",
    index=False
)

category_all.to_csv(
    OUTPUT_DIR / "category_fluency_all.csv",
    index=False
)

letter_all.to_csv(
    OUTPUT_DIR / "letter_fluency_all.csv",
    index=False
)

category_binary.to_csv(
    OUTPUT_DIR / "category_AD_vs_Control.csv",
    index=False
)

letter_binary.to_csv(
    OUTPUT_DIR / "letter_AD_vs_Control.csv",
    index=False
)

summary.to_csv(
    OUTPUT_DIR / "dataset_summary.csv",
    index=False
)

integrity_report.to_csv(
    OUTPUT_DIR / "integrity_report.csv",
    index=False
)

if errors:
    pd.DataFrame(errors).to_csv(
        OUTPUT_DIR / "preprocessing_errors.csv",
        index=False
    )

if not duplicate_rows.empty:
    duplicate_rows.to_csv(
        OUTPUT_DIR / "duplicate_recordings_removed.csv",
        index=False
    )


# ============================================================
# FINAL OUTPUT
# ============================================================

print()
print("=" * 100)
print("PREPROCESSING COMPLETE")
print("=" * 100)

print()
print("DATASET SUMMARY")
display(summary)

print()
print("INTEGRITY REPORT")
display(integrity_report)

print()
print("CATEGORY AD VS CONTROL")
print("Rows:", len(category_binary))
print(
    category_binary["binary_label"]
    .value_counts(dropna=False)
    .sort_index()
)

print()
print("LETTER AD VS CONTROL")
print("Rows:", len(letter_binary))
print(
    letter_binary["binary_label"]
    .value_counts(dropna=False)
    .sort_index()
)

print()
print("FILES SAVED TO:")
print(OUTPUT_DIR)

for output_file in sorted(
    OUTPUT_DIR.glob("*.csv")
):
    print(" -", output_file.name)

print()
print("IMPORTANT:")
print(
    "Category fluency is the cleanest first model because "
    "Pitt and WLS both use animal naming."
)

print(
    "Pitt letter fluency uses F words. "
    "WLS letter fluency may use a different letter, so do not "
    "combine them into one diagnostic model until the prompts "
    "are verified and matched."
)

In [ ]:
# ============================================================
# COMPLETE FLUENCY DATA QUALITY AUDIT
# Run after the preprocessing cell
# ============================================================

import re
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


# ============================================================
# PATHS
# ============================================================

PROCESSED_DIR = Path("/content/drive/MyDrive/Fluency_Processed")
AUDIT_DIR = PROCESSED_DIR / "Audit"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)

CATEGORY_PATH = PROCESSED_DIR / "category_fluency_all.csv"
LETTER_PATH = PROCESSED_DIR / "letter_fluency_all.csv"
ALL_PATH = PROCESSED_DIR / "fluency_all.csv"


# ============================================================
# SETTINGS
# ============================================================

RANDOM_SEED = 42
RANDOM_ROWS_TO_INSPECT = 30

MIN_CATEGORY_ITEMS = 3
MAX_CATEGORY_ITEMS = 80

MIN_LETTER_ITEMS = 2
MAX_LETTER_ITEMS = 70

MAX_REASONABLE_REPETITION_RATE = 0.60
MAX_REASONABLE_TYPE_TOKEN_RATIO = 1.00

COMMON_JUNK_WORDS = {
    "okay", "ok", "yes", "yeah", "no", "well",
    "think", "know", "remember", "maybe", "probably",
    "minute", "time", "animal", "animals",
    "word", "words", "letter",
    "start", "begin", "beginning",
    "say", "said", "tell", "told",
    "name", "named", "naming",
    "more", "another", "other",
    "thing", "things",
    "uh", "um", "er", "erm", "hmm", "hm",
    "dont", "cant", "cannot",
    "let", "lets", "see"
}

NON_ANIMAL_SUSPECT_WORDS = {
    "table", "chair", "house", "car", "road", "tree",
    "water", "food", "fruit", "person", "people",
    "man", "woman", "boy", "girl",
    "kitchen", "cookie", "plate", "sink",
    "shirt", "shoe", "door", "window"
}


# ============================================================
# LOAD FILES
# ============================================================

required_files = [
    CATEGORY_PATH,
    LETTER_PATH,
    ALL_PATH
]

missing_files = [
    str(path)
    for path in required_files
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Missing required files:\n" + "\n".join(missing_files)
    )

category = pd.read_csv(CATEGORY_PATH)
letter = pd.read_csv(LETTER_PATH)
all_data = pd.read_csv(ALL_PATH)

print("Loaded:")
print("Category rows:", len(category))
print("Letter rows:", len(letter))
print("All rows:", len(all_data))


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def normalize_text(value):
    if pd.isna(value):
        return ""

    value = str(value).lower().strip()
    value = re.sub(r"\s+", " ", value)

    return value


def tokenize(value):
    text = normalize_text(value)

    return [
        token
        for token in re.findall(r"[a-z][a-z'-]*", text)
        if token
    ]


def safe_divide(numerator, denominator):
    if denominator == 0 or pd.isna(denominator):
        return np.nan

    return numerator / denominator


def repetition_rate(row):
    total = row.get("total_items", np.nan)
    repetitions = row.get("repetition_count", np.nan)

    return safe_divide(repetitions, total)


def timing_sum(row):
    columns = [
        "items_0_15s",
        "items_15_30s",
        "items_30_45s",
        "items_45_60s"
    ]

    values = [
        pd.to_numeric(row.get(column), errors="coerce")
        for column in columns
    ]

    if all(pd.isna(value) for value in values):
        return np.nan

    return np.nansum(values)


def production_decline(row):
    first = pd.to_numeric(
        row.get("items_0_15s"),
        errors="coerce"
    )

    last = pd.to_numeric(
        row.get("items_45_60s"),
        errors="coerce"
    )

    if pd.isna(first) or pd.isna(last):
        return np.nan

    return first - last


def find_junk_words(text):
    tokens = tokenize(text)

    return sorted(
        set(tokens) & COMMON_JUNK_WORDS
    )


def find_non_animal_suspects(text):
    tokens = tokenize(text)

    return sorted(
        set(tokens) & NON_ANIMAL_SUSPECT_WORDS
    )


def check_letter_f_only(text):
    tokens = tokenize(text)

    invalid = [
        token
        for token in tokens
        if not token.startswith("f")
    ]

    return invalid


def add_derived_columns(frame):
    frame = frame.copy()

    numeric_columns = [
        "total_items",
        "unique_items",
        "repetition_count",
        "type_token_ratio",
        "items_0_15s",
        "items_15_30s",
        "items_30_45s",
        "items_45_60s",
        "age",
        "mmse",
        "binary_label"
    ]

    for column in numeric_columns:
        if column in frame.columns:
            frame[column] = pd.to_numeric(
                frame[column],
                errors="coerce"
            )

    frame["computed_token_count"] = frame[
        "items_text"
    ].apply(lambda value: len(tokenize(value)))

    frame["computed_unique_count"] = frame[
        "items_text"
    ].apply(lambda value: len(set(tokenize(value))))

    frame["computed_repetition_count"] = (
        frame["computed_token_count"]
        - frame["computed_unique_count"]
    )

    frame["repetition_rate"] = frame.apply(
        repetition_rate,
        axis=1
    )

    frame["timing_bin_sum"] = frame.apply(
        timing_sum,
        axis=1
    )

    frame["production_decline"] = frame.apply(
        production_decline,
        axis=1
    )

    frame["junk_words"] = frame[
        "items_text"
    ].apply(find_junk_words)

    frame["junk_word_count"] = frame[
        "junk_words"
    ].apply(len)

    frame["empty_items_text"] = (
        frame["items_text"]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
    )

    frame["empty_response_text"] = (
        frame["response_text"]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
    )

    return frame


category = add_derived_columns(category)
letter = add_derived_columns(letter)
all_data = add_derived_columns(all_data)


# ============================================================
# CHECK 1: BASIC SCHEMA
# ============================================================

required_columns = [
    "dataset",
    "task",
    "task_prompt",
    "subject_id",
    "recording_id",
    "diagnosis_group",
    "binary_label",
    "response_text",
    "items_text",
    "total_items",
    "unique_items",
    "repetition_count",
    "type_token_ratio",
    "items_0_15s",
    "items_15_30s",
    "items_30_45s",
    "items_45_60s"
]

missing_columns = [
    column
    for column in required_columns
    if column not in all_data.columns
]

schema_report = pd.DataFrame({
    "check": [
        "required_columns_present",
        "number_of_missing_columns"
    ],
    "result": [
        len(missing_columns) == 0,
        len(missing_columns)
    ],
    "details": [
        "All required columns found"
        if not missing_columns
        else ", ".join(missing_columns),
        ", ".join(missing_columns)
        if missing_columns
        else ""
    ]
})

display(schema_report)


# ============================================================
# CHECK 2: TASK CONSISTENCY
# ============================================================

task_consistency = pd.DataFrame({
    "check": [
        "category_file_only_category",
        "letter_file_only_letter"
    ],
    "passed": [
        set(category["task"].dropna().unique()) == {"category"},
        set(letter["task"].dropna().unique()) == {"letter"}
    ],
    "observed_values": [
        str(sorted(category["task"].dropna().unique())),
        str(sorted(letter["task"].dropna().unique()))
    ]
})

display(task_consistency)


# ============================================================
# CHECK 3: DUPLICATE IDS AND DUPLICATE TEXT
# ============================================================

duplicate_recording_ids = all_data[
    all_data.duplicated(
        subset=["recording_id"],
        keep=False
    )
].sort_values("recording_id")

duplicate_subject_recording_task = all_data[
    all_data.duplicated(
        subset=[
            "subject_id",
            "source_recording_id",
            "task"
        ],
        keep=False
    )
].sort_values(
    [
        "subject_id",
        "source_recording_id",
        "task"
    ]
)

duplicate_text = all_data[
    all_data.duplicated(
        subset=[
            "dataset",
            "task",
            "items_text"
        ],
        keep=False
    )
    & all_data["items_text"].notna()
    & all_data["items_text"].astype(str).str.strip().ne("")
].sort_values(
    [
        "dataset",
        "task",
        "items_text"
    ]
)

duplicate_report = pd.DataFrame({
    "check": [
        "duplicate_recording_ids",
        "duplicate_subject_recording_task_rows",
        "duplicate_items_text_rows"
    ],
    "count": [
        len(duplicate_recording_ids),
        len(duplicate_subject_recording_task),
        len(duplicate_text)
    ]
})

display(duplicate_report)


# ============================================================
# CHECK 4: SUBJECT LABEL CONFLICTS
# ============================================================

subject_label_counts = (
    all_data.dropna(subset=["binary_label"])
    .groupby("subject_id")["binary_label"]
    .nunique()
)

conflicting_subject_ids = subject_label_counts[
    subject_label_counts > 1
].index.tolist()

conflicting_subject_rows = all_data[
    all_data["subject_id"].isin(
        conflicting_subject_ids
    )
].sort_values(
    [
        "subject_id",
        "task",
        "recording_id"
    ]
)

print(
    "Subjects with conflicting binary labels:",
    len(conflicting_subject_ids)
)


# ============================================================
# CHECK 5: SUBJECT APPEARANCE ACROSS DATASETS
# ============================================================

raw_subject_cross_dataset = (
    all_data.groupby("source_subject_id")["dataset"]
    .nunique()
)

cross_dataset_raw_ids = raw_subject_cross_dataset[
    raw_subject_cross_dataset > 1
].index.tolist()

cross_dataset_rows = all_data[
    all_data["source_subject_id"].isin(
        cross_dataset_raw_ids
    )
].sort_values(
    [
        "source_subject_id",
        "dataset"
    ]
)

print(
    "Raw subject IDs appearing in multiple datasets:",
    len(cross_dataset_raw_ids)
)

print(
    "This is usually harmless because Pitt and WLS use separate "
    "namespaced subject_id values."
)


# ============================================================
# CHECK 6: MULTIPLE VISITS AND WAVES
# ============================================================

subject_recording_counts = (
    all_data.groupby(
        [
            "dataset",
            "subject_id",
            "task"
        ]
    )["recording_id"]
    .nunique()
    .reset_index(name="recordings")
)

multi_recording_subjects = subject_recording_counts[
    subject_recording_counts["recordings"] > 1
].sort_values(
    "recordings",
    ascending=False
)

print(
    "Subject-task combinations with multiple recordings:",
    len(multi_recording_subjects)
)

print(
    "These subjects must remain entirely within one train/validation/test split."
)


# ============================================================
# CHECK 7: LABEL AND DIAGNOSIS COUNTS
# ============================================================

label_summary = (
    all_data.groupby(
        [
            "task",
            "dataset",
            "diagnosis_group",
            "binary_label"
        ],
        dropna=False
    )
    .agg(
        rows=("recording_id", "size"),
        recordings=("recording_id", "nunique"),
        subjects=("subject_id", "nunique")
    )
    .reset_index()
)

display(label_summary)


# ============================================================
# CHECK 8: EMPTY OR VERY SHORT RESPONSES
# ============================================================

empty_or_short = all_data[
    all_data["empty_items_text"]
    | all_data["empty_response_text"]
    | (all_data["total_items"] <= 1)
].copy()

print(
    "Empty or extremely short rows:",
    len(empty_or_short)
)


# ============================================================
# CHECK 9: COUNT CONSISTENCY
# ============================================================

count_mismatch = all_data[
    (all_data["total_items"] != all_data["computed_token_count"])
    | (all_data["unique_items"] != all_data["computed_unique_count"])
    | (
        all_data["repetition_count"]
        != all_data["computed_repetition_count"]
    )
].copy()

print(
    "Rows where saved item counts disagree with items_text:",
    len(count_mismatch)
)


# ============================================================
# CHECK 10: IMPOSSIBLE OR SUSPICIOUS NUMERIC VALUES
# ============================================================

numeric_flags = []

for index, row in all_data.iterrows():
    reasons = []

    total_items = row["total_items"]
    unique_items = row["unique_items"]
    repetitions = row["repetition_count"]
    ttr = row["type_token_ratio"]

    if pd.isna(total_items):
        reasons.append("missing_total_items")

    if pd.isna(unique_items):
        reasons.append("missing_unique_items")

    if total_items < 0:
        reasons.append("negative_total_items")

    if unique_items < 0:
        reasons.append("negative_unique_items")

    if unique_items > total_items:
        reasons.append("unique_items_exceeds_total")

    if repetitions < 0:
        reasons.append("negative_repetition_count")

    if repetitions > total_items:
        reasons.append("repetitions_exceed_total")

    if pd.notna(ttr) and (
        ttr < 0
        or ttr > MAX_REASONABLE_TYPE_TOKEN_RATIO
    ):
        reasons.append("invalid_type_token_ratio")

    if pd.notna(row["repetition_rate"]) and (
        row["repetition_rate"]
        > MAX_REASONABLE_REPETITION_RATE
    ):
        reasons.append("very_high_repetition_rate")

    if reasons:
        numeric_flags.append({
            "row_index": index,
            "recording_id": row["recording_id"],
            "subject_id": row["subject_id"],
            "dataset": row["dataset"],
            "task": row["task"],
            "reasons": ";".join(reasons)
        })

numeric_flags = pd.DataFrame(numeric_flags)

print(
    "Rows with suspicious numeric values:",
    len(numeric_flags)
)


# ============================================================
# CHECK 11: CATEGORY-SPECIFIC FLAGS
# ============================================================

category_flags = []

for index, row in category.iterrows():
    reasons = []

    total_items = row["total_items"]

    if total_items < MIN_CATEGORY_ITEMS:
        reasons.append("too_few_category_items")

    if total_items > MAX_CATEGORY_ITEMS:
        reasons.append("too_many_category_items")

    if row["junk_word_count"] > 0:
        reasons.append(
            "junk_words:" + ",".join(row["junk_words"])
        )

    non_animal_suspects = find_non_animal_suspects(
        row["items_text"]
    )

    if non_animal_suspects:
        reasons.append(
            "possible_non_animals:"
            + ",".join(non_animal_suspects)
        )

    if reasons:
        category_flags.append({
            "row_index": index,
            "recording_id": row["recording_id"],
            "subject_id": row["subject_id"],
            "dataset": row["dataset"],
            "diagnosis_group": row["diagnosis_group"],
            "total_items": row["total_items"],
            "unique_items": row["unique_items"],
            "items_text": row["items_text"],
            "reasons": ";".join(reasons)
        })

category_flags = pd.DataFrame(category_flags)

print(
    "Category rows flagged for manual review:",
    len(category_flags)
)


# ============================================================
# CHECK 12: LETTER-SPECIFIC FLAGS
# ============================================================

letter_flags = []

for index, row in letter.iterrows():
    reasons = []

    total_items = row["total_items"]

    if total_items < MIN_LETTER_ITEMS:
        reasons.append("too_few_letter_items")

    if total_items > MAX_LETTER_ITEMS:
        reasons.append("too_many_letter_items")

    if row["junk_word_count"] > 0:
        reasons.append(
            "junk_words:" + ",".join(row["junk_words"])
        )

    if (
        row["dataset"] == "Pitt"
        and row["task_prompt"] == "letter_F"
    ):
        invalid_f_words = check_letter_f_only(
            row["items_text"]
        )

        if invalid_f_words:
            reasons.append(
                "non_f_words:"
                + ",".join(
                    sorted(set(invalid_f_words))[:20]
                )
            )

    if reasons:
        letter_flags.append({
            "row_index": index,
            "recording_id": row["recording_id"],
            "subject_id": row["subject_id"],
            "dataset": row["dataset"],
            "task_prompt": row["task_prompt"],
            "diagnosis_group": row["diagnosis_group"],
            "total_items": row["total_items"],
            "unique_items": row["unique_items"],
            "items_text": row["items_text"],
            "reasons": ";".join(reasons)
        })

letter_flags = pd.DataFrame(letter_flags)

print(
    "Letter rows flagged for manual review:",
    len(letter_flags)
)


# ============================================================
# CHECK 13: TIMING BIN CONSISTENCY
# ============================================================

timing_available = all_data[
    all_data[
        [
            "items_0_15s",
            "items_15_30s",
            "items_30_45s",
            "items_45_60s"
        ]
    ].notna().any(axis=1)
].copy()

timing_mismatch = timing_available[
    timing_available["timing_bin_sum"]
    != timing_available["total_items"]
].copy()

timing_mismatch["difference"] = (
    timing_mismatch["timing_bin_sum"]
    - timing_mismatch["total_items"]
)

print(
    "Rows where timing-bin sum differs from total_items:",
    len(timing_mismatch)
)


# ============================================================
# CHECK 14: WLS WAVE DUPLICATION
# ============================================================

wls = all_data[
    all_data["dataset"] == "WLS"
].copy()

wls_subject_task_waves = (
    wls.groupby(
        [
            "subject_id",
            "task"
        ]
    )["wave"]
    .nunique()
    .reset_index(name="number_of_waves")
)

wls_multiple_waves = wls_subject_task_waves[
    wls_subject_task_waves["number_of_waves"] > 1
].copy()

print(
    "WLS subject-task combinations in multiple waves:",
    len(wls_multiple_waves)
)


# ============================================================
# CHECK 15: DATASET CONFOUNDING REPORT
# ============================================================

binary_only = all_data[
    all_data["binary_label"].isin([0, 1])
].copy()

confounding_table = pd.crosstab(
    [
        binary_only["task"],
        binary_only["dataset"]
    ],
    binary_only["binary_label"],
    margins=True
)

print()
print("DATASET VS LABEL CONFOUNDING TABLE")
display(confounding_table)

confounding_warning = (
    "Most AD rows are Pitt while most control rows are WLS. "
    "A model may learn dataset source instead of disease."
)

print(confounding_warning)


# ============================================================
# CHECK 16: DESCRIPTIVE STATISTICS
# ============================================================

feature_columns = [
    "total_items",
    "unique_items",
    "repetition_count",
    "repetition_rate",
    "type_token_ratio",
    "items_0_15s",
    "items_15_30s",
    "items_30_45s",
    "items_45_60s",
    "production_decline"
]

descriptive_statistics = (
    all_data.groupby(
        [
            "task",
            "dataset",
            "diagnosis_group"
        ],
        dropna=False
    )[feature_columns]
    .agg(
        [
            "count",
            "mean",
            "median",
            "std",
            "min",
            "max"
        ]
    )
)

display(descriptive_statistics)


# ============================================================
# CHECK 17: RANDOM MANUAL INSPECTION SAMPLES
# ============================================================

random_pitt_category = (
    category[
        category["dataset"] == "Pitt"
    ]
    .sample(
        n=min(
            RANDOM_ROWS_TO_INSPECT,
            len(
                category[
                    category["dataset"] == "Pitt"
                ]
            )
        ),
        random_state=RANDOM_SEED
    )
    [
        [
            "recording_id",
            "subject_id",
            "diagnosis_group",
            "response_text",
            "items_text",
            "total_items",
            "unique_items",
            "repetition_count",
            "items_0_15s",
            "items_15_30s",
            "items_30_45s",
            "items_45_60s"
        ]
    ]
)

random_wls_category = (
    category[
        category["dataset"] == "WLS"
    ]
    .sample(
        n=min(
            RANDOM_ROWS_TO_INSPECT,
            len(
                category[
                    category["dataset"] == "WLS"
                ]
            )
        ),
        random_state=RANDOM_SEED
    )
    [
        [
            "recording_id",
            "subject_id",
            "wave",
            "items_text",
            "total_items",
            "unique_items",
            "repetition_count",
            "items_0_15s",
            "items_15_30s",
            "items_30_45s",
            "items_45_60s"
        ]
    ]
)

print()
print("RANDOM PITT CATEGORY ROWS")
display(random_pitt_category)

print()
print("RANDOM WLS CATEGORY ROWS")
display(random_wls_category)


# ============================================================
# CHECK 18: DISTRIBUTION PLOTS
# ============================================================

plot_data = category[
    category["diagnosis_group"].isin(
        [
            "AD",
            "Control"
        ]
    )
].copy()

for feature in [
    "total_items",
    "unique_items",
    "repetition_count",
    "type_token_ratio",
    "items_0_15s",
    "items_15_30s",
    "items_30_45s",
    "items_45_60s"
]:
    plt.figure(figsize=(8, 5))

    for diagnosis in [
        "AD",
        "Control"
    ]:
        values = plot_data.loc[
            plot_data["diagnosis_group"] == diagnosis,
            feature
        ].dropna()

        if len(values) > 0:
            plt.hist(
                values,
                bins=30,
                alpha=0.5,
                label=diagnosis
            )

    plt.title(
        f"Category fluency: {feature}"
    )
    plt.xlabel(feature)
    plt.ylabel("Number of recordings")
    plt.legend()
    plt.tight_layout()

    output_path = AUDIT_DIR / (
        f"category_distribution_{feature}.png"
    )

    plt.savefig(
        output_path,
        dpi=150,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()


# ============================================================
# CHECK 19: CREATE SUBJECT-LEVEL SPLIT TABLE
# ============================================================

subject_table = (
    all_data.groupby(
        [
            "subject_id",
            "dataset",
            "diagnosis_group",
            "binary_label"
        ],
        dropna=False
    )
    .agg(
        number_of_recordings=(
            "recording_id",
            "nunique"
        ),
        number_of_tasks=(
            "task",
            "nunique"
        ),
        first_wave=(
            "wave",
            "min"
        ),
        last_wave=(
            "wave",
            "max"
        )
    )
    .reset_index()
)

subject_table["subject_split_group"] = subject_table[
    "subject_id"
]

display(subject_table.head())


# ============================================================
# CHECK 20: CREATE RECOMMENDED CATEGORY MODELING DATA
# ============================================================

category_binary = category[
    category["binary_label"].isin([0, 1])
].copy()

# Use one WLS recording per subject for the first model.
# Prefer the earliest available wave for consistency.
wls_category_controls = (
    category_binary[
        (category_binary["dataset"] == "WLS")
        & (category_binary["binary_label"] == 0)
    ]
    .sort_values(
        [
            "subject_id",
            "wave"
        ]
    )
    .drop_duplicates(
        subset=["subject_id"],
        keep="first"
    )
)

pitt_category_ad = category_binary[
    (category_binary["dataset"] == "Pitt")
    & (category_binary["binary_label"] == 1)
].copy()

# Keep all Pitt visits for now, but preserve subject grouping.
# Sampling controls at subject level.
number_of_ad_subjects = pitt_category_ad[
    "subject_id"
].nunique()

number_of_controls_to_sample = min(
    len(wls_category_controls),
    number_of_ad_subjects
)

sampled_control_subjects = (
    wls_category_controls[
        ["subject_id"]
    ]
    .drop_duplicates()
    .sample(
        n=number_of_controls_to_sample,
        random_state=RANDOM_SEED
    )["subject_id"]
)

balanced_category = pd.concat(
    [
        pitt_category_ad,
        wls_category_controls[
            wls_category_controls["subject_id"].isin(
                sampled_control_subjects
            )
        ]
    ],
    ignore_index=True
)

print()
print("RECOMMENDED BALANCED CATEGORY DATASET")
print(
    "AD subjects:",
    balanced_category.loc[
        balanced_category["binary_label"] == 1,
        "subject_id"
    ].nunique()
)

print(
    "Control subjects:",
    balanced_category.loc[
        balanced_category["binary_label"] == 0,
        "subject_id"
    ].nunique()
)

print(
    "Total rows:",
    len(balanced_category)
)


# ============================================================
# OVERALL AUDIT SUMMARY
# ============================================================

audit_summary = pd.DataFrame({
    "check": [
        "missing_required_columns",
        "duplicate_recording_id_rows",
        "duplicate_subject_recording_task_rows",
        "duplicate_text_rows",
        "conflicting_subject_labels",
        "empty_or_short_rows",
        "count_mismatch_rows",
        "suspicious_numeric_rows",
        "category_manual_review_rows",
        "letter_manual_review_rows",
        "timing_mismatch_rows",
        "WLS_subject_task_multiple_wave_rows",
        "raw_subject_ids_crossing_datasets"
    ],
    "count": [
        len(missing_columns),
        len(duplicate_recording_ids),
        len(duplicate_subject_recording_task),
        len(duplicate_text),
        len(conflicting_subject_ids),
        len(empty_or_short),
        len(count_mismatch),
        len(numeric_flags),
        len(category_flags),
        len(letter_flags),
        len(timing_mismatch),
        len(wls_multiple_waves),
        len(cross_dataset_raw_ids)
    ]
})

print()
print("=" * 100)
print("FINAL AUDIT SUMMARY")
print("=" * 100)

display(audit_summary)


# ============================================================
# SAVE ALL REPORTS
# ============================================================

schema_report.to_csv(
    AUDIT_DIR / "schema_report.csv",
    index=False
)

task_consistency.to_csv(
    AUDIT_DIR / "task_consistency.csv",
    index=False
)

duplicate_report.to_csv(
    AUDIT_DIR / "duplicate_report.csv",
    index=False
)

duplicate_recording_ids.to_csv(
    AUDIT_DIR / "duplicate_recording_ids.csv",
    index=False
)

duplicate_subject_recording_task.to_csv(
    AUDIT_DIR / "duplicate_subject_recording_task.csv",
    index=False
)

duplicate_text.to_csv(
    AUDIT_DIR / "duplicate_text_rows.csv",
    index=False
)

conflicting_subject_rows.to_csv(
    AUDIT_DIR / "conflicting_subject_labels.csv",
    index=False
)

cross_dataset_rows.to_csv(
    AUDIT_DIR / "raw_subject_ids_crossing_datasets.csv",
    index=False
)

multi_recording_subjects.to_csv(
    AUDIT_DIR / "subjects_with_multiple_recordings.csv",
    index=False
)

label_summary.to_csv(
    AUDIT_DIR / "label_summary.csv",
    index=False
)

empty_or_short.to_csv(
    AUDIT_DIR / "empty_or_short_rows.csv",
    index=False
)

count_mismatch.to_csv(
    AUDIT_DIR / "count_mismatch_rows.csv",
    index=False
)

numeric_flags.to_csv(
    AUDIT_DIR / "suspicious_numeric_rows.csv",
    index=False
)

category_flags.to_csv(
    AUDIT_DIR / "category_manual_review.csv",
    index=False
)

letter_flags.to_csv(
    AUDIT_DIR / "letter_manual_review.csv",
    index=False
)

timing_mismatch.to_csv(
    AUDIT_DIR / "timing_mismatch_rows.csv",
    index=False
)

wls_multiple_waves.to_csv(
    AUDIT_DIR / "wls_subjects_multiple_waves.csv",
    index=False
)

descriptive_statistics.to_csv(
    AUDIT_DIR / "descriptive_statistics.csv"
)

random_pitt_category.to_csv(
    AUDIT_DIR / "random_pitt_category_review.csv",
    index=False
)

random_wls_category.to_csv(
    AUDIT_DIR / "random_wls_category_review.csv",
    index=False
)

subject_table.to_csv(
    AUDIT_DIR / "subject_level_table.csv",
    index=False
)

balanced_category.to_csv(
    AUDIT_DIR / "category_balanced_first_model.csv",
    index=False
)

audit_summary.to_csv(
    AUDIT_DIR / "audit_summary.csv",
    index=False
)


# ============================================================
# FINAL MESSAGE
# ============================================================

print()
print("=" * 100)
print("AUDIT COMPLETE")
print("=" * 100)

print()
print("Reports saved to:")
print(AUDIT_DIR)

print()
print("Most important files:")
print(
    AUDIT_DIR / "audit_summary.csv"
)

print(
    AUDIT_DIR / "category_manual_review.csv"
)

print(
    AUDIT_DIR / "random_pitt_category_review.csv"
)

print(
    AUDIT_DIR / "count_mismatch_rows.csv"
)

print(
    AUDIT_DIR / "timing_mismatch_rows.csv"
)

print(
    AUDIT_DIR / "category_balanced_first_model.csv"
)

print()
print(
    "Do not train until you review the final audit summary "
    "and the flagged Pitt category rows."
)

In [ ]:
import pandas as pd
from pathlib import Path

SOURCE_ROOT = Path("/content/fluency_source")

files = sorted(
    list(SOURCE_ROOT.rglob("CogCat*.xls"))
    + list(SOURCE_ROOT.rglob("CogCat*.xlsx"))
)

if not files:
    raise FileNotFoundError("No CogCat files found.")

for path in files:
    print("\n" + "=" * 120)
    print("FILE:", path)
    print("=" * 120)

    df = pd.read_excel(path)

    print("\nSHAPE:", df.shape)

    print("\nALL COLUMNS:")
    for i, column in enumerate(df.columns):
        print(f"{i:03d}: {column}")

    response_columns = [
        column for column in df.columns
        if str(column).upper().startswith(
            ("FIR15", "SEC15", "THI15", "FOR15", "FIF15")
        )
    ]

    metadata_columns = [
        column for column in df.columns
        if column not in response_columns
    ]

    print("\nMETADATA COLUMNS:")
    print(metadata_columns)

    print("\nFIRST 20 METADATA ROWS:")
    display(df[metadata_columns].head(20))

    print("\nUNIQUE VALUES IN SMALL METADATA COLUMNS:")

    for column in metadata_columns:
        values = df[column].dropna()
        unique_count = values.nunique()

        if unique_count <= 50:
            print(f"\nCOLUMN: {column}")
            print(f"UNIQUE COUNT: {unique_count}")
            print(values.astype(str).value_counts().head(50))

    print("\nSAMPLE RESPONSE + METADATA ROWS:")

    sample_columns = metadata_columns + response_columns[:12]
    display(df[sample_columns].sample(
        n=min(30, len(df)),
        random_state=42
    ))

In [ ]:
# ============================================================
# SEPARATE WLS COGCAT INTO ANIMAL VS FOOD FLUENCY
# ============================================================

import re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd


# ============================================================
# PATHS
# ============================================================

SOURCE_DIR = Path("/content/fluency_source/WLS/Category_Fluency")
OUTPUT_DIR = Path("/content/drive/MyDrive/Fluency_Processed/WLS_Category_Filtered")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# LEXICONS
# ============================================================

ANIMAL_WORDS = {
    "aardvark", "alligator", "alpaca", "ant", "anteater", "antelope",
    "ape", "apes", "armadillo", "baboon", "badger", "bat", "bear",
    "bears", "beaver", "bee", "bees", "bird", "birds", "bison",
    "bobcat", "buffalo", "bull", "bulls", "bunny", "butterfly",
    "camel", "camels", "canary", "cat", "cats", "calf", "calves",
    "caribou", "caterpillar", "chicken", "chickens", "chimpanzee",
    "chimpanzees", "chipmunk", "chipmunks", "cobra", "cod", "colt",
    "cougar", "cow", "cows", "coyote", "coyotes", "crab", "crabs",
    "crocodile", "crow", "deer", "dingo", "dog", "dogs", "dolphin",
    "donkey", "donkeys", "dove", "duck", "ducks", "eagle", "eagles",
    "eel", "elephant", "elephants", "elk", "emus", "falcon", "ferret",
    "fish", "flamingo", "fly", "fox", "foxes", "frog", "frogs",
    "gazelle", "geese", "giraffe", "giraffes", "goat", "goats",
    "goose", "gorilla", "gorillas", "grasshopper", "hamster",
    "hare", "hawk", "hedgehog", "hen", "hens", "hippo",
    "hippopotamus", "hippopotamuses", "hog", "hogs", "horse",
    "horses", "hummingbird", "hyena", "iguana", "insect", "insects",
    "jackal", "jaguar", "kangaroo", "kangaroos", "kitten", "kittens",
    "koala", "lamb", "lambs", "leopard", "leopards", "lion", "lions",
    "lizard", "llama", "lobster", "lynx", "mink", "mole", "monkey",
    "monkeys", "moose", "mosquito", "moth", "mouse", "mule", "mules",
    "muskrat", "octopus", "ocelot", "opossum", "orangutan", "ostrich",
    "otter", "owl", "ox", "oxen", "panda", "panther", "parrot",
    "peacock", "pelican", "penguin", "pheasant", "pig", "pigs",
    "pigeon", "pony", "ponies", "porcupine", "possum", "puma",
    "puppy", "puppies", "rabbit", "rabbits", "raccoon", "raccoons",
    "ram", "rat", "rats", "raven", "reindeer", "rhino", "rhinoceros",
    "rooster", "salmon", "seal", "sealion", "shark", "sheep",
    "skunk", "skunks", "snake", "snakes", "sparrow", "spider",
    "squirrel", "squirrels", "swan", "tiger", "tigers", "toad",
    "tortoise", "toucan", "trout", "turkey", "turkeys", "turtle",
    "walrus", "weasel", "whale", "wolf", "wolves", "worm", "worms",
    "zebra", "zebras"
}

FOOD_WORDS = {
    "apple", "apples", "apricot", "banana", "bananas", "bacon",
    "bagel", "beans", "beef", "beer", "berries", "blueberries",
    "bread", "breads", "broccoli", "bratwurst", "breakfastcereal",
    "buns", "butter", "cabbage", "cake", "candy", "cantaloupe",
    "carrot", "carrots", "cauliflower", "celery", "cereal",
    "cheese", "cherries", "cherry", "chili", "chocolate",
    "coffee", "cookie", "cookies", "corn", "crackers", "crisco",
    "dessert", "desserts", "dinner", "egg", "eggs", "egg_salad",
    "fish", "flour", "food", "foods", "frankfurter", "frankfurters",
    "french_fries", "fries", "frozenfood", "frozenfoods",
    "fruit", "fruits", "fruitcake", "garlic", "grape", "grapes",
    "grains", "greenbeans", "ham", "hamburger", "hamburgers",
    "hotdog", "hotdogs", "hot_dog", "icecream", "ice-cream",
    "juice", "ketchup", "lasagna", "lemon", "lemons", "lettuce",
    "macaroni", "mango", "meal", "meals", "meat", "meatloaf",
    "melon", "melons", "milk", "muffin", "muffins", "mustard",
    "nectarine", "nectarines", "noodles", "oatmeal", "oleo",
    "onion", "onions", "orange", "oranges", "orange_juice",
    "pasta", "peach", "peaches", "pear", "pears", "peas",
    "pineapple", "pizza", "plum", "plums", "popcorn", "pork",
    "porkchops", "pork_chops", "pork_loin", "potato", "potatoes",
    "potatochips", "pudding", "raisins", "rice", "roast",
    "roastbeef", "roast_beef", "rolls", "salad", "salads",
    "salmon", "sandwich", "sandwiches", "sauerkraut", "sausage",
    "seafood", "soup", "spaghetti", "squash", "steak", "steaks",
    "strawberries", "strawberry", "sugar", "tea", "toast",
    "tomato", "tomatoes", "tuna", "veal", "vegetable", "vegetables",
    "watermelon", "yogurt"
}

# Some terms may occur in either category.
# "fish", "salmon", and "chicken" are examples.
AMBIGUOUS_WORDS = {
    "fish", "salmon", "chicken", "turkey", "lamb", "pig", "pigs",
    "cow", "cows", "calf", "duck", "ducks"
}


# ============================================================
# CLEANING
# ============================================================

def normalize_item(value):
    if pd.isna(value):
        return ""

    value = str(value).strip().lower()
    value = value.lstrip("@!;?")
    value = value.replace(" ", "_")
    value = value.replace("-", "_")
    value = re.sub(r"[^a-z_]", "", value)
    value = re.sub(r"_+", "_", value).strip("_")

    return value


def flatten_response(row, response_columns):
    items = []

    for column in response_columns:
        item = normalize_item(row[column])

        if item:
            items.append(item)

    return items


# ============================================================
# CLASSIFICATION
# ============================================================

def classify_category(items):
    if not items:
        return {
            "inferred_prompt": "empty",
            "animal_matches": 0,
            "food_matches": 0,
            "animal_score": 0.0,
            "food_score": 0.0,
            "known_fraction": 0.0,
            "classification_confidence": 0.0
        }

    animal_matches = 0
    food_matches = 0
    known_matches = 0

    for item in items:
        animal_hit = item in ANIMAL_WORDS
        food_hit = item in FOOD_WORDS

        if animal_hit or food_hit:
            known_matches += 1

        # Do not let ambiguous words strongly favor either class.
        if item in AMBIGUOUS_WORDS:
            if animal_hit:
                animal_matches += 0.25
            if food_hit:
                food_matches += 0.25
        else:
            if animal_hit:
                animal_matches += 1
            if food_hit:
                food_matches += 1

    total = len(items)

    animal_score = animal_matches / total
    food_score = food_matches / total
    known_fraction = known_matches / total

    score_difference = abs(animal_score - food_score)

    # High-confidence classification rules.
    if (
        animal_matches >= 3
        and animal_score >= 0.50
        and animal_score >= food_score + 0.25
    ):
        inferred_prompt = "animals"

    elif (
        food_matches >= 3
        and food_score >= 0.50
        and food_score >= animal_score + 0.25
    ):
        inferred_prompt = "foods"

    else:
        inferred_prompt = "uncertain"

    confidence = min(
        1.0,
        score_difference + (known_fraction * 0.5)
    )

    return {
        "inferred_prompt": inferred_prompt,
        "animal_matches": animal_matches,
        "food_matches": food_matches,
        "animal_score": animal_score,
        "food_score": food_score,
        "known_fraction": known_fraction,
        "classification_confidence": confidence
    }


# ============================================================
# PROCESS BOTH WAVES
# ============================================================

all_rows = []

files = sorted(
    list(SOURCE_DIR.glob("CogCat*.xls"))
    + list(SOURCE_DIR.glob("CogCat*.xlsx"))
)

if not files:
    raise FileNotFoundError(
        f"No CogCat files found in {SOURCE_DIR}"
    )

for path in files:
    print("Processing:", path.name)

    df = pd.read_excel(path)

    response_columns = [
        column
        for column in df.columns
        if re.match(
            r"^(FIR15C?|SEC15C?|THI15C?|FOR15C?|FIF15C?)_\d+$",
            str(column)
        )
    ]

    def response_order(column):
        prefix_order = {
            "FIR": 0,
            "SEC": 1,
            "THI": 2,
            "FOR": 3,
            "FIF": 4
        }

        text = str(column)
        prefix = text[:3]
        number_match = re.search(r"_(\d+)$", text)
        number = int(number_match.group(1)) if number_match else 999

        return prefix_order.get(prefix, 99), number

    response_columns = sorted(
        response_columns,
        key=response_order
    )

    wave_match = re.search(r"(03|11)", path.stem)
    wave = (
        2000 + int(wave_match.group(1))
        if wave_match
        else np.nan
    )

    for _, row in df.iterrows():
        raw_id = row["idtlkbnk"]

        if pd.isna(raw_id):
            continue

        try:
            subject_id = str(int(raw_id))
        except Exception:
            subject_id = str(raw_id).strip()

        items = flatten_response(
            row,
            response_columns
        )

        result = classify_category(items)

        all_rows.append({
            "dataset": "WLS",
            "subject_id": f"WLS_{subject_id}",
            "source_subject_id": subject_id,
            "wave": wave,
            "recording_id": f"WLS_{subject_id}_{wave}_category",
            "items_text": " ".join(items),
            "total_items": len(items),
            **result,
            "source_file": path.name
        })


classified = pd.DataFrame(all_rows)


# ============================================================
# RESULTS
# ============================================================

summary = (
    classified.groupby(
        ["wave", "inferred_prompt"]
    )
    .agg(
        recordings=("recording_id", "nunique"),
        subjects=("subject_id", "nunique"),
        median_items=("total_items", "median"),
        median_known_fraction=("known_fraction", "median"),
        median_confidence=("classification_confidence", "median")
    )
    .reset_index()
)

print()
print("=" * 100)
print("CLASSIFICATION SUMMARY")
print("=" * 100)
display(summary)


# ============================================================
# CREATE CLEAN ANIMAL SUBSET
# ============================================================

animal_rows = classified[
    (classified["inferred_prompt"] == "animals")
    & (classified["animal_score"] >= 0.60)
    & (classified["known_fraction"] >= 0.60)
    & (classified["total_items"] >= 3)
].copy()

food_rows = classified[
    classified["inferred_prompt"] == "foods"
].copy()

uncertain_rows = classified[
    classified["inferred_prompt"].isin(
        ["uncertain", "empty"]
    )
].copy()


# ============================================================
# SUBJECT-LEVEL DEDUPLICATION
# ============================================================

# Keep the most recent high-confidence animal wave per subject.
animal_one_per_subject = (
    animal_rows
    .sort_values(
        [
            "subject_id",
            "classification_confidence",
            "wave"
        ],
        ascending=[True, False, False]
    )
    .drop_duplicates(
        subset=["subject_id"],
        keep="first"
    )
    .reset_index(drop=True)
)


# ============================================================
# SAVE
# ============================================================

classified.to_csv(
    OUTPUT_DIR / "wls_category_all_classified.csv",
    index=False
)

animal_rows.to_csv(
    OUTPUT_DIR / "wls_animal_fluency_all_waves.csv",
    index=False
)

animal_one_per_subject.to_csv(
    OUTPUT_DIR / "wls_animal_fluency_one_per_subject.csv",
    index=False
)

food_rows.to_csv(
    OUTPUT_DIR / "wls_food_fluency.csv",
    index=False
)

uncertain_rows.to_csv(
    OUTPUT_DIR / "wls_category_uncertain_review.csv",
    index=False
)

summary.to_csv(
    OUTPUT_DIR / "wls_category_classification_summary.csv",
    index=False
)


# ============================================================
# SHOW RANDOM SAMPLES
# ============================================================

print()
print("HIGH-CONFIDENCE ANIMAL SAMPLE")
display(
    animal_rows.sample(
        n=min(20, len(animal_rows)),
        random_state=42
    )[
        [
            "recording_id",
            "wave",
            "items_text",
            "animal_score",
            "food_score",
            "known_fraction",
            "classification_confidence"
        ]
    ]
)

print()
print("HIGH-CONFIDENCE FOOD SAMPLE")
display(
    food_rows.sample(
        n=min(20, len(food_rows)),
        random_state=42
    )[
        [
            "recording_id",
            "wave",
            "items_text",
            "animal_score",
            "food_score",
            "known_fraction",
            "classification_confidence"
        ]
    ]
)

print()
print("UNCERTAIN SAMPLE")
display(
    uncertain_rows.sample(
        n=min(20, len(uncertain_rows)),
        random_state=42
    )[
        [
            "recording_id",
            "wave",
            "items_text",
            "animal_score",
            "food_score",
            "known_fraction",
            "classification_confidence"
        ]
    ]
)


# ============================================================
# FINAL COUNTS
# ============================================================

print()
print("=" * 100)
print("FILTERING COMPLETE")
print("=" * 100)

print("All WLS category rows:", len(classified))
print("High-confidence animal rows:", len(animal_rows))
print("Unique animal subjects:", animal_rows["subject_id"].nunique())
print("One animal row per subject:", len(animal_one_per_subject))
print("Food rows:", len(food_rows))
print("Uncertain/empty rows:", len(uncertain_rows))

print()
print("Saved to:")
print(OUTPUT_DIR)

for path in sorted(OUTPUT_DIR.glob("*.csv")):
    print(" -", path.name)

In [ ]:
# ============================================================
# CLEAN PITT ANIMAL RESPONSES + BUILD FINAL BALANCED DATASET
# ============================================================

!pip -q install nltk

import re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import nltk

from nltk.corpus import wordnet as wn
from nltk.stem import WordNetLemmatizer

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)


# ============================================================
# PATHS
# ============================================================

PROCESSED_DIR = Path("/content/drive/MyDrive/Fluency_Processed")

PITT_PATH = PROCESSED_DIR / "category_fluency_all.csv"

WLS_PATH = (
    PROCESSED_DIR
    / "WLS_Category_Filtered"
    / "wls_animal_fluency_one_per_subject.csv"
)

OUTPUT_DIR = PROCESSED_DIR / "Category_Final"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# SETTINGS
# ============================================================

RANDOM_SEED = 42

# Use one recording per subject for the cleanest first experiment.
ONE_PITT_RECORDING_PER_SUBJECT = True

# Controls sampled to equal the number of AD subjects.
BALANCE_CONTROLS_TO_AD = True

MIN_VALID_ANIMALS = 3


# ============================================================
# MANUAL ANIMAL LEXICON
# ============================================================

ANIMAL_WORDS = {
    "aardvark", "alligator", "alpaca", "ant", "anteater", "antelope",
    "ape", "armadillo", "baboon", "badger", "bat", "bear", "beaver",
    "bee", "bird", "bison", "bobcat", "buffalo", "bull", "bunny",
    "butterfly", "calf", "camel", "canary", "caribou", "cat",
    "caterpillar", "chicken", "chimpanzee", "chipmunk", "cobra",
    "cod", "colt", "cougar", "cow", "coyote", "crab", "crocodile",
    "crow", "deer", "dingo", "dog", "dolphin", "donkey", "dove",
    "duck", "eagle", "eel", "elephant", "elk", "emu", "falcon",
    "ferret", "fish", "flamingo", "fly", "fox", "frog", "gazelle",
    "giraffe", "goat", "goose", "gorilla", "grasshopper", "hamster",
    "hare", "hawk", "hedgehog", "hen", "hippo", "hippopotamus",
    "hog", "horse", "hummingbird", "hyena", "iguana", "insect",
    "jackal", "jaguar", "kangaroo", "kitten", "koala", "lamb",
    "leopard", "lion", "lizard", "llama", "lobster", "lynx",
    "mink", "mole", "monkey", "moose", "mosquito", "moth",
    "mouse", "mule", "muskrat", "octopus", "ocelot", "opossum",
    "orangutan", "ostrich", "otter", "owl", "ox", "panda",
    "panther", "parrot", "peacock", "pelican", "penguin",
    "pheasant", "pig", "pigeon", "pony", "porcupine", "possum",
    "puma", "puppy", "rabbit", "raccoon", "ram", "rat", "raven",
    "reindeer", "rhino", "rhinoceros", "rooster", "salmon",
    "seal", "shark", "sheep", "skunk", "snake", "sparrow",
    "spider", "squirrel", "swan", "tiger", "toad", "tortoise",
    "toucan", "trout", "turkey", "turtle", "walrus", "weasel",
    "whale", "wolf", "worm", "zebra",

    # Common variants
    "sealion", "sea_lion", "polar_bear", "mountain_lion",
    "guinea_pig", "prairie_dog", "black_bear", "brown_bear",
    "grizzly_bear", "rattle_snake", "rattlesnake"
}


# ============================================================
# WORDS THAT MUST NEVER BE COUNTED AS ANIMALS
# ============================================================

BLOCKED_WORDS = {
    "oh", "okay", "ok", "well", "yes", "yeah", "no",
    "gee", "whiz", "jeez", "really", "right",
    "think", "thought", "know", "remember", "guess",
    "cant", "cannot", "dont", "didnt", "any",
    "more", "many", "else", "another",
    "people", "person", "man", "woman",
    "zoo", "farm", "jungle", "forest",
    "pencil", "paper", "throat",
    "pg", "hos", "til", "until", "hadta",
    "give", "wait", "have", "can", "get",
    "those", "that", "what", "how",
    "animal", "animals"
}


# ============================================================
# NORMALIZATION
# ============================================================

lemmatizer = WordNetLemmatizer()


def normalize_word(value):
    if pd.isna(value):
        return ""

    value = str(value).lower().strip()
    value = value.lstrip("@!?;")
    value = value.replace("-", "_")
    value = value.replace(" ", "_")
    value = re.sub(r"[^a-z_']", "", value)
    value = re.sub(r"_+", "_", value).strip("_'")

    return value


def singularize(word):
    word = normalize_word(word)

    if not word:
        return ""

    # WordNet lemmatizer handles dogs -> dog, horses -> horse, etc.
    result = lemmatizer.lemmatize(word, pos="n")

    # Some common irregulars
    irregular = {
        "mice": "mouse",
        "geese": "goose",
        "wolves": "wolf",
        "calves": "calf",
        "oxen": "ox",
        "sheep": "sheep",
        "deer": "deer",
        "fish": "fish",
        "people": "person",
        "puppies": "puppy",
        "kittens": "kitten",
        "ponies": "pony"
    }

    return irregular.get(word, result)


# ============================================================
# WORDNET ANIMAL CHECK
# ============================================================

ANIMAL_SYNSET = wn.synset("animal.n.01")


def is_wordnet_animal(word):
    word = singularize(word)

    if not word or word in BLOCKED_WORDS:
        return False

    lookup = word.replace("_", " ")

    synsets = wn.synsets(lookup, pos=wn.NOUN)

    for synset in synsets:
        for path in synset.hypernym_paths():
            if ANIMAL_SYNSET in path:
                return True

    return False


def is_valid_animal(word):
    normalized = normalize_word(word)
    singular = singularize(normalized)

    if not normalized or singular in BLOCKED_WORDS:
        return False

    if normalized in ANIMAL_WORDS:
        return True

    if singular in ANIMAL_WORDS:
        return True

    return is_wordnet_animal(singular)


# ============================================================
# EXTRACT VALID ANIMALS
# ============================================================

def extract_valid_animals(text):
    if pd.isna(text):
        return []

    tokens = re.findall(
        r"[A-Za-z]+(?:[_-][A-Za-z]+)*",
        str(text)
    )

    valid = []

    for token in tokens:
        normalized = normalize_word(token)
        singular = singularize(normalized)

        if is_valid_animal(singular):
            valid.append(singular)

    return valid


def calculate_features(items):
    counts = Counter(items)
    unique_ordered = list(dict.fromkeys(items))

    return {
        "clean_items_text": " ".join(items),
        "clean_total_items": len(items),
        "clean_unique_items": len(unique_ordered),
        "clean_repetition_count": sum(
            max(0, count - 1)
            for count in counts.values()
        ),
        "clean_type_token_ratio": (
            len(unique_ordered) / len(items)
            if items
            else np.nan
        )
    }


# ============================================================
# LOAD DATA
# ============================================================

if not PITT_PATH.exists():
    raise FileNotFoundError(PITT_PATH)

if not WLS_PATH.exists():
    raise FileNotFoundError(WLS_PATH)

pitt = pd.read_csv(PITT_PATH)
wls = pd.read_csv(WLS_PATH)

pitt = pitt[
    (pitt["dataset"] == "Pitt")
    & (pitt["task"] == "category")
].copy()

print("Pitt category rows loaded:", len(pitt))
print("WLS animal-control rows loaded:", len(wls))


# ============================================================
# CLEAN PITT RESPONSES
# ============================================================

clean_results = []

for text in pitt["response_text"]:
    items = extract_valid_animals(text)
    clean_results.append(calculate_features(items))

clean_results = pd.DataFrame(clean_results)

pitt = pd.concat(
    [
        pitt.reset_index(drop=True),
        clean_results.reset_index(drop=True)
    ],
    axis=1
)


# ============================================================
# FLAG DIFFERENCES
# ============================================================

pitt["removed_item_count"] = (
    pd.to_numeric(
        pitt["total_items"],
        errors="coerce"
    ).fillna(0)
    - pitt["clean_total_items"]
)

pitt["cleaning_removed_fraction"] = np.where(
    pd.to_numeric(pitt["total_items"], errors="coerce") > 0,
    pitt["removed_item_count"]
    / pd.to_numeric(pitt["total_items"], errors="coerce"),
    np.nan
)

pitt["valid_for_model"] = (
    pitt["clean_total_items"] >= MIN_VALID_ANIMALS
)


# ============================================================
# SHOW CLEANING EXAMPLES
# ============================================================

review_columns = [
    "recording_id",
    "subject_id",
    "diagnosis_group",
    "response_text",
    "items_text",
    "clean_items_text",
    "total_items",
    "clean_total_items",
    "unique_items",
    "clean_unique_items",
    "removed_item_count",
    "cleaning_removed_fraction"
]

print()
print("=" * 100)
print("RANDOM CLEANING REVIEW")
print("=" * 100)

display(
    pitt.sample(
        n=min(40, len(pitt)),
        random_state=RANDOM_SEED
    )[review_columns]
)


# ============================================================
# CLEANING SUMMARY
# ============================================================

cleaning_summary = (
    pitt.groupby(
        "diagnosis_group",
        dropna=False
    )
    .agg(
        recordings=("recording_id", "nunique"),
        subjects=("subject_id", "nunique"),
        median_original_items=("total_items", "median"),
        median_clean_items=("clean_total_items", "median"),
        median_clean_unique=("clean_unique_items", "median"),
        median_removed_fraction=("cleaning_removed_fraction", "median"),
        invalid_short_rows=(
            "valid_for_model",
            lambda values: int((~values).sum())
        )
    )
    .reset_index()
)

print()
print("=" * 100)
print("PITT CLEANING SUMMARY")
print("=" * 100)
display(cleaning_summary)


# ============================================================
# PREPARE PITT AD
# ============================================================

pitt_ad = pitt[
    (pitt["diagnosis_group"] == "AD")
    & (pitt["binary_label"] == 1)
    & (pitt["valid_for_model"])
].copy()

# For a first experiment, use one visit per person.
# Prefer the visit with the greatest number of valid unique animals.
if ONE_PITT_RECORDING_PER_SUBJECT:
    pitt_ad = (
        pitt_ad.sort_values(
            [
                "subject_id",
                "clean_unique_items",
                "clean_total_items",
                "visit"
            ],
            ascending=[
                True,
                False,
                False,
                True
            ]
        )
        .drop_duplicates(
            subset=["subject_id"],
            keep="first"
        )
        .reset_index(drop=True)
    )


# ============================================================
# PREPARE WLS CONTROLS
# ============================================================

wls_controls = wls.copy()

wls_controls["binary_label"] = 0
wls_controls["diagnosis_group"] = "Control"
wls_controls["dataset"] = "WLS"
wls_controls["task"] = "category"
wls_controls["task_prompt"] = "animals"

# Align WLS columns to clean Pitt column names.
wls_controls["clean_items_text"] = wls_controls[
    "items_text"
]

wls_controls["clean_total_items"] = pd.to_numeric(
    wls_controls["total_items"],
    errors="coerce"
)

wls_controls["clean_unique_items"] = (
    wls_controls["clean_items_text"]
    .fillna("")
    .apply(
        lambda text: len(
            set(str(text).split())
        )
    )
)

wls_controls["clean_repetition_count"] = (
    wls_controls["clean_total_items"]
    - wls_controls["clean_unique_items"]
)

wls_controls["clean_type_token_ratio"] = np.where(
    wls_controls["clean_total_items"] > 0,
    wls_controls["clean_unique_items"]
    / wls_controls["clean_total_items"],
    np.nan
)

wls_controls = wls_controls[
    wls_controls["clean_total_items"] >= MIN_VALID_ANIMALS
].copy()


# ============================================================
# SAMPLE CONTROLS AT SUBJECT LEVEL
# ============================================================

number_of_ad_subjects = pitt_ad["subject_id"].nunique()

if BALANCE_CONTROLS_TO_AD:
    wls_controls = wls_controls.sample(
        n=min(
            number_of_ad_subjects,
            len(wls_controls)
        ),
        random_state=RANDOM_SEED
    ).reset_index(drop=True)


# ============================================================
# STANDARDIZE FINAL COLUMNS
# ============================================================

final_columns = [
    "dataset",
    "subject_id",
    "recording_id",
    "diagnosis_group",
    "binary_label",
    "task",
    "task_prompt",
    "clean_items_text",
    "clean_total_items",
    "clean_unique_items",
    "clean_repetition_count",
    "clean_type_token_ratio",
    "items_0_15s",
    "items_15_30s",
    "items_30_45s",
    "items_45_60s"
]

for column in final_columns:
    if column not in pitt_ad.columns:
        pitt_ad[column] = np.nan

    if column not in wls_controls.columns:
        wls_controls[column] = np.nan


pitt_final = pitt_ad[final_columns].copy()
wls_final = wls_controls[final_columns].copy()

final_balanced = pd.concat(
    [
        pitt_final,
        wls_final
    ],
    ignore_index=True
)

final_balanced = final_balanced.sample(
    frac=1,
    random_state=RANDOM_SEED
).reset_index(drop=True)


# ============================================================
# FINAL INTEGRITY CHECKS
# ============================================================

duplicate_recordings = final_balanced[
    final_balanced.duplicated(
        subset=["recording_id"],
        keep=False
    )
]

subject_label_conflicts = (
    final_balanced.groupby(
        "subject_id"
    )["binary_label"]
    .nunique()
)

subject_label_conflicts = subject_label_conflicts[
    subject_label_conflicts > 1
]

empty_clean_text = final_balanced[
    final_balanced["clean_items_text"]
    .fillna("")
    .str.strip()
    .eq("")
]

if len(duplicate_recordings) > 0:
    raise RuntimeError(
        f"Duplicate recordings found: {len(duplicate_recordings)}"
    )

if len(subject_label_conflicts) > 0:
    raise RuntimeError(
        "Subjects with conflicting labels found."
    )

if len(empty_clean_text) > 0:
    raise RuntimeError(
        f"Empty cleaned responses found: {len(empty_clean_text)}"
    )


# ============================================================
# FINAL SUMMARY
# ============================================================

final_summary = (
    final_balanced.groupby(
        [
            "dataset",
            "diagnosis_group",
            "binary_label"
        ]
    )
    .agg(
        rows=("recording_id", "size"),
        subjects=("subject_id", "nunique"),
        median_total_items=("clean_total_items", "median"),
        median_unique_items=("clean_unique_items", "median"),
        median_repetitions=("clean_repetition_count", "median")
    )
    .reset_index()
)

print()
print("=" * 100)
print("FINAL BALANCED DATASET SUMMARY")
print("=" * 100)

display(final_summary)

print()
print("Total rows:", len(final_balanced))
print(
    "Total subjects:",
    final_balanced["subject_id"].nunique()
)

print(
    "AD subjects:",
    final_balanced.loc[
        final_balanced["binary_label"] == 1,
        "subject_id"
    ].nunique()
)

print(
    "Control subjects:",
    final_balanced.loc[
        final_balanced["binary_label"] == 0,
        "subject_id"
    ].nunique()
)


# ============================================================
# SAVE OUTPUTS
# ============================================================

pitt.to_csv(
    OUTPUT_DIR / "pitt_category_cleaned_all.csv",
    index=False
)

pitt[
    ~pitt["valid_for_model"]
].to_csv(
    OUTPUT_DIR / "pitt_category_rejected_short.csv",
    index=False
)

pitt.sort_values(
    "cleaning_removed_fraction",
    ascending=False
).head(100).to_csv(
    OUTPUT_DIR / "pitt_category_high_removal_review.csv",
    index=False
)

pitt_final.to_csv(
    OUTPUT_DIR / "pitt_AD_one_recording_per_subject.csv",
    index=False
)

wls_final.to_csv(
    OUTPUT_DIR / "wls_controls_sampled.csv",
    index=False
)

final_balanced.to_csv(
    OUTPUT_DIR / "category_final_balanced.csv",
    index=False
)

final_summary.to_csv(
    OUTPUT_DIR / "category_final_summary.csv",
    index=False
)

print()
print("Saved files:")

for path in sorted(OUTPUT_DIR.glob("*.csv")):
    print(" -", path)

print()
print("NEXT MODELING FILE:")
print(
    OUTPUT_DIR / "category_final_balanced.csv"
)

In [ ]:
# ============================================================
# CATEGORY FLUENCY BASELINE
# Subject-independent repeated cross-validation
# Uses only robust numeric fluency features
# ============================================================

!pip -q install xgboost

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    RepeatedStratifiedKFold,
    cross_validate,
    cross_val_predict,
    StratifiedKFold
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

from xgboost import XGBClassifier


# ============================================================
# PATHS
# ============================================================

DATA_PATH = Path(
    "/content/drive/MyDrive/Fluency_Processed/"
    "Category_Final/category_final_balanced.csv"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Fluency_Processed/"
    "Category_Model_Baseline"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# LOAD DATA
# ============================================================

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}"
    )

data = pd.read_csv(DATA_PATH)

required_columns = [
    "subject_id",
    "dataset",
    "diagnosis_group",
    "binary_label",
    "clean_total_items",
    "clean_unique_items",
    "clean_repetition_count",
    "clean_type_token_ratio"
]

missing_columns = [
    column
    for column in required_columns
    if column not in data.columns
]

if missing_columns:
    raise ValueError(
        "Missing required columns: "
        + ", ".join(missing_columns)
    )

data = data[
    data["binary_label"].isin([0, 1])
].copy()

data["binary_label"] = (
    pd.to_numeric(
        data["binary_label"],
        errors="raise"
    )
    .astype(int)
)

numeric_columns = [
    "clean_total_items",
    "clean_unique_items",
    "clean_repetition_count",
    "clean_type_token_ratio"
]

for column in numeric_columns:
    data[column] = pd.to_numeric(
        data[column],
        errors="coerce"
    )


# ============================================================
# SUBJECT AND DUPLICATE CHECKS
# ============================================================

if data["recording_id"].duplicated().any():
    duplicate_ids = data.loc[
        data["recording_id"].duplicated(keep=False),
        "recording_id"
    ].tolist()

    raise ValueError(
        "Duplicate recording IDs found: "
        + ", ".join(duplicate_ids[:20])
    )

subject_label_counts = (
    data.groupby("subject_id")["binary_label"]
    .nunique()
)

conflicting_subjects = subject_label_counts[
    subject_label_counts > 1
].index.tolist()

if conflicting_subjects:
    raise ValueError(
        "Subjects with conflicting labels found: "
        + ", ".join(conflicting_subjects[:20])
    )

if data["subject_id"].duplicated().any():
    duplicate_subjects = data.loc[
        data["subject_id"].duplicated(keep=False),
        "subject_id"
    ].unique()

    raise ValueError(
        "This baseline expects one row per subject. "
        "Duplicate subjects found: "
        + ", ".join(duplicate_subjects[:20])
    )


# ============================================================
# FEATURES AND LABELS
# ============================================================

feature_columns = [
    "clean_total_items",
    "clean_unique_items",
    "clean_repetition_count",
    "clean_type_token_ratio"
]

X = data[feature_columns].copy()
y = data["binary_label"].copy()


# ============================================================
# DATASET SUMMARY
# ============================================================

print("=" * 100)
print("DATASET SUMMARY")
print("=" * 100)

print("Rows:", len(data))
print("Subjects:", data["subject_id"].nunique())

print("\nClass counts:")
print(
    y.value_counts()
    .sort_index()
    .rename(
        index={
            0: "Control",
            1: "AD"
        }
    )
)

print("\nDataset and class counts:")
display(
    pd.crosstab(
        data["dataset"],
        data["binary_label"]
    ).rename(
        columns={
            0: "Control",
            1: "AD"
        }
    )
)

print("\nMissing feature values:")
print(X.isna().sum())

print("\nFeature summary by diagnosis:")
display(
    data.groupby(
        "diagnosis_group"
    )[feature_columns]
    .agg(
        [
            "mean",
            "median",
            "std",
            "min",
            "max"
        ]
    )
)


# ============================================================
# DEFINE MODELS
# ============================================================

models = {
    "Logistic Regression": Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "classifier",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=5000,
                    random_state=42
                )
            )
        ]
    ),

    "Random Forest": Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "classifier",
                RandomForestClassifier(
                    n_estimators=500,
                    max_depth=4,
                    min_samples_split=10,
                    min_samples_leaf=5,
                    class_weight="balanced",
                    random_state=42,
                    n_jobs=-1
                )
            )
        ]
    ),

    "XGBoost": Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "classifier",
                XGBClassifier(
                    n_estimators=300,
                    max_depth=3,
                    learning_rate=0.03,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    min_child_weight=3,
                    reg_alpha=0.5,
                    reg_lambda=2.0,
                    objective="binary:logistic",
                    eval_metric="logloss",
                    random_state=42,
                    n_jobs=-1
                )
            )
        ]
    )
}


# ============================================================
# REPEATED CROSS-VALIDATION
# ============================================================

repeated_cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "average_precision": "average_precision"
}

cross_validation_rows = []

print("\n" + "=" * 100)
print("REPEATED 5-FOLD CROSS-VALIDATION")
print("=" * 100)

for model_name, model in models.items():
    print(f"\nEvaluating: {model_name}")

    scores = cross_validate(
        estimator=model,
        X=X,
        y=y,
        cv=repeated_cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False,
        error_score="raise"
    )

    result = {
        "model": model_name
    }

    for metric_name in scoring:
        metric_values = scores[
            f"test_{metric_name}"
        ]

        result[
            f"{metric_name}_mean"
        ] = metric_values.mean()

        result[
            f"{metric_name}_std"
        ] = metric_values.std()

    cross_validation_rows.append(result)

cross_validation_results = pd.DataFrame(
    cross_validation_rows
).sort_values(
    "roc_auc_mean",
    ascending=False
).reset_index(drop=True)

display(cross_validation_results)

cross_validation_results.to_csv(
    OUTPUT_DIR / "repeated_cross_validation_results.csv",
    index=False
)


# ============================================================
# SELECT BEST MODEL
# ============================================================

best_model_name = cross_validation_results.loc[
    0,
    "model"
]

best_model = models[best_model_name]

print(
    "\nBest model by mean ROC-AUC:",
    best_model_name
)


# ============================================================
# OUT-OF-FOLD PREDICTIONS
# ============================================================

evaluation_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=123
)

predicted_probabilities = cross_val_predict(
    estimator=best_model,
    X=X,
    y=y,
    cv=evaluation_cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

DEFAULT_THRESHOLD = 0.50

predicted_labels = (
    predicted_probabilities
    >= DEFAULT_THRESHOLD
).astype(int)


# ============================================================
# METRICS
# ============================================================

accuracy = accuracy_score(
    y,
    predicted_labels
)

balanced_accuracy = balanced_accuracy_score(
    y,
    predicted_labels
)

precision = precision_score(
    y,
    predicted_labels,
    zero_division=0
)

recall = recall_score(
    y,
    predicted_labels,
    zero_division=0
)

f1 = f1_score(
    y,
    predicted_labels,
    zero_division=0
)

roc_auc = roc_auc_score(
    y,
    predicted_probabilities
)

average_precision = average_precision_score(
    y,
    predicted_probabilities
)

mcc = matthews_corrcoef(
    y,
    predicted_labels
)

confusion = confusion_matrix(
    y,
    predicted_labels,
    labels=[0, 1]
)

tn, fp, fn, tp = confusion.ravel()

specificity = (
    tn / (tn + fp)
    if (tn + fp) > 0
    else np.nan
)

metric_rows = [
    {
        "metric": "Accuracy",
        "value": accuracy
    },
    {
        "metric": "Balanced accuracy",
        "value": balanced_accuracy
    },
    {
        "metric": "Precision",
        "value": precision
    },
    {
        "metric": "Recall/Sensitivity",
        "value": recall
    },
    {
        "metric": "Specificity",
        "value": specificity
    },
    {
        "metric": "F1 score",
        "value": f1
    },
    {
        "metric": "ROC-AUC",
        "value": roc_auc
    },
    {
        "metric": "Average precision",
        "value": average_precision
    },
    {
        "metric": "MCC",
        "value": mcc
    },
    {
        "metric": "TN",
        "value": tn
    },
    {
        "metric": "FP",
        "value": fp
    },
    {
        "metric": "FN",
        "value": fn
    },
    {
        "metric": "TP",
        "value": tp
    }
]

metrics_df = pd.DataFrame(metric_rows)

print("\n" + "=" * 100)
print("OUT-OF-FOLD PERFORMANCE")
print("=" * 100)

display(metrics_df)

print(
    classification_report(
        y,
        predicted_labels,
        target_names=[
            "Control",
            "AD"
        ],
        digits=4,
        zero_division=0
    )
)

metrics_df.to_csv(
    OUTPUT_DIR / "out_of_fold_metrics.csv",
    index=False
)


# ============================================================
# SAVE OUT-OF-FOLD PREDICTIONS
# ============================================================

predictions = data[
    [
        "subject_id",
        "recording_id",
        "dataset",
        "diagnosis_group",
        "binary_label"
    ]
].copy()

predictions[
    "predicted_probability_AD"
] = predicted_probabilities

predictions[
    "predicted_label"
] = predicted_labels

predictions[
    "correct"
] = (
    predictions["predicted_label"]
    == predictions["binary_label"]
)

predictions.to_csv(
    OUTPUT_DIR / "out_of_fold_predictions.csv",
    index=False
)


# ============================================================
# ROC CURVE
# ============================================================

RocCurveDisplay.from_predictions(
    y_true=y,
    y_pred=predicted_probabilities
)

plt.title(
    f"Category Fluency ROC Curve\n{best_model_name}"
)

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "roc_curve.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()
plt.close()


# ============================================================
# PRECISION-RECALL CURVE
# ============================================================

PrecisionRecallDisplay.from_predictions(
    y_true=y,
    y_pred=predicted_probabilities
)

plt.title(
    f"Category Fluency Precision-Recall Curve\n"
    f"{best_model_name}"
)

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "precision_recall_curve.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()
plt.close()


# ============================================================
# CONFUSION MATRIX PLOT
# ============================================================

plt.figure(figsize=(6, 5))

plt.imshow(confusion)

plt.title(
    f"Confusion Matrix\n{best_model_name}"
)

plt.xticks(
    ticks=[0, 1],
    labels=[
        "Control",
        "AD"
    ]
)

plt.yticks(
    ticks=[0, 1],
    labels=[
        "Control",
        "AD"
    ]
)

plt.xlabel("Predicted label")
plt.ylabel("True label")

for row_index in range(2):
    for column_index in range(2):
        plt.text(
            column_index,
            row_index,
            confusion[
                row_index,
                column_index
            ],
            ha="center",
            va="center"
        )

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "confusion_matrix.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()
plt.close()


# ============================================================
# TRAIN FINAL MODEL ON ALL DATA
# ============================================================

best_model.fit(
    X,
    y
)

import joblib

model_package = {
    "model": best_model,
    "model_name": best_model_name,
    "feature_columns": feature_columns,
    "threshold": DEFAULT_THRESHOLD,
    "training_rows": len(data),
    "training_subjects": data[
        "subject_id"
    ].nunique()
}

joblib.dump(
    model_package,
    OUTPUT_DIR / "category_fluency_baseline.joblib"
)


# ============================================================
# FEATURE IMPORTANCE
# ============================================================

feature_importance = None

classifier = best_model.named_steps[
    "classifier"
]

if best_model_name == "Logistic Regression":
    coefficients = classifier.coef_[0]

    feature_importance = pd.DataFrame({
        "feature": feature_columns,
        "importance": coefficients,
        "absolute_importance": np.abs(
            coefficients
        )
    }).sort_values(
        "absolute_importance",
        ascending=False
    )

elif best_model_name in {
    "Random Forest",
    "XGBoost"
}:
    importance_values = (
        classifier.feature_importances_
    )

    feature_importance = pd.DataFrame({
        "feature": feature_columns,
        "importance": importance_values,
        "absolute_importance": np.abs(
            importance_values
        )
    }).sort_values(
        "absolute_importance",
        ascending=False
    )

if feature_importance is not None:
    print("\nFeature importance:")
    display(feature_importance)

    feature_importance.to_csv(
        OUTPUT_DIR / "feature_importance.csv",
        index=False
    )


# ============================================================
# FINAL REPORT
# ============================================================

print("\n" + "=" * 100)
print("CATEGORY FLUENCY BASELINE COMPLETE")
print("=" * 100)

print("Best model:", best_model_name)
print(f"Threshold:          {DEFAULT_THRESHOLD:.2f}")
print(f"Accuracy:           {accuracy:.4f}")
print(f"Balanced accuracy:  {balanced_accuracy:.4f}")
print(f"Precision:          {precision:.4f}")
print(f"Recall/Sensitivity: {recall:.4f}")
print(f"Specificity:        {specificity:.4f}")
print(f"F1 score:           {f1:.4f}")
print(f"ROC-AUC:            {roc_auc:.4f}")
print(f"Average precision:  {average_precision:.4f}")
print(f"MCC:                {mcc:.4f}")

print("\nConfusion matrix values:")
print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)

print("\nSaved files:")

for output_path in sorted(
    OUTPUT_DIR.iterdir()
):
    print(" -", output_path.name)

print("\nOutput folder:")
print(OUTPUT_DIR)

In [ ]:
# ============================================================
# FEATURE-SET ABLATION TEST
# ============================================================

feature_sets = {
    "Total only": [
        "clean_total_items"
    ],

    "Unique only": [
        "clean_unique_items"
    ],

    "Total + unique": [
        "clean_total_items",
        "clean_unique_items"
    ],

    "Repetition only": [
        "clean_repetition_count",
        "clean_type_token_ratio"
    ],

    "All features": [
        "clean_total_items",
        "clean_unique_items",
        "clean_repetition_count",
        "clean_type_token_ratio"
    ]
}

repeated_cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "average_precision": "average_precision"
}

ablation_rows = []

for feature_set_name, columns in feature_sets.items():

    print("\n" + "=" * 100)
    print("FEATURE SET:", feature_set_name)
    print("FEATURES:", columns)
    print("=" * 100)

    X_subset = data[columns].copy()

    model = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "classifier",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=5000,
                    random_state=42
                )
            )
        ]
    )

    scores = cross_validate(
        estimator=model,
        X=X_subset,
        y=y,
        cv=repeated_cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False,
        error_score="raise"
    )

    row = {
        "feature_set": feature_set_name,
        "features": ", ".join(columns)
    }

    for metric_name in scoring:
        values = scores[f"test_{metric_name}"]

        row[f"{metric_name}_mean"] = values.mean()
        row[f"{metric_name}_std"] = values.std()

    ablation_rows.append(row)

ablation_results = pd.DataFrame(
    ablation_rows
).sort_values(
    "roc_auc_mean",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 100)
print("ABLATION RESULTS")
print("=" * 100)

display(ablation_results)

ablation_results.to_csv(
    OUTPUT_DIR / "feature_set_ablation_results.csv",
    index=False
)

In [ ]:
# ============================================================
# RESTORE WLS 15-SECOND TIMING COLUMNS
# ============================================================

import re
from pathlib import Path

import numpy as np
import pandas as pd


SOURCE_DIR = Path(
    "/content/fluency_source/WLS/Category_Fluency"
)

FILTERED_DIR = Path(
    "/content/drive/MyDrive/Fluency_Processed/"
    "WLS_Category_Filtered"
)

FILTERED_PATH = (
    FILTERED_DIR
    / "wls_animal_fluency_one_per_subject.csv"
)

OUTPUT_PATH = (
    FILTERED_DIR
    / "wls_animal_fluency_one_per_subject_with_timing.csv"
)


if not FILTERED_PATH.exists():
    raise FileNotFoundError(
        f"Filtered WLS file not found: {FILTERED_PATH}"
    )


def normalize_id(value):
    if pd.isna(value):
        return None

    try:
        return str(int(value))
    except Exception:
        return str(value).strip()


def get_wave(path):
    match = re.search(
        r"(03|11)",
        path.stem
    )

    if not match:
        raise ValueError(
            f"Could not determine wave from {path.name}"
        )

    return 2000 + int(
        match.group(1)
    )


def response_column_number(column):
    match = re.search(
        r"_(\d+)$",
        str(column)
    )

    return (
        int(match.group(1))
        if match
        else 999
    )


def find_bin_columns(columns, prefix):
    matching = [
        column
        for column in columns
        if str(column).upper().startswith(
            prefix
        )
        and re.search(
            r"_\d+$",
            str(column)
        )
    ]

    return sorted(
        matching,
        key=response_column_number
    )


all_timing_rows = []

excel_files = sorted(
    list(SOURCE_DIR.glob("CogCat*.xls"))
    + list(SOURCE_DIR.glob("CogCat*.xlsx"))
)

if not excel_files:
    raise FileNotFoundError(
        f"No CogCat spreadsheets found in {SOURCE_DIR}"
    )


for path in excel_files:
    print("Reading:", path.name)

    wave = get_wave(path)
    df = pd.read_excel(path)

    required = ["idtlkbnk"]

    for column in required:
        if column not in df.columns:
            raise ValueError(
                f"{path.name} is missing {column}"
            )

    first_columns = find_bin_columns(
        df.columns,
        "FIR15C_"
    )

    second_columns = find_bin_columns(
        df.columns,
        "SEC15C_"
    )

    third_columns = find_bin_columns(
        df.columns,
        "THI15C_"
    )

    fourth_columns = find_bin_columns(
        df.columns,
        "FOR15C_"
    )

    if not all([
        first_columns,
        second_columns,
        third_columns,
        fourth_columns
    ]):
        raise ValueError(
            f"Timing columns were not found correctly in {path.name}"
        )

    for _, row in df.iterrows():
        source_subject_id = normalize_id(
            row["idtlkbnk"]
        )

        if source_subject_id is None:
            continue

        def count_nonempty(columns):
            count = 0

            for column in columns:
                value = row[column]

                if pd.isna(value):
                    continue

                if str(value).strip():
                    count += 1

            return count

        all_timing_rows.append({
            "source_subject_id": source_subject_id,
            "wave": wave,
            "recording_id": (
                f"WLS_{source_subject_id}_"
                f"{wave}_category"
            ),
            "items_0_15s": count_nonempty(
                first_columns
            ),
            "items_15_30s": count_nonempty(
                second_columns
            ),
            "items_30_45s": count_nonempty(
                third_columns
            ),
            "items_45_60s": count_nonempty(
                fourth_columns
            )
        })


timing = pd.DataFrame(
    all_timing_rows
)

if timing.empty:
    raise RuntimeError(
        "No WLS timing rows were created."
    )

if timing["recording_id"].duplicated().any():
    duplicate_ids = timing.loc[
        timing["recording_id"].duplicated(
            keep=False
        ),
        "recording_id"
    ].unique()

    raise RuntimeError(
        "Duplicate WLS timing recording IDs found: "
        + ", ".join(
            duplicate_ids[:20]
        )
    )


filtered = pd.read_csv(
    FILTERED_PATH
)

filtered["source_subject_id"] = (
    filtered["source_subject_id"]
    .apply(normalize_id)
)

filtered["wave"] = pd.to_numeric(
    filtered["wave"],
    errors="coerce"
).astype("Int64")


merged = filtered.merge(
    timing[
        [
            "recording_id",
            "items_0_15s",
            "items_15_30s",
            "items_30_45s",
            "items_45_60s"
        ]
    ],
    on="recording_id",
    how="left",
    validate="one_to_one"
)


timing_columns = [
    "items_0_15s",
    "items_15_30s",
    "items_30_45s",
    "items_45_60s"
]

missing_rows = merged[
    timing_columns
].isna().any(axis=1)

print()
print(
    "Filtered animal rows:",
    len(merged)
)

print(
    "Rows missing timing after merge:",
    int(missing_rows.sum())
)

if missing_rows.any():
    display(
        merged.loc[
            missing_rows,
            [
                "recording_id",
                "subject_id",
                "wave"
            ]
        ].head(30)
    )

    raise RuntimeError(
        "Some filtered WLS animal rows could not be matched "
        "to the original spreadsheets."
    )


for column in timing_columns:
    merged[column] = (
        pd.to_numeric(
            merged[column],
            errors="raise"
        )
        .astype(int)
    )


merged["timing_bin_sum"] = merged[
    timing_columns
].sum(axis=1)

merged["timing_total_difference"] = (
    merged["timing_bin_sum"]
    - pd.to_numeric(
        merged["total_items"],
        errors="coerce"
    )
)

mismatches = merged[
    merged["timing_total_difference"] != 0
].copy()

print(
    "Rows where WLS timing-bin sum differs from total_items:",
    len(mismatches)
)

merged.to_csv(
    OUTPUT_PATH,
    index=False
)

print()
print("Saved corrected WLS file:")
print(OUTPUT_PATH)

print()
print("Timing columns:")
display(
    merged[
        [
            "recording_id",
            "items_0_15s",
            "items_15_30s",
            "items_30_45s",
            "items_45_60s",
            "timing_bin_sum",
            "total_items"
        ]
    ].head(10)
)

In [ ]:
WLS_PATH = Path(
    "/content/drive/MyDrive/Fluency_Processed/"
    "WLS_Category_Filtered/"
    "wls_animal_fluency_one_per_subject_with_timing.csv"
)

In [ ]:
# ============================================================
# FIX WLS FILE IN PLACE BY ADDING 15-SECOND TIMING COLUMNS
# Then rerun the full timing preprocessing cell unchanged.
# ============================================================

import re
from pathlib import Path

import pandas as pd


SOURCE_DIR = Path(
    "/content/fluency_source/WLS/Category_Fluency"
)

WLS_FILE = Path(
    "/content/drive/MyDrive/Fluency_Processed/"
    "WLS_Category_Filtered/"
    "wls_animal_fluency_one_per_subject.csv"
)

BACKUP_FILE = Path(
    "/content/drive/MyDrive/Fluency_Processed/"
    "WLS_Category_Filtered/"
    "wls_animal_fluency_one_per_subject_before_timing.csv"
)


def normalize_subject_id(value):
    if pd.isna(value):
        return None

    try:
        return str(int(float(value)))
    except Exception:
        return str(value).strip()


def get_wave(path):
    match = re.search(r"(03|11)", path.stem)

    if not match:
        raise ValueError(
            f"Could not determine wave from {path.name}"
        )

    return 2000 + int(match.group(1))


def get_column_number(column):
    match = re.search(r"_(\d+)$", str(column))

    return int(match.group(1)) if match else 999


def get_bin_columns(columns, prefix):
    matching = [
        column
        for column in columns
        if str(column).upper().startswith(prefix)
        and re.search(r"_\d+$", str(column))
    ]

    return sorted(
        matching,
        key=get_column_number
    )


def count_nonempty(row, columns):
    count = 0

    for column in columns:
        value = row[column]

        if pd.isna(value):
            continue

        if str(value).strip():
            count += 1

    return count


if not WLS_FILE.exists():
    raise FileNotFoundError(
        f"Filtered WLS file not found: {WLS_FILE}"
    )

excel_files = sorted(
    list(SOURCE_DIR.glob("CogCat*.xls"))
    + list(SOURCE_DIR.glob("CogCat*.xlsx"))
)

if not excel_files:
    raise FileNotFoundError(
        f"No CogCat spreadsheets found in {SOURCE_DIR}"
    )


# ============================================================
# READ ORIGINAL SPREADSHEETS AND RECREATE TIMING COUNTS
# ============================================================

timing_rows = []

for path in excel_files:
    print("Reading:", path.name)

    wave = get_wave(path)
    dataframe = pd.read_excel(path)

    if "idtlkbnk" not in dataframe.columns:
        raise ValueError(
            f"{path.name} is missing idtlkbnk"
        )

    first_columns = get_bin_columns(
        dataframe.columns,
        "FIR15C_"
    )

    second_columns = get_bin_columns(
        dataframe.columns,
        "SEC15C_"
    )

    third_columns = get_bin_columns(
        dataframe.columns,
        "THI15C_"
    )

    fourth_columns = get_bin_columns(
        dataframe.columns,
        "FOR15C_"
    )

    if not all([
        first_columns,
        second_columns,
        third_columns,
        fourth_columns
    ]):
        raise ValueError(
            f"Could not find all four timing-bin column groups in {path.name}"
        )

    for _, row in dataframe.iterrows():
        source_subject_id = normalize_subject_id(
            row["idtlkbnk"]
        )

        if source_subject_id is None:
            continue

        recording_id = (
            f"WLS_{source_subject_id}_"
            f"{wave}_category"
        )

        timing_rows.append({
            "recording_id": recording_id,
            "items_0_15s": count_nonempty(
                row,
                first_columns
            ),
            "items_15_30s": count_nonempty(
                row,
                second_columns
            ),
            "items_30_45s": count_nonempty(
                row,
                third_columns
            ),
            "items_45_60s": count_nonempty(
                row,
                fourth_columns
            )
        })


timing = pd.DataFrame(timing_rows)

if timing.empty:
    raise RuntimeError(
        "No WLS timing rows were created."
    )

if timing["recording_id"].duplicated().any():
    duplicate_ids = timing.loc[
        timing["recording_id"].duplicated(keep=False),
        "recording_id"
    ].unique()

    raise RuntimeError(
        "Duplicate timing recording IDs found: "
        + ", ".join(duplicate_ids[:20])
    )


# ============================================================
# LOAD FILTERED ANIMAL FILE
# ============================================================

filtered = pd.read_csv(WLS_FILE)

print("\nFiltered animal rows:", len(filtered))

required_filtered_columns = [
    "recording_id",
    "subject_id",
    "wave",
    "total_items"
]

missing = [
    column
    for column in required_filtered_columns
    if column not in filtered.columns
]

if missing:
    raise ValueError(
        "Filtered WLS file is missing: "
        + ", ".join(missing)
    )


# Remove any old timing columns before merging.
timing_columns = [
    "items_0_15s",
    "items_15_30s",
    "items_30_45s",
    "items_45_60s"
]

filtered = filtered.drop(
    columns=[
        column
        for column in timing_columns
        if column in filtered.columns
    ],
    errors="ignore"
)


# ============================================================
# MERGE TIMING DATA
# ============================================================

merged = filtered.merge(
    timing,
    on="recording_id",
    how="left",
    validate="one_to_one"
)

missing_timing_mask = merged[
    timing_columns
].isna().any(axis=1)

print(
    "Rows missing timing after merge:",
    int(missing_timing_mask.sum())
)

if missing_timing_mask.any():
    display(
        merged.loc[
            missing_timing_mask,
            [
                "recording_id",
                "subject_id",
                "wave"
            ]
        ].head(50)
    )

    raise RuntimeError(
        "Some WLS animal rows did not match their original spreadsheet rows."
    )


for column in timing_columns:
    merged[column] = (
        pd.to_numeric(
            merged[column],
            errors="raise"
        )
        .astype(int)
    )


# ============================================================
# VALIDATE TIMING TOTALS
# ============================================================

merged["timing_bin_sum"] = merged[
    timing_columns
].sum(axis=1)

merged["timing_total_difference"] = (
    merged["timing_bin_sum"]
    - pd.to_numeric(
        merged["total_items"],
        errors="coerce"
    )
)

mismatch_count = int(
    (
        merged["timing_total_difference"]
        != 0
    ).sum()
)

print(
    "Rows where timing-bin sum differs from total_items:",
    mismatch_count
)


# ============================================================
# BACK UP OLD FILE AND OVERWRITE IT WITH CORRECTED VERSION
# ============================================================

if not BACKUP_FILE.exists():
    filtered.to_csv(
        BACKUP_FILE,
        index=False
    )

merged.to_csv(
    WLS_FILE,
    index=False
)


# ============================================================
# CONFIRM
# ============================================================

reloaded = pd.read_csv(WLS_FILE)

still_missing = [
    column
    for column in timing_columns
    if column not in reloaded.columns
]

if still_missing:
    raise RuntimeError(
        "Timing columns were not saved correctly: "
        + ", ".join(still_missing)
    )

print("\n" + "=" * 100)
print("WLS TIMING FIX COMPLETE")
print("=" * 100)

print("Corrected file:")
print(WLS_FILE)

print("\nConfirmed timing columns:")
print(timing_columns)

display(
    reloaded[
        [
            "recording_id",
            "items_0_15s",
            "items_15_30s",
            "items_30_45s",
            "items_45_60s",
            "timing_bin_sum",
            "total_items"
        ]
    ].head(10)
)

In [ ]:
# ============================================================
# CLEAN PITT 15-SECOND ANIMAL BINS
# BUILD TIMING-ENHANCED CATEGORY DATASET
# FULL CORRECTED VERSION
# ============================================================

import re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import nltk

from nltk.corpus import wordnet as wn
from nltk.stem import WordNetLemmatizer


# ============================================================
# DOWNLOAD NLTK DATA
# ============================================================

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)


# ============================================================
# PATHS
# ============================================================

SOURCE_ROOT = Path("/content/fluency_source")

PITT_CLEAN_PATH = Path(
    "/content/drive/MyDrive/Fluency_Processed/"
    "Category_Final/pitt_category_cleaned_all.csv"
)

WLS_PATH = Path(
    "/content/drive/MyDrive/Fluency_Processed/"
    "WLS_Category_Filtered/"
    "wls_animal_fluency_one_per_subject.csv"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Fluency_Processed/"
    "Category_Timing_Final"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RANDOM_SEED = 42
MIN_VALID_ANIMALS = 3


# ============================================================
# CHAT REGULAR EXPRESSIONS
# ============================================================

BULLET_RE = re.compile(
    r"\x15(\d+)_(\d+)\x15"
)

CHAT_CODE_RE = re.compile(
    r"\[[^\]]*\]"
)

ANGLE_RETRACE_RE = re.compile(
    r"<([^<>]*)>\s*\[/\]"
)

EVENT_RE = re.compile(
    r"&=[A-Za-z_]+"
)

FRAGMENT_RE = re.compile(
    r"&[-+][A-Za-z]+"
)

AT_SUFFIX_RE = re.compile(
    r"@[\w:]+"
)

NON_WORD_RE = re.compile(
    r"[^A-Za-z'\-\s]"
)

MULTISPACE_RE = re.compile(
    r"\s+"
)


# ============================================================
# ANIMAL LEXICON
# ============================================================

ANIMAL_WORDS = {
    "aardvark",
    "alligator",
    "alpaca",
    "ant",
    "anteater",
    "antelope",
    "ape",
    "armadillo",
    "baboon",
    "badger",
    "bat",
    "bear",
    "beaver",
    "bee",
    "bird",
    "bison",
    "bobcat",
    "buffalo",
    "bull",
    "bunny",
    "butterfly",
    "calf",
    "camel",
    "canary",
    "caribou",
    "cat",
    "caterpillar",
    "chicken",
    "chimpanzee",
    "chipmunk",
    "cobra",
    "cod",
    "colt",
    "cougar",
    "cow",
    "coyote",
    "crab",
    "crocodile",
    "crow",
    "deer",
    "dingo",
    "dog",
    "dolphin",
    "donkey",
    "dove",
    "duck",
    "eagle",
    "eel",
    "elephant",
    "elk",
    "emu",
    "falcon",
    "ferret",
    "fish",
    "flamingo",
    "fly",
    "fox",
    "frog",
    "gazelle",
    "giraffe",
    "goat",
    "goose",
    "gorilla",
    "grasshopper",
    "hamster",
    "hare",
    "hawk",
    "hedgehog",
    "hen",
    "hippo",
    "hippopotamus",
    "hog",
    "horse",
    "hummingbird",
    "hyena",
    "iguana",
    "insect",
    "jackal",
    "jaguar",
    "kangaroo",
    "kitten",
    "koala",
    "lamb",
    "leopard",
    "lion",
    "lizard",
    "llama",
    "lobster",
    "lynx",
    "mink",
    "mole",
    "monkey",
    "moose",
    "mosquito",
    "moth",
    "mouse",
    "mule",
    "muskrat",
    "octopus",
    "ocelot",
    "opossum",
    "orangutan",
    "ostrich",
    "otter",
    "owl",
    "ox",
    "panda",
    "panther",
    "parrot",
    "peacock",
    "pelican",
    "penguin",
    "pheasant",
    "pig",
    "pigeon",
    "pony",
    "porcupine",
    "possum",
    "puma",
    "puppy",
    "rabbit",
    "raccoon",
    "ram",
    "rat",
    "raven",
    "reindeer",
    "rhino",
    "rhinoceros",
    "rooster",
    "salmon",
    "seal",
    "shark",
    "sheep",
    "skunk",
    "snake",
    "sparrow",
    "spider",
    "squirrel",
    "swan",
    "tiger",
    "toad",
    "tortoise",
    "toucan",
    "trout",
    "turkey",
    "turtle",
    "walrus",
    "weasel",
    "whale",
    "wolf",
    "worm",
    "zebra",

    "sealion",
    "sea_lion",
    "polar_bear",
    "mountain_lion",
    "guinea_pig",
    "prairie_dog",
    "black_bear",
    "brown_bear",
    "grizzly_bear",
    "rattlesnake",
    "goldfish",
    "fowl"
}


# ============================================================
# BLOCKED NON-ANIMAL WORDS
# ============================================================

BLOCKED_WORDS = {
    "oh",
    "okay",
    "ok",
    "well",
    "yes",
    "yeah",
    "no",
    "gee",
    "whiz",
    "jeez",
    "really",
    "right",
    "think",
    "thought",
    "know",
    "remember",
    "guess",
    "cant",
    "cannot",
    "dont",
    "didnt",
    "any",
    "more",
    "many",
    "else",
    "another",
    "people",
    "person",
    "man",
    "woman",
    "zoo",
    "farm",
    "jungle",
    "forest",
    "pencil",
    "paper",
    "throat",
    "pg",
    "hos",
    "til",
    "until",
    "hadta",
    "give",
    "wait",
    "have",
    "can",
    "get",
    "those",
    "that",
    "what",
    "how",
    "animal",
    "animals"
}


# ============================================================
# IRREGULAR PLURALS
# ============================================================

IRREGULAR = {
    "mice": "mouse",
    "geese": "goose",
    "wolves": "wolf",
    "calves": "calf",
    "oxen": "ox",
    "puppies": "puppy",
    "kittens": "kitten",
    "ponies": "pony",
    "men": "man",
    "women": "woman"
}


# ============================================================
# WORDNET SETUP
# ============================================================

lemmatizer = WordNetLemmatizer()
ANIMAL_SYNSET = wn.synset("animal.n.01")


# ============================================================
# CLEANING FUNCTIONS
# ============================================================

def normalize_word(value):
    if pd.isna(value):
        return ""

    value = str(value).lower().strip()
    value = value.lstrip("@!?;")
    value = value.replace("-", "_")
    value = value.replace(" ", "_")

    value = re.sub(
        r"[^a-z_']",
        "",
        value
    )

    value = re.sub(
        r"_+",
        "_",
        value
    ).strip("_'")

    return value


def singularize(word):
    word = normalize_word(word)

    if not word:
        return ""

    if word in IRREGULAR:
        return IRREGULAR[word]

    return lemmatizer.lemmatize(
        word,
        pos="n"
    )


def is_wordnet_animal(word):
    if not word:
        return False

    if word in BLOCKED_WORDS:
        return False

    lookup = word.replace(
        "_",
        " "
    )

    synsets = wn.synsets(
        lookup,
        pos=wn.NOUN
    )

    for synset in synsets:
        for hypernym_path in synset.hypernym_paths():
            if ANIMAL_SYNSET in hypernym_path:
                return True

    return False


def is_valid_animal(word):
    normalized = normalize_word(word)
    singular = singularize(normalized)

    if not singular:
        return False

    if singular in BLOCKED_WORDS:
        return False

    if normalized in ANIMAL_WORDS:
        return True

    if singular in ANIMAL_WORDS:
        return True

    return is_wordnet_animal(
        singular
    )


def clean_chat_utterance(text):
    text = BULLET_RE.sub(
        " ",
        text
    )

    text = ANGLE_RETRACE_RE.sub(
        r"\1",
        text
    )

    text = CHAT_CODE_RE.sub(
        " ",
        text
    )

    text = EVENT_RE.sub(
        " ",
        text
    )

    text = FRAGMENT_RE.sub(
        " ",
        text
    )

    text = AT_SUFFIX_RE.sub(
        " ",
        text
    )

    text = text.replace(
        "xxx",
        " "
    )

    text = text.replace(
        "yyy",
        " "
    )

    text = text.replace(
        "(",
        " "
    )

    text = text.replace(
        ")",
        " "
    )

    text = NON_WORD_RE.sub(
        " ",
        text
    )

    text = MULTISPACE_RE.sub(
        " ",
        text
    ).strip()

    return text


def extract_animals(text):
    tokens = re.findall(
        r"[A-Za-z]+(?:[_-][A-Za-z]+)*",
        str(text)
    )

    animals = []

    for token in tokens:
        normalized = normalize_word(
            token
        )

        singular = singularize(
            normalized
        )

        if is_valid_animal(
            singular
        ):
            animals.append(
                singular
            )

    return animals


# ============================================================
# FIND PITT FLUENCY CHAT FILES
# ============================================================

pitt_files = [
    path
    for path in SOURCE_ROOT.rglob("*.cha")
    if "pitt" in str(path).lower()
    and "fluency" in str(path).lower()
]

if not pitt_files:
    raise FileNotFoundError(
        "No Pitt fluency .cha files found under "
        f"{SOURCE_ROOT}"
    )

print(
    "Pitt fluency CHAT files found:",
    len(pitt_files)
)


# ============================================================
# PARSE ONE PITT FILE
# ============================================================

def parse_pitt_category_timing(path):
    raw_text = path.read_text(
        encoding="utf-8",
        errors="replace"
    )

    lines = raw_text.splitlines()

    media_name = path.stem
    current_task = None
    pending_text = None
    pending_start = None

    animal_utterances = []

    for line in lines:
        if line.startswith("@Media:"):
            media_name = (
                line.split(":", 1)[1]
                .split(",", 1)[0]
                .strip()
            )
            break

    def flush_pending():
        nonlocal pending_text
        nonlocal pending_start

        if (
            current_task == "category"
            and pending_text is not None
        ):
            cleaned_text = clean_chat_utterance(
                pending_text
            )

            animals = extract_animals(
                cleaned_text
            )

            if animals:
                animal_utterances.append({
                    "start_ms": pending_start,
                    "animals": animals
                })

        pending_text = None
        pending_start = None

    for line in lines:
        if line.startswith("@G:"):
            flush_pending()

            group_name = (
                line.split(":", 1)[1]
                .strip()
                .lower()
            )

            if "animal" in group_name:
                current_task = "category"
            else:
                current_task = None

            continue

        if line.startswith("*"):
            flush_pending()

            if (
                line.startswith("*PAR:")
                and current_task == "category"
            ):
                payload = (
                    line.split(":", 1)[1]
                    .strip()
                )

                timing_match = BULLET_RE.search(
                    payload
                )

                if timing_match:
                    pending_start = int(
                        timing_match.group(1)
                    )
                else:
                    pending_start = None

                pending_text = payload

            continue

        if (
            pending_text is not None
            and line.startswith("\t")
        ):
            pending_text += (
                " " + line.strip()
            )

    flush_pending()

    if not animal_utterances:
        return None

    known_times = [
        entry["start_ms"]
        for entry in animal_utterances
        if entry["start_ms"] is not None
    ]

    if not known_times:
        return None

    task_start = min(
        known_times
    )

    bins = [
        [],
        [],
        [],
        []
    ]

    for entry in animal_utterances:
        start_ms = entry[
            "start_ms"
        ]

        if start_ms is None:
            continue

        relative_seconds = max(
            0.0,
            (
                start_ms
                - task_start
            ) / 1000.0
        )

        bin_index = min(
            3,
            int(
                relative_seconds // 15
            )
        )

        bins[bin_index].extend(
            entry["animals"]
        )

    all_animals = (
        bins[0]
        + bins[1]
        + bins[2]
        + bins[3]
    )

    if not all_animals:
        return None

    counts = Counter(
        all_animals
    )

    return {
        "recording_id": (
            f"Pitt_{media_name}_category"
        ),

        "source_recording_id": media_name,

        "clean_animals_0_15s": len(
            bins[0]
        ),

        "clean_animals_15_30s": len(
            bins[1]
        ),

        "clean_animals_30_45s": len(
            bins[2]
        ),

        "clean_animals_45_60s": len(
            bins[3]
        ),

        "clean_total_from_bins": len(
            all_animals
        ),

        "clean_unique_from_bins": len(
            set(all_animals)
        ),

        "clean_repetitions_from_bins": sum(
            max(
                0,
                count - 1
            )
            for count in counts.values()
        ),

        "clean_items_from_bins": " ".join(
            all_animals
        )
    }


# ============================================================
# PARSE ALL PITT FILES
# ============================================================

timing_rows = []
timing_errors = []

for path in sorted(
    pitt_files
):
    try:
        result = parse_pitt_category_timing(
            path
        )

        if result is not None:
            timing_rows.append(
                result
            )

    except Exception as error:
        timing_errors.append({
            "source_path": str(path),
            "error": repr(error)
        })


timing = pd.DataFrame(
    timing_rows
)

if timing.empty:
    raise RuntimeError(
        "No Pitt category timing rows were produced."
    )

print(
    "Pitt category recordings with clean timing:",
    len(timing)
)

print(
    "Pitt CHAT files with parsing errors:",
    len(timing_errors)
)

if timing_errors:
    timing_errors_df = pd.DataFrame(
        timing_errors
    )

    timing_errors_df.to_csv(
        OUTPUT_DIR
        / "pitt_timing_parsing_errors.csv",
        index=False
    )

    display(
        timing_errors_df.head(20)
    )


# ============================================================
# CHECK FOR DUPLICATE TIMING IDS
# ============================================================

duplicate_timing_ids = timing[
    timing.duplicated(
        subset=[
            "recording_id"
        ],
        keep=False
    )
]

if not duplicate_timing_ids.empty:
    display(
        duplicate_timing_ids.head(20)
    )

    raise RuntimeError(
        "Duplicate Pitt timing recording IDs found."
    )


# ============================================================
# LOAD CLEANED PITT CATEGORY DATA
# ============================================================

if not PITT_CLEAN_PATH.exists():
    raise FileNotFoundError(
        PITT_CLEAN_PATH
    )

pitt = pd.read_csv(
    PITT_CLEAN_PATH
)

pitt = pitt[
    (
        pitt["dataset"]
        == "Pitt"
    )
    & (
        pitt["task"]
        == "category"
    )
].copy()

print(
    "Cleaned Pitt category rows loaded:",
    len(pitt)
)


# ============================================================
# MERGE CLEAN TIMING WITH PITT DATA
# ============================================================

pitt_timed = pitt.merge(
    timing,
    on=[
        "recording_id",
        "source_recording_id"
    ],
    how="left",
    validate="one_to_one"
)


# ============================================================
# CHECK TIMING AVAILABILITY
# ============================================================

pitt_timed[
    "timing_total_difference"
] = (
    pitt_timed[
        "clean_total_from_bins"
    ]
    - pitt_timed[
        "clean_total_items"
    ]
)

timing_missing = pitt_timed[
    pitt_timed[
        "clean_total_from_bins"
    ].isna()
].copy()

timing_mismatch = pitt_timed[
    pitt_timed[
        "clean_total_from_bins"
    ].notna()
    & (
        pitt_timed[
            "timing_total_difference"
        ]
        != 0
    )
].copy()

print()
print(
    "Pitt rows missing timing:",
    len(timing_missing)
)

print(
    "Pitt rows with timing-total mismatch:",
    len(timing_mismatch)
)


# ============================================================
# USE TIMING-DERIVED TOTALS
# ============================================================

pitt_timed[
    "clean_total_items"
] = pitt_timed[
    "clean_total_from_bins"
]

pitt_timed[
    "clean_unique_items"
] = pitt_timed[
    "clean_unique_from_bins"
]

pitt_timed[
    "clean_repetition_count"
] = pitt_timed[
    "clean_repetitions_from_bins"
]

pitt_timed[
    "clean_type_token_ratio"
] = np.where(
    pitt_timed[
        "clean_total_items"
    ] > 0,

    pitt_timed[
        "clean_unique_items"
    ]
    / pitt_timed[
        "clean_total_items"
    ],

    np.nan
)


# ============================================================
# DERIVED PITT TIMING FEATURES
# ============================================================

pitt_timed[
    "clean_first_half"
] = (
    pitt_timed[
        "clean_animals_0_15s"
    ]
    + pitt_timed[
        "clean_animals_15_30s"
    ]
)

pitt_timed[
    "clean_second_half"
] = (
    pitt_timed[
        "clean_animals_30_45s"
    ]
    + pitt_timed[
        "clean_animals_45_60s"
    ]
)

pitt_timed[
    "clean_first_last_decline"
] = (
    pitt_timed[
        "clean_animals_0_15s"
    ]
    - pitt_timed[
        "clean_animals_45_60s"
    ]
)

pitt_timed[
    "clean_half_decline"
] = (
    pitt_timed[
        "clean_first_half"
    ]
    - pitt_timed[
        "clean_second_half"
    ]
)

pitt_timed[
    "clean_empty_bins"
] = (
    pitt_timed[
        [
            "clean_animals_0_15s",
            "clean_animals_15_30s",
            "clean_animals_30_45s",
            "clean_animals_45_60s"
        ]
    ]
    .eq(0)
    .sum(axis=1)
)

pitt_timed[
    "clean_max_bin"
] = pitt_timed[
    [
        "clean_animals_0_15s",
        "clean_animals_15_30s",
        "clean_animals_30_45s",
        "clean_animals_45_60s"
    ]
].max(axis=1)

pitt_timed[
    "clean_min_bin"
] = pitt_timed[
    [
        "clean_animals_0_15s",
        "clean_animals_15_30s",
        "clean_animals_30_45s",
        "clean_animals_45_60s"
    ]
].min(axis=1)


# ============================================================
# SELECT ONE PITT AD RECORDING PER SUBJECT
# ============================================================

pitt_ad = pitt_timed[
    (
        pitt_timed[
            "diagnosis_group"
        ]
        == "AD"
    )
    & (
        pitt_timed[
            "binary_label"
        ]
        == 1
    )
    & (
        pitt_timed[
            "clean_total_items"
        ]
        >= MIN_VALID_ANIMALS
    )
    & pitt_timed[
        "clean_total_from_bins"
    ].notna()
].copy()

pitt_ad = (
    pitt_ad
    .sort_values(
        [
            "subject_id",
            "clean_unique_items",
            "clean_total_items",
            "visit"
        ],
        ascending=[
            True,
            False,
            False,
            True
        ]
    )
    .drop_duplicates(
        subset=[
            "subject_id"
        ],
        keep="first"
    )
    .reset_index(
        drop=True
    )
)

print(
    "Pitt AD subjects retained:",
    pitt_ad[
        "subject_id"
    ].nunique()
)


# ============================================================
# LOAD WLS ANIMAL CONTROLS
# ============================================================

if not WLS_PATH.exists():
    raise FileNotFoundError(
        WLS_PATH
    )

wls = pd.read_csv(
    WLS_PATH
)

wls[
    "dataset"
] = "WLS"

wls[
    "diagnosis_group"
] = "Control"

wls[
    "binary_label"
] = 0

wls[
    "task"
] = "category"

wls[
    "task_prompt"
] = "animals"


# ============================================================
# WLS BASIC CLEAN FEATURES
# ============================================================

wls[
    "clean_total_items"
] = pd.to_numeric(
    wls[
        "total_items"
    ],
    errors="coerce"
)

wls[
    "clean_unique_items"
] = (
    wls[
        "items_text"
    ]
    .fillna("")
    .apply(
        lambda value: len(
            set(
                str(value).split()
            )
        )
    )
)

wls[
    "clean_repetition_count"
] = (
    wls[
        "clean_total_items"
    ]
    - wls[
        "clean_unique_items"
    ]
)

wls[
    "clean_type_token_ratio"
] = np.where(
    wls[
        "clean_total_items"
    ] > 0,

    wls[
        "clean_unique_items"
    ]
    / wls[
        "clean_total_items"
    ],

    np.nan
)


# ============================================================
# VERIFY WLS TIMING COLUMNS
# ============================================================

required_wls_timing_columns = [
    "items_0_15s",
    "items_15_30s",
    "items_30_45s",
    "items_45_60s"
]

missing_wls_columns = [
    column
    for column in required_wls_timing_columns
    if column not in wls.columns
]

if missing_wls_columns:
    raise ValueError(
        "Missing WLS timing columns: "
        + ", ".join(
            missing_wls_columns
        )
    )


# ============================================================
# WLS TIMING FEATURES
# ============================================================

wls[
    "clean_animals_0_15s"
] = pd.to_numeric(
    wls[
        "items_0_15s"
    ],
    errors="coerce"
)

wls[
    "clean_animals_15_30s"
] = pd.to_numeric(
    wls[
        "items_15_30s"
    ],
    errors="coerce"
)

wls[
    "clean_animals_30_45s"
] = pd.to_numeric(
    wls[
        "items_30_45s"
    ],
    errors="coerce"
)

wls[
    "clean_animals_45_60s"
] = pd.to_numeric(
    wls[
        "items_45_60s"
    ],
    errors="coerce"
)

wls[
    "clean_first_half"
] = (
    wls[
        "clean_animals_0_15s"
    ]
    + wls[
        "clean_animals_15_30s"
    ]
)

wls[
    "clean_second_half"
] = (
    wls[
        "clean_animals_30_45s"
    ]
    + wls[
        "clean_animals_45_60s"
    ]
)

wls[
    "clean_first_last_decline"
] = (
    wls[
        "clean_animals_0_15s"
    ]
    - wls[
        "clean_animals_45_60s"
    ]
)

wls[
    "clean_half_decline"
] = (
    wls[
        "clean_first_half"
    ]
    - wls[
        "clean_second_half"
    ]
)

wls[
    "clean_empty_bins"
] = (
    wls[
        [
            "clean_animals_0_15s",
            "clean_animals_15_30s",
            "clean_animals_30_45s",
            "clean_animals_45_60s"
        ]
    ]
    .eq(0)
    .sum(axis=1)
)

wls[
    "clean_max_bin"
] = wls[
    [
        "clean_animals_0_15s",
        "clean_animals_15_30s",
        "clean_animals_30_45s",
        "clean_animals_45_60s"
    ]
].max(axis=1)

wls[
    "clean_min_bin"
] = wls[
    [
        "clean_animals_0_15s",
        "clean_animals_15_30s",
        "clean_animals_30_45s",
        "clean_animals_45_60s"
    ]
].min(axis=1)


# ============================================================
# FILTER WLS CONTROLS
# ============================================================

wls_controls = wls[
    wls[
        "clean_total_items"
    ]
    >= MIN_VALID_ANIMALS
].copy()

number_of_ad_subjects = pitt_ad[
    "subject_id"
].nunique()

if len(
    wls_controls
) < number_of_ad_subjects:
    raise RuntimeError(
        "Not enough WLS controls to balance Pitt AD subjects."
    )

wls_controls = wls_controls.sample(
    n=number_of_ad_subjects,
    random_state=RANDOM_SEED
).reset_index(
    drop=True
)

print(
    "WLS control subjects sampled:",
    wls_controls[
        "subject_id"
    ].nunique()
)


# ============================================================
# FINAL FEATURE COLUMNS
# ============================================================

feature_columns = [
    "clean_total_items",
    "clean_unique_items",
    "clean_repetition_count",
    "clean_type_token_ratio",

    "clean_animals_0_15s",
    "clean_animals_15_30s",
    "clean_animals_30_45s",
    "clean_animals_45_60s",

    "clean_first_half",
    "clean_second_half",
    "clean_first_last_decline",
    "clean_half_decline",

    "clean_empty_bins",
    "clean_max_bin",
    "clean_min_bin"
]

identity_columns = [
    "dataset",
    "subject_id",
    "recording_id",
    "diagnosis_group",
    "binary_label",
    "task",
    "task_prompt"
]

final_columns = (
    identity_columns
    + feature_columns
)


# ============================================================
# ENSURE ALL COLUMNS EXIST
# ============================================================

for column in final_columns:
    if column not in pitt_ad.columns:
        pitt_ad[
            column
        ] = np.nan

    if column not in wls_controls.columns:
        wls_controls[
            column
        ] = np.nan


# ============================================================
# COMBINE FINAL DATASET
# ============================================================

final_data = pd.concat(
    [
        pitt_ad[
            final_columns
        ],
        wls_controls[
            final_columns
        ]
    ],
    ignore_index=True
)

final_data = final_data.sample(
    frac=1,
    random_state=RANDOM_SEED
).reset_index(
    drop=True
)


# ============================================================
# FINAL INTEGRITY CHECKS
# ============================================================

if final_data[
    "subject_id"
].duplicated().any():

    duplicate_subjects = final_data.loc[
        final_data[
            "subject_id"
        ].duplicated(
            keep=False
        ),
        "subject_id"
    ].unique()

    raise RuntimeError(
        "Duplicate subjects found: "
        + ", ".join(
            duplicate_subjects[:20]
        )
    )


missing_feature_counts = final_data[
    feature_columns
].isna().sum()

missing_feature_counts = missing_feature_counts[
    missing_feature_counts > 0
]

if len(
    missing_feature_counts
) > 0:
    print(
        "Missing feature values:"
    )

    print(
        missing_feature_counts
    )

    raise RuntimeError(
        "Missing feature values remain in final timing dataset."
    )


if final_data[
    "binary_label"
].nunique() != 2:
    raise RuntimeError(
        "Final dataset does not contain both classes."
    )


# ============================================================
# DATASET SUMMARY
# ============================================================

summary = (
    final_data
    .groupby(
        [
            "dataset",
            "diagnosis_group",
            "binary_label"
        ]
    )
    .agg(
        rows=(
            "recording_id",
            "size"
        ),

        subjects=(
            "subject_id",
            "nunique"
        ),

        median_total=(
            "clean_total_items",
            "median"
        ),

        median_unique=(
            "clean_unique_items",
            "median"
        ),

        median_first_bin=(
            "clean_animals_0_15s",
            "median"
        ),

        median_second_bin=(
            "clean_animals_15_30s",
            "median"
        ),

        median_third_bin=(
            "clean_animals_30_45s",
            "median"
        ),

        median_last_bin=(
            "clean_animals_45_60s",
            "median"
        ),

        median_half_decline=(
            "clean_half_decline",
            "median"
        ),

        median_empty_bins=(
            "clean_empty_bins",
            "median"
        )
    )
    .reset_index()
)


print()
print(
    "=" * 100
)

print(
    "TIMING DATASET SUMMARY"
)

print(
    "=" * 100
)

display(
    summary
)

print(
    "Total rows:",
    len(final_data)
)

print(
    "Total subjects:",
    final_data[
        "subject_id"
    ].nunique()
)

print(
    "AD subjects:",
    final_data.loc[
        final_data[
            "binary_label"
        ] == 1,
        "subject_id"
    ].nunique()
)

print(
    "Control subjects:",
    final_data.loc[
        final_data[
            "binary_label"
        ] == 0,
        "subject_id"
    ].nunique()
)


# ============================================================
# SAVE OUTPUTS
# ============================================================

pitt_timed.to_csv(
    OUTPUT_DIR
    / "pitt_category_clean_timing_all.csv",
    index=False
)

timing_missing.to_csv(
    OUTPUT_DIR
    / "pitt_timing_missing.csv",
    index=False
)

timing_mismatch.to_csv(
    OUTPUT_DIR
    / "pitt_timing_mismatch.csv",
    index=False
)

pitt_ad[
    final_columns
].to_csv(
    OUTPUT_DIR
    / "pitt_AD_timing_one_per_subject.csv",
    index=False
)

wls_controls[
    final_columns
].to_csv(
    OUTPUT_DIR
    / "wls_controls_timing_sampled.csv",
    index=False
)

final_data.to_csv(
    OUTPUT_DIR
    / "category_timing_balanced.csv",
    index=False
)

summary.to_csv(
    OUTPUT_DIR
    / "category_timing_summary.csv",
    index=False
)


# ============================================================
# FINAL OUTPUT
# ============================================================

print()
print(
    "=" * 100
)

print(
    "TIMING PREPROCESSING COMPLETE"
)

print(
    "=" * 100
)

print(
    "Pitt rows missing timing:",
    len(timing_missing)
)

print(
    "Pitt rows with timing-total mismatch:",
    len(timing_mismatch)
)

print()
print(
    "Saved files:"
)

for output_path in sorted(
    OUTPUT_DIR.glob("*.csv")
):
    print(
        " -",
        output_path
    )

print()
print(
    "NEXT MODELING FILE:"
)

print(
    OUTPUT_DIR
    / "category_timing_balanced.csv"
)

In [ ]:
# ============================================================
# CATEGORY FLUENCY TIMING ABLATION
# Compares count features, timing features, and all features
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import (
    RepeatedStratifiedKFold,
    cross_validate
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


# ============================================================
# PATHS
# ============================================================

DATA_PATH = Path(
    "/content/drive/MyDrive/Fluency_Processed/"
    "Category_Timing_Final/category_timing_balanced.csv"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Fluency_Processed/"
    "Category_Timing_Model"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# LOAD DATA
# ============================================================

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}"
    )

data = pd.read_csv(DATA_PATH)

data = data[
    data["binary_label"].isin([0, 1])
].copy()

data["binary_label"] = pd.to_numeric(
    data["binary_label"],
    errors="raise"
).astype(int)


# ============================================================
# SUBJECT CHECKS
# ============================================================

if data["subject_id"].duplicated().any():
    duplicate_subjects = data.loc[
        data["subject_id"].duplicated(keep=False),
        "subject_id"
    ].unique()

    raise ValueError(
        "Duplicate subjects found: "
        + ", ".join(duplicate_subjects[:20])
    )

if data["binary_label"].nunique() != 2:
    raise ValueError(
        "Dataset must contain both Control and AD."
    )


# ============================================================
# FEATURE SETS
# ============================================================

feature_sets = {
    "Total only": [
        "clean_total_items"
    ],

    "Unique only": [
        "clean_unique_items"
    ],

    "Total + unique": [
        "clean_total_items",
        "clean_unique_items"
    ],

    "Basic count features": [
        "clean_total_items",
        "clean_unique_items",
        "clean_repetition_count",
        "clean_type_token_ratio"
    ],

    "Raw timing bins only": [
        "clean_animals_0_15s",
        "clean_animals_15_30s",
        "clean_animals_30_45s",
        "clean_animals_45_60s"
    ],

    "Derived timing only": [
        "clean_first_half",
        "clean_second_half",
        "clean_first_last_decline",
        "clean_half_decline",
        "clean_empty_bins",
        "clean_max_bin",
        "clean_min_bin"
    ],

    "Raw + derived timing": [
        "clean_animals_0_15s",
        "clean_animals_15_30s",
        "clean_animals_30_45s",
        "clean_animals_45_60s",
        "clean_first_half",
        "clean_second_half",
        "clean_first_last_decline",
        "clean_half_decline",
        "clean_empty_bins",
        "clean_max_bin",
        "clean_min_bin"
    ],

    "Counts + raw timing": [
        "clean_total_items",
        "clean_unique_items",
        "clean_repetition_count",
        "clean_type_token_ratio",
        "clean_animals_0_15s",
        "clean_animals_15_30s",
        "clean_animals_30_45s",
        "clean_animals_45_60s"
    ],

    "All features": [
        "clean_total_items",
        "clean_unique_items",
        "clean_repetition_count",
        "clean_type_token_ratio",
        "clean_animals_0_15s",
        "clean_animals_15_30s",
        "clean_animals_30_45s",
        "clean_animals_45_60s",
        "clean_first_half",
        "clean_second_half",
        "clean_first_last_decline",
        "clean_half_decline",
        "clean_empty_bins",
        "clean_max_bin",
        "clean_min_bin"
    ]
}


# ============================================================
# VERIFY FEATURES
# ============================================================

all_required_features = sorted(
    {
        feature
        for features in feature_sets.values()
        for feature in features
    }
)

missing_columns = [
    column
    for column in all_required_features
    if column not in data.columns
]

if missing_columns:
    raise ValueError(
        "Missing feature columns: "
        + ", ".join(missing_columns)
    )

for column in all_required_features:
    data[column] = pd.to_numeric(
        data[column],
        errors="coerce"
    )


# ============================================================
# LABEL
# ============================================================

y = data["binary_label"]


# ============================================================
# MODEL
# ============================================================

def make_model():
    return Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "classifier",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=5000,
                    random_state=42
                )
            )
        ]
    )


# ============================================================
# CROSS-VALIDATION
# ============================================================

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "average_precision": "average_precision"
}


# ============================================================
# RUN ABLATION
# ============================================================

results_rows = []

for feature_set_name, columns in feature_sets.items():

    print()
    print("=" * 100)
    print("FEATURE SET:", feature_set_name)
    print("FEATURES:", columns)
    print("=" * 100)

    X = data[columns].copy()

    model = make_model()

    scores = cross_validate(
        estimator=model,
        X=X,
        y=y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False,
        error_score="raise"
    )

    result = {
        "feature_set": feature_set_name,
        "number_of_features": len(columns),
        "features": ", ".join(columns)
    }

    for metric_name in scoring:
        values = scores[
            f"test_{metric_name}"
        ]

        result[
            f"{metric_name}_mean"
        ] = values.mean()

        result[
            f"{metric_name}_std"
        ] = values.std()

    results_rows.append(result)


# ============================================================
# RESULTS
# ============================================================

results = pd.DataFrame(
    results_rows
).sort_values(
    "roc_auc_mean",
    ascending=False
).reset_index(drop=True)

print()
print("=" * 100)
print("TIMING ABLATION RESULTS")
print("=" * 100)

display(results)


# ============================================================
# SAVE
# ============================================================

results.to_csv(
    OUTPUT_DIR / "timing_feature_ablation_results.csv",
    index=False
)


# ============================================================
# SHOW COMPACT TABLE
# ============================================================

compact_columns = [
    "feature_set",
    "number_of_features",
    "accuracy_mean",
    "accuracy_std",
    "balanced_accuracy_mean",
    "f1_mean",
    "roc_auc_mean",
    "roc_auc_std",
    "average_precision_mean"
]

print()
print("COMPACT COMPARISON")

display(
    results[
        compact_columns
    ]
)


# ============================================================
# BEST FEATURE SET
# ============================================================

best_row = results.iloc[0]

print()
print("=" * 100)
print("BEST FEATURE SET")
print("=" * 100)

print(
    "Feature set:",
    best_row["feature_set"]
)

print(
    "Number of features:",
    int(best_row["number_of_features"])
)

print(
    f"Accuracy: {best_row['accuracy_mean']:.4f} "
    f"± {best_row['accuracy_std']:.4f}"
)

print(
    f"Balanced accuracy: "
    f"{best_row['balanced_accuracy_mean']:.4f}"
)

print(
    f"F1: {best_row['f1_mean']:.4f}"
)

print(
    f"ROC-AUC: {best_row['roc_auc_mean']:.4f} "
    f"± {best_row['roc_auc_std']:.4f}"
)

print(
    f"Average precision: "
    f"{best_row['average_precision_mean']:.4f}"
)

print()
print("Saved to:")
print(
    OUTPUT_DIR / "timing_feature_ablation_results.csv"
)

In [ ]:
# ============================================================
# FINAL CATEGORY-FLUENCY MODEL
# Nested threshold tuning + out-of-fold predictions
# Bootstrap 95% confidence intervals
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import (
    StratifiedKFold,
    RepeatedStratifiedKFold,
    cross_validate
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report,
    RocCurveDisplay,
    PrecisionRecallDisplay
)


# ============================================================
# PATHS
# ============================================================

DATA_PATH = Path(
    "/content/drive/MyDrive/Fluency_Processed/"
    "Category_Timing_Final/category_timing_balanced.csv"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Fluency_Processed/"
    "Category_Final_Model"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# SETTINGS
# ============================================================

RANDOM_SEED = 42
N_OUTER_FOLDS = 5
N_INNER_FOLDS = 5
N_BOOTSTRAPS = 5000

THRESHOLD_GRID = np.arange(
    0.20,
    0.81,
    0.01
)

FEATURE_COLUMNS = [
    "clean_total_items",
    "clean_unique_items",
    "clean_repetition_count",
    "clean_type_token_ratio"
]


# ============================================================
# LOAD DATA
# ============================================================

if not DATA_PATH.exists():
    raise FileNotFoundError(DATA_PATH)

data = pd.read_csv(DATA_PATH)

required_columns = [
    "subject_id",
    "recording_id",
    "dataset",
    "diagnosis_group",
    "binary_label",
    *FEATURE_COLUMNS
]

missing_columns = [
    column
    for column in required_columns
    if column not in data.columns
]

if missing_columns:
    raise ValueError(
        "Missing columns: "
        + ", ".join(missing_columns)
    )

data = data[
    data["binary_label"].isin([0, 1])
].copy()

data["binary_label"] = pd.to_numeric(
    data["binary_label"],
    errors="raise"
).astype(int)

for column in FEATURE_COLUMNS:
    data[column] = pd.to_numeric(
        data[column],
        errors="coerce"
    )

if data["subject_id"].duplicated().any():
    raise ValueError(
        "Duplicate subjects exist. "
        "This final dataset must contain one row per subject."
    )

if data["binary_label"].nunique() != 2:
    raise ValueError(
        "Both Control and AD classes are required."
    )

X = data[FEATURE_COLUMNS].reset_index(drop=True)
y = data["binary_label"].reset_index(drop=True)


# ============================================================
# MODEL
# ============================================================

def make_model():
    return Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "classifier",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=5000,
                    random_state=RANDOM_SEED
                )
            )
        ]
    )


# ============================================================
# METRIC FUNCTION
# ============================================================

def calculate_metrics(
    true_labels,
    probabilities,
    threshold
):
    predicted_labels = (
        probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        true_labels,
        predicted_labels,
        labels=[0, 1]
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    return {
        "accuracy": accuracy_score(
            true_labels,
            predicted_labels
        ),

        "balanced_accuracy": balanced_accuracy_score(
            true_labels,
            predicted_labels
        ),

        "precision": precision_score(
            true_labels,
            predicted_labels,
            zero_division=0
        ),

        "recall": recall_score(
            true_labels,
            predicted_labels,
            zero_division=0
        ),

        "specificity": specificity,

        "f1": f1_score(
            true_labels,
            predicted_labels,
            zero_division=0
        ),

        "roc_auc": roc_auc_score(
            true_labels,
            probabilities
        ),

        "average_precision": average_precision_score(
            true_labels,
            probabilities
        ),

        "mcc": matthews_corrcoef(
            true_labels,
            predicted_labels
        ),

        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }


# ============================================================
# INNER-FOLD THRESHOLD SELECTION
# ============================================================

def select_threshold(
    X_train,
    y_train,
    random_state
):
    inner_cv = StratifiedKFold(
        n_splits=N_INNER_FOLDS,
        shuffle=True,
        random_state=random_state
    )

    inner_probabilities = np.zeros(
        len(y_train),
        dtype=float
    )

    for inner_train_indices, inner_validation_indices in inner_cv.split(
        X_train,
        y_train
    ):
        model = make_model()

        model.fit(
            X_train.iloc[inner_train_indices],
            y_train.iloc[inner_train_indices]
        )

        inner_probabilities[
            inner_validation_indices
        ] = model.predict_proba(
            X_train.iloc[
                inner_validation_indices
            ]
        )[:, 1]

    threshold_rows = []

    for threshold in THRESHOLD_GRID:
        metrics = calculate_metrics(
            y_train,
            inner_probabilities,
            threshold
        )

        threshold_rows.append({
            "threshold": threshold,
            "balanced_accuracy": metrics[
                "balanced_accuracy"
            ],
            "mcc": metrics["mcc"],
            "f1": metrics["f1"],
            "recall": metrics["recall"],
            "specificity": metrics[
                "specificity"
            ]
        })

    threshold_results = pd.DataFrame(
        threshold_rows
    )

    threshold_results["selection_score"] = (
        threshold_results[
            "balanced_accuracy"
        ]
        + threshold_results["mcc"]
    )

    best_row = threshold_results.sort_values(
        [
            "selection_score",
            "balanced_accuracy",
            "mcc",
            "threshold"
        ],
        ascending=[
            False,
            False,
            False,
            True
        ]
    ).iloc[0]

    return (
        float(best_row["threshold"]),
        threshold_results
    )


# ============================================================
# NESTED OUT-OF-FOLD EVALUATION
# ============================================================

outer_cv = StratifiedKFold(
    n_splits=N_OUTER_FOLDS,
    shuffle=True,
    random_state=RANDOM_SEED
)

oof_probabilities = np.zeros(
    len(y),
    dtype=float
)

oof_predictions = np.zeros(
    len(y),
    dtype=int
)

fold_assignments = np.zeros(
    len(y),
    dtype=int
)

fold_thresholds = []
fold_metrics_rows = []

for fold_number, (
    train_indices,
    test_indices
) in enumerate(
    outer_cv.split(X, y),
    start=1
):
    X_train = X.iloc[
        train_indices
    ].reset_index(drop=True)

    y_train = y.iloc[
        train_indices
    ].reset_index(drop=True)

    X_test = X.iloc[
        test_indices
    ]

    y_test = y.iloc[
        test_indices
    ]

    best_threshold, threshold_results = (
        select_threshold(
            X_train,
            y_train,
            random_state=RANDOM_SEED + fold_number
        )
    )

    model = make_model()

    model.fit(
        X_train,
        y_train
    )

    test_probabilities = model.predict_proba(
        X_test
    )[:, 1]

    test_predictions = (
        test_probabilities
        >= best_threshold
    ).astype(int)

    oof_probabilities[
        test_indices
    ] = test_probabilities

    oof_predictions[
        test_indices
    ] = test_predictions

    fold_assignments[
        test_indices
    ] = fold_number

    fold_thresholds.append(
        best_threshold
    )

    fold_metrics = calculate_metrics(
        y_test,
        test_probabilities,
        best_threshold
    )

    fold_metrics_rows.append({
        "fold": fold_number,
        "threshold": best_threshold,
        **fold_metrics
    })

    threshold_results.to_csv(
        OUTPUT_DIR
        / f"threshold_search_fold_{fold_number}.csv",
        index=False
    )

    print(
        f"Fold {fold_number}: "
        f"threshold={best_threshold:.2f}, "
        f"balanced_accuracy="
        f"{fold_metrics['balanced_accuracy']:.4f}, "
        f"ROC-AUC={fold_metrics['roc_auc']:.4f}"
    )


fold_metrics_df = pd.DataFrame(
    fold_metrics_rows
)

fold_metrics_df.to_csv(
    OUTPUT_DIR / "nested_fold_metrics.csv",
    index=False
)


# ============================================================
# FINAL OOF METRICS
# ============================================================

overall_metrics = calculate_metrics(
    y,
    oof_probabilities,
    threshold=0.50
)

tuned_metrics = {
    "accuracy": accuracy_score(
        y,
        oof_predictions
    ),

    "balanced_accuracy": balanced_accuracy_score(
        y,
        oof_predictions
    ),

    "precision": precision_score(
        y,
        oof_predictions,
        zero_division=0
    ),

    "recall": recall_score(
        y,
        oof_predictions,
        zero_division=0
    ),

    "f1": f1_score(
        y,
        oof_predictions,
        zero_division=0
    ),

    "roc_auc": roc_auc_score(
        y,
        oof_probabilities
    ),

    "average_precision": average_precision_score(
        y,
        oof_probabilities
    ),

    "mcc": matthews_corrcoef(
        y,
        oof_predictions
    )
}

tn, fp, fn, tp = confusion_matrix(
    y,
    oof_predictions,
    labels=[0, 1]
).ravel()

tuned_metrics["specificity"] = (
    tn / (tn + fp)
    if (tn + fp) > 0
    else np.nan
)

tuned_metrics["tn"] = tn
tuned_metrics["fp"] = fp
tuned_metrics["fn"] = fn
tuned_metrics["tp"] = tp


# ============================================================
# BOOTSTRAP CONFIDENCE INTERVALS
# ============================================================

rng = np.random.default_rng(
    RANDOM_SEED
)

metric_names = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "roc_auc",
    "average_precision",
    "mcc"
]

bootstrap_values = {
    metric_name: []
    for metric_name in metric_names
}

valid_bootstraps = 0

for _ in range(N_BOOTSTRAPS):
    sampled_indices = rng.integers(
        0,
        len(y),
        size=len(y)
    )

    sampled_y = y.iloc[
        sampled_indices
    ].to_numpy()

    sampled_probabilities = (
        oof_probabilities[
            sampled_indices
        ]
    )

    sampled_predictions = (
        oof_predictions[
            sampled_indices
        ]
    )

    if len(
        np.unique(sampled_y)
    ) < 2:
        continue

    sampled_tn, sampled_fp, sampled_fn, sampled_tp = confusion_matrix(
        sampled_y,
        sampled_predictions,
        labels=[0, 1]
    ).ravel()

    sampled_specificity = (
        sampled_tn
        / (
            sampled_tn
            + sampled_fp
        )
        if (
            sampled_tn
            + sampled_fp
        ) > 0
        else np.nan
    )

    bootstrap_values[
        "accuracy"
    ].append(
        accuracy_score(
            sampled_y,
            sampled_predictions
        )
    )

    bootstrap_values[
        "balanced_accuracy"
    ].append(
        balanced_accuracy_score(
            sampled_y,
            sampled_predictions
        )
    )

    bootstrap_values[
        "precision"
    ].append(
        precision_score(
            sampled_y,
            sampled_predictions,
            zero_division=0
        )
    )

    bootstrap_values[
        "recall"
    ].append(
        recall_score(
            sampled_y,
            sampled_predictions,
            zero_division=0
        )
    )

    bootstrap_values[
        "specificity"
    ].append(
        sampled_specificity
    )

    bootstrap_values[
        "f1"
    ].append(
        f1_score(
            sampled_y,
            sampled_predictions,
            zero_division=0
        )
    )

    bootstrap_values[
        "roc_auc"
    ].append(
        roc_auc_score(
            sampled_y,
            sampled_probabilities
        )
    )

    bootstrap_values[
        "average_precision"
    ].append(
        average_precision_score(
            sampled_y,
            sampled_probabilities
        )
    )

    bootstrap_values[
        "mcc"
    ].append(
        matthews_corrcoef(
            sampled_y,
            sampled_predictions
        )
    )

    valid_bootstraps += 1


confidence_interval_rows = []

for metric_name in metric_names:
    values = np.asarray(
        bootstrap_values[
            metric_name
        ],
        dtype=float
    )

    point_estimate = tuned_metrics[
        metric_name
    ]

    lower = np.nanpercentile(
        values,
        2.5
    )

    upper = np.nanpercentile(
        values,
        97.5
    )

    confidence_interval_rows.append({
        "metric": metric_name,
        "estimate": point_estimate,
        "ci_95_lower": lower,
        "ci_95_upper": upper,
        "valid_bootstraps": valid_bootstraps
    })

confidence_intervals = pd.DataFrame(
    confidence_interval_rows
)

confidence_intervals.to_csv(
    OUTPUT_DIR
    / "bootstrap_confidence_intervals.csv",
    index=False
)


# ============================================================
# SAVE OOF PREDICTIONS
# ============================================================

predictions = data[
    [
        "subject_id",
        "recording_id",
        "dataset",
        "diagnosis_group",
        "binary_label"
    ]
].copy()

predictions[
    "outer_fold"
] = fold_assignments

predictions[
    "predicted_probability_AD"
] = oof_probabilities

predictions[
    "predicted_label"
] = oof_predictions

predictions[
    "correct"
] = (
    predictions[
        "predicted_label"
    ]
    == predictions[
        "binary_label"
    ]
)

predictions.to_csv(
    OUTPUT_DIR
    / "nested_out_of_fold_predictions.csv",
    index=False
)


# ============================================================
# SAVE METRICS
# ============================================================

metrics_df = pd.DataFrame([
    {
        "evaluation": "nested_threshold_tuned",
        **tuned_metrics
    }
])

metrics_df.to_csv(
    OUTPUT_DIR / "final_metrics.csv",
    index=False
)


# ============================================================
# PLOTS
# ============================================================

RocCurveDisplay.from_predictions(
    y,
    oof_probabilities
)

plt.title(
    "Category Fluency Nested OOF ROC Curve"
)

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "final_roc_curve.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()
plt.close()


PrecisionRecallDisplay.from_predictions(
    y,
    oof_probabilities
)

plt.title(
    "Category Fluency Nested OOF Precision-Recall Curve"
)

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR
    / "final_precision_recall_curve.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()
plt.close()


confusion = confusion_matrix(
    y,
    oof_predictions,
    labels=[0, 1]
)

plt.figure(
    figsize=(6, 5)
)

plt.imshow(
    confusion
)

plt.xticks(
    [0, 1],
    ["Control", "AD"]
)

plt.yticks(
    [0, 1],
    ["Control", "AD"]
)

plt.xlabel(
    "Predicted label"
)

plt.ylabel(
    "True label"
)

plt.title(
    "Nested OOF Confusion Matrix"
)

for row in range(2):
    for column in range(2):
        plt.text(
            column,
            row,
            confusion[row, column],
            ha="center",
            va="center"
        )

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR
    / "final_confusion_matrix.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()
plt.close()


# ============================================================
# TRAIN FINAL DEPLOYMENT MODEL
# ============================================================

final_threshold = float(
    np.median(
        fold_thresholds
    )
)

final_model = make_model()

final_model.fit(
    X,
    y
)

classifier = final_model.named_steps[
    "classifier"
]

coefficients = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "coefficient": classifier.coef_[0],
    "absolute_coefficient": np.abs(
        classifier.coef_[0]
    )
}).sort_values(
    "absolute_coefficient",
    ascending=False
)

coefficients.to_csv(
    OUTPUT_DIR / "final_coefficients.csv",
    index=False
)

model_package = {
    "model": final_model,
    "feature_columns": FEATURE_COLUMNS,
    "threshold": final_threshold,
    "fold_thresholds": fold_thresholds,
    "training_rows": len(data),
    "training_subjects": data[
        "subject_id"
    ].nunique(),
    "model_type": "Logistic Regression",
    "task": "Category animal fluency",
    "important_limitation": (
        "AD samples are Pitt and controls are WLS; "
        "external validation is required."
    )
}

joblib.dump(
    model_package,
    OUTPUT_DIR
    / "category_fluency_final_model.joblib"
)


# ============================================================
# FINAL REPORT
# ============================================================

print()
print("=" * 100)
print("FINAL CATEGORY-FLUENCY MODEL RESULTS")
print("=" * 100)

print(
    "Fold thresholds:",
    [
        round(value, 2)
        for value in fold_thresholds
    ]
)

print(
    f"Deployment threshold: {final_threshold:.2f}"
)

print()
print(
    f"Accuracy:           "
    f"{tuned_metrics['accuracy']:.4f}"
)

print(
    f"Balanced accuracy:  "
    f"{tuned_metrics['balanced_accuracy']:.4f}"
)

print(
    f"Precision:          "
    f"{tuned_metrics['precision']:.4f}"
)

print(
    f"Recall/Sensitivity: "
    f"{tuned_metrics['recall']:.4f}"
)

print(
    f"Specificity:        "
    f"{tuned_metrics['specificity']:.4f}"
)

print(
    f"F1 score:           "
    f"{tuned_metrics['f1']:.4f}"
)

print(
    f"ROC-AUC:            "
    f"{tuned_metrics['roc_auc']:.4f}"
)

print(
    f"Average precision:  "
    f"{tuned_metrics['average_precision']:.4f}"
)

print(
    f"MCC:                "
    f"{tuned_metrics['mcc']:.4f}"
)

print()
print("Confusion matrix:")
print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)

print()
print("95% confidence intervals:")
display(
    confidence_intervals
)

print()
print("Coefficients:")
display(
    coefficients
)

print()
print(
    classification_report(
        y,
        oof_predictions,
        target_names=[
            "Control",
            "AD"
        ],
        digits=4,
        zero_division=0
    )
)

print()
print("Saved to:")
print(OUTPUT_DIR)

for path in sorted(
    OUTPUT_DIR.iterdir()
):
    print(" -", path.name)

In [ ]:
# ============================================================
# WLS CONTROL RESAMPLING STABILITY TEST
# Repeats nested subject-level evaluation across many
# independently sampled WLS control groups
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    confusion_matrix
)


# ============================================================
# PATHS
# ============================================================

PITT_PATH = Path(
    "/content/drive/MyDrive/Fluency_Processed/"
    "Category_Timing_Final/"
    "pitt_AD_timing_one_per_subject.csv"
)

WLS_PATH = Path(
    "/content/drive/MyDrive/Fluency_Processed/"
    "WLS_Category_Filtered/"
    "wls_animal_fluency_one_per_subject_with_timing.csv"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Fluency_Processed/"
    "Category_Control_Resampling"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# SETTINGS
# ============================================================

RANDOM_SEED = 42

N_CONTROL_RESAMPLES = 100

N_OUTER_FOLDS = 5
N_INNER_FOLDS = 5

THRESHOLD_GRID = np.arange(
    0.20,
    0.81,
    0.01
)

FEATURE_COLUMNS = [
    "clean_total_items",
    "clean_unique_items",
    "clean_repetition_count",
    "clean_type_token_ratio"
]


# ============================================================
# LOAD PITT AD DATA
# ============================================================

if not PITT_PATH.exists():
    raise FileNotFoundError(
        f"Pitt file not found: {PITT_PATH}"
    )

pitt = pd.read_csv(PITT_PATH)

required_pitt_columns = [
    "subject_id",
    "recording_id",
    "binary_label",
    *FEATURE_COLUMNS
]

missing_pitt_columns = [
    column
    for column in required_pitt_columns
    if column not in pitt.columns
]

if missing_pitt_columns:
    raise ValueError(
        "Pitt file is missing columns: "
        + ", ".join(missing_pitt_columns)
    )

pitt = pitt[
    pitt["binary_label"] == 1
].copy()

pitt = pitt.drop_duplicates(
    subset=["subject_id"],
    keep="first"
).reset_index(drop=True)


# ============================================================
# LOAD FULL WLS CONTROL POOL
# ============================================================

if not WLS_PATH.exists():
    raise FileNotFoundError(
        f"WLS file not found: {WLS_PATH}"
    )

wls = pd.read_csv(WLS_PATH)

required_wls_columns = [
    "subject_id",
    "recording_id",
    "items_text",
    "total_items"
]

missing_wls_columns = [
    column
    for column in required_wls_columns
    if column not in wls.columns
]

if missing_wls_columns:
    raise ValueError(
        "WLS file is missing columns: "
        + ", ".join(missing_wls_columns)
    )


# ============================================================
# BUILD WLS MODEL FEATURES
# ============================================================

wls["clean_total_items"] = pd.to_numeric(
    wls["total_items"],
    errors="coerce"
)

wls["clean_unique_items"] = (
    wls["items_text"]
    .fillna("")
    .apply(
        lambda text: len(
            set(str(text).split())
        )
    )
)

wls["clean_repetition_count"] = (
    wls["clean_total_items"]
    - wls["clean_unique_items"]
)

wls["clean_type_token_ratio"] = np.where(
    wls["clean_total_items"] > 0,
    wls["clean_unique_items"]
    / wls["clean_total_items"],
    np.nan
)

wls["binary_label"] = 0
wls["dataset"] = "WLS"
wls["diagnosis_group"] = "Control"

wls = wls.drop_duplicates(
    subset=["subject_id"],
    keep="first"
).reset_index(drop=True)

wls = wls[
    wls["clean_total_items"] >= 3
].copy()


# ============================================================
# NUMERIC CONVERSION
# ============================================================

for column in FEATURE_COLUMNS:
    pitt[column] = pd.to_numeric(
        pitt[column],
        errors="coerce"
    )

    wls[column] = pd.to_numeric(
        wls[column],
        errors="coerce"
    )


# ============================================================
# BASIC CHECKS
# ============================================================

number_of_ad_subjects = pitt[
    "subject_id"
].nunique()

number_of_wls_subjects = wls[
    "subject_id"
].nunique()

print("=" * 100)
print("CONTROL RESAMPLING SETUP")
print("=" * 100)

print(
    "Pitt AD subjects:",
    number_of_ad_subjects
)

print(
    "Available WLS control subjects:",
    number_of_wls_subjects
)

print(
    "Controls sampled per repetition:",
    number_of_ad_subjects
)

print(
    "Number of repetitions:",
    N_CONTROL_RESAMPLES
)

if number_of_wls_subjects < number_of_ad_subjects:
    raise RuntimeError(
        "There are fewer WLS controls than Pitt AD subjects."
    )


# ============================================================
# MODEL
# ============================================================

def make_model():
    return Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "classifier",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=5000,
                    random_state=RANDOM_SEED
                )
            )
        ]
    )


# ============================================================
# METRIC FUNCTION
# ============================================================

def calculate_metrics(
    true_labels,
    probabilities,
    threshold
):
    predicted_labels = (
        probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        true_labels,
        predicted_labels,
        labels=[0, 1]
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    return {
        "accuracy": accuracy_score(
            true_labels,
            predicted_labels
        ),

        "balanced_accuracy": balanced_accuracy_score(
            true_labels,
            predicted_labels
        ),

        "precision": precision_score(
            true_labels,
            predicted_labels,
            zero_division=0
        ),

        "recall": recall_score(
            true_labels,
            predicted_labels,
            zero_division=0
        ),

        "specificity": specificity,

        "f1": f1_score(
            true_labels,
            predicted_labels,
            zero_division=0
        ),

        "roc_auc": roc_auc_score(
            true_labels,
            probabilities
        ),

        "average_precision": average_precision_score(
            true_labels,
            probabilities
        ),

        "mcc": matthews_corrcoef(
            true_labels,
            predicted_labels
        ),

        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }


# ============================================================
# INNER THRESHOLD SELECTION
# ============================================================

def choose_threshold(
    X_train,
    y_train,
    random_state
):
    inner_cv = StratifiedKFold(
        n_splits=N_INNER_FOLDS,
        shuffle=True,
        random_state=random_state
    )

    inner_probabilities = np.zeros(
        len(y_train),
        dtype=float
    )

    for (
        inner_train_indices,
        inner_validation_indices
    ) in inner_cv.split(
        X_train,
        y_train
    ):
        model = make_model()

        model.fit(
            X_train.iloc[
                inner_train_indices
            ],
            y_train.iloc[
                inner_train_indices
            ]
        )

        inner_probabilities[
            inner_validation_indices
        ] = model.predict_proba(
            X_train.iloc[
                inner_validation_indices
            ]
        )[:, 1]

    threshold_rows = []

    for threshold in THRESHOLD_GRID:
        metrics = calculate_metrics(
            y_train,
            inner_probabilities,
            threshold
        )

        threshold_rows.append({
            "threshold": threshold,
            "balanced_accuracy": metrics[
                "balanced_accuracy"
            ],
            "mcc": metrics["mcc"],
            "f1": metrics["f1"]
        })

    threshold_table = pd.DataFrame(
        threshold_rows
    )

    threshold_table["selection_score"] = (
        threshold_table[
            "balanced_accuracy"
        ]
        + threshold_table["mcc"]
    )

    best_row = threshold_table.sort_values(
        [
            "selection_score",
            "balanced_accuracy",
            "mcc",
            "threshold"
        ],
        ascending=[
            False,
            False,
            False,
            True
        ]
    ).iloc[0]

    return float(
        best_row["threshold"]
    )


# ============================================================
# ONE RESAMPLING EXPERIMENT
# ============================================================

def evaluate_one_control_sample(
    sampled_controls,
    repetition_number
):
    dataset = pd.concat(
        [
            pitt,
            sampled_controls
        ],
        ignore_index=True
    )

    dataset = dataset.sample(
        frac=1,
        random_state=(
            RANDOM_SEED
            + repetition_number
        )
    ).reset_index(drop=True)

    X = dataset[
        FEATURE_COLUMNS
    ].copy()

    y = dataset[
        "binary_label"
    ].astype(int).copy()

    outer_cv = StratifiedKFold(
        n_splits=N_OUTER_FOLDS,
        shuffle=True,
        random_state=(
            RANDOM_SEED
            + repetition_number
        )
    )

    oof_probabilities = np.zeros(
        len(dataset),
        dtype=float
    )

    oof_predictions = np.zeros(
        len(dataset),
        dtype=int
    )

    thresholds = []

    for fold_number, (
        train_indices,
        test_indices
    ) in enumerate(
        outer_cv.split(X, y),
        start=1
    ):
        X_train = X.iloc[
            train_indices
        ].reset_index(drop=True)

        y_train = y.iloc[
            train_indices
        ].reset_index(drop=True)

        X_test = X.iloc[
            test_indices
        ]

        threshold = choose_threshold(
            X_train,
            y_train,
            random_state=(
                RANDOM_SEED
                + repetition_number
                + fold_number
            )
        )

        model = make_model()

        model.fit(
            X_train,
            y_train
        )

        probabilities = model.predict_proba(
            X_test
        )[:, 1]

        predictions = (
            probabilities >= threshold
        ).astype(int)

        oof_probabilities[
            test_indices
        ] = probabilities

        oof_predictions[
            test_indices
        ] = predictions

        thresholds.append(
            threshold
        )

    tn, fp, fn, tp = confusion_matrix(
        y,
        oof_predictions,
        labels=[0, 1]
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    return {
        "repetition": repetition_number,
        "control_seed": (
            RANDOM_SEED
            + repetition_number
        ),
        "number_of_AD_subjects": int(
            (y == 1).sum()
        ),
        "number_of_control_subjects": int(
            (y == 0).sum()
        ),
        "median_threshold": float(
            np.median(thresholds)
        ),
        "minimum_threshold": float(
            np.min(thresholds)
        ),
        "maximum_threshold": float(
            np.max(thresholds)
        ),

        "accuracy": accuracy_score(
            y,
            oof_predictions
        ),

        "balanced_accuracy": balanced_accuracy_score(
            y,
            oof_predictions
        ),

        "precision": precision_score(
            y,
            oof_predictions,
            zero_division=0
        ),

        "recall": recall_score(
            y,
            oof_predictions,
            zero_division=0
        ),

        "specificity": specificity,

        "f1": f1_score(
            y,
            oof_predictions,
            zero_division=0
        ),

        "roc_auc": roc_auc_score(
            y,
            oof_probabilities
        ),

        "average_precision": average_precision_score(
            y,
            oof_probabilities
        ),

        "mcc": matthews_corrcoef(
            y,
            oof_predictions
        ),

        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }


# ============================================================
# RUN ALL CONTROL RESAMPLES
# ============================================================

results = []

for repetition in range(
    1,
    N_CONTROL_RESAMPLES + 1
):
    sample_seed = (
        RANDOM_SEED
        + repetition
    )

    sampled_controls = wls.sample(
        n=number_of_ad_subjects,
        replace=False,
        random_state=sample_seed
    ).copy()

    result = evaluate_one_control_sample(
        sampled_controls,
        repetition
    )

    results.append(
        result
    )

    if (
        repetition == 1
        or repetition % 10 == 0
        or repetition == N_CONTROL_RESAMPLES
    ):
        print(
            f"Completed {repetition}/"
            f"{N_CONTROL_RESAMPLES} | "
            f"accuracy={result['accuracy']:.4f} | "
            f"ROC-AUC={result['roc_auc']:.4f}"
        )


results_df = pd.DataFrame(
    results
)


# ============================================================
# SUMMARY STATISTICS
# ============================================================

metric_columns = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "roc_auc",
    "average_precision",
    "mcc",
    "median_threshold"
]

summary_rows = []

for metric in metric_columns:
    values = results_df[
        metric
    ].dropna().to_numpy()

    summary_rows.append({
        "metric": metric,
        "mean": np.mean(values),
        "standard_deviation": np.std(
            values,
            ddof=1
        ),
        "minimum": np.min(values),
        "percentile_2_5": np.percentile(
            values,
            2.5
        ),
        "median": np.median(values),
        "percentile_97_5": np.percentile(
            values,
            97.5
        ),
        "maximum": np.max(values)
    })

summary_df = pd.DataFrame(
    summary_rows
)


# ============================================================
# DISPLAY
# ============================================================

print()
print("=" * 100)
print("CONTROL RESAMPLING STABILITY RESULTS")
print("=" * 100)

display(summary_df)

print()
print("First 10 repetitions:")

display(
    results_df.head(10)
)


# ============================================================
# STABILITY COUNTS
# ============================================================

accuracy_above_85 = (
    results_df["accuracy"] >= 0.85
).mean()

accuracy_above_90 = (
    results_df["accuracy"] >= 0.90
).mean()

auc_above_90 = (
    results_df["roc_auc"] >= 0.90
).mean()

auc_above_95 = (
    results_df["roc_auc"] >= 0.95
).mean()

print()
print("Fraction of runs with accuracy >= 0.85:")
print(
    f"{accuracy_above_85:.3f}"
)

print(
    "Fraction of runs with accuracy >= 0.90:"
)
print(
    f"{accuracy_above_90:.3f}"
)

print(
    "Fraction of runs with ROC-AUC >= 0.90:"
)
print(
    f"{auc_above_90:.3f}"
)

print(
    "Fraction of runs with ROC-AUC >= 0.95:"
)
print(
    f"{auc_above_95:.3f}"
)


# ============================================================
# PLOTS
# ============================================================

for metric in [
    "accuracy",
    "roc_auc",
    "mcc",
    "recall",
    "specificity"
]:
    plt.figure(
        figsize=(8, 5)
    )

    plt.hist(
        results_df[metric].dropna(),
        bins=20
    )

    plt.axvline(
        results_df[metric].mean(),
        linestyle="--",
        label=(
            f"Mean = "
            f"{results_df[metric].mean():.3f}"
        )
    )

    plt.xlabel(metric)
    plt.ylabel("Number of resampling runs")

    plt.title(
        f"WLS Control Resampling: {metric}"
    )

    plt.legend()
    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR
        / f"resampling_distribution_{metric}.png",
        dpi=200,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()


# ============================================================
# SAVE
# ============================================================

results_df.to_csv(
    OUTPUT_DIR
    / "control_resampling_all_runs.csv",
    index=False
)

summary_df.to_csv(
    OUTPUT_DIR
    / "control_resampling_summary.csv",
    index=False
)

stability_summary = pd.DataFrame({
    "criterion": [
        "accuracy_at_least_0.85",
        "accuracy_at_least_0.90",
        "roc_auc_at_least_0.90",
        "roc_auc_at_least_0.95"
    ],
    "fraction_of_runs": [
        accuracy_above_85,
        accuracy_above_90,
        auc_above_90,
        auc_above_95
    ]
})

stability_summary.to_csv(
    OUTPUT_DIR
    / "control_resampling_stability_counts.csv",
    index=False
)


# ============================================================
# FINAL MESSAGE
# ============================================================

print()
print("=" * 100)
print("CONTROL RESAMPLING COMPLETE")
print("=" * 100)

print("Output folder:")
print(OUTPUT_DIR)

for output_path in sorted(
    OUTPUT_DIR.iterdir()
):
    print(" -", output_path.name)

In [ ]:
# ============================================================
# WLS WAVE-HELD-OUT VALIDATION
#
# Direction 1:
#   Train controls = WLS 2003
#   Test controls  = WLS 2011
#
# Direction 2:
#   Train controls = WLS 2011
#   Test controls  = WLS 2003
#
# Each external test set also contains held-out Pitt AD subjects,
# so accuracy, sensitivity, specificity, ROC-AUC, and MCC are valid.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    StratifiedKFold,
    train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    confusion_matrix
)


# ============================================================
# PATHS
# ============================================================

PITT_PATH = Path(
    "/content/drive/MyDrive/Fluency_Processed/"
    "Category_Timing_Final/"
    "pitt_AD_timing_one_per_subject.csv"
)

WLS_PATH = Path(
    "/content/drive/MyDrive/Fluency_Processed/"
    "WLS_Category_Filtered/"
    "wls_animal_fluency_all_waves.csv"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Fluency_Processed/"
    "Category_Wave_Held_Out"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# SETTINGS
# ============================================================

RANDOM_SEED = 42

N_REPETITIONS = 50

# Fraction of Pitt AD subjects held out for external testing.
AD_TEST_FRACTION = 0.25

# Threshold is selected only from training data.
N_INNER_FOLDS = 5

THRESHOLD_GRID = np.arange(
    0.20,
    0.81,
    0.01
)


# ============================================================
# FEATURE SETS
# ============================================================

FEATURE_SETS = {
    "Four features": [
        "clean_total_items",
        "clean_unique_items",
        "clean_repetition_count",
        "clean_type_token_ratio"
    ],

    "Total + unique": [
        "clean_total_items",
        "clean_unique_items"
    ]
}


# ============================================================
# LOAD PITT AD DATA
# ============================================================

if not PITT_PATH.exists():
    raise FileNotFoundError(
        f"Pitt file not found: {PITT_PATH}"
    )

pitt = pd.read_csv(
    PITT_PATH
)

required_pitt_columns = [
    "subject_id",
    "recording_id",
    "binary_label"
]

missing_pitt_columns = [
    column
    for column in required_pitt_columns
    if column not in pitt.columns
]

if missing_pitt_columns:
    raise ValueError(
        "Pitt file is missing columns: "
        + ", ".join(missing_pitt_columns)
    )

pitt = pitt[
    pd.to_numeric(
        pitt["binary_label"],
        errors="coerce"
    ) == 1
].copy()

pitt["binary_label"] = 1
pitt["dataset"] = "Pitt"
pitt["diagnosis_group"] = "AD"

pitt = pitt.drop_duplicates(
    subset=["subject_id"],
    keep="first"
).reset_index(drop=True)


# ============================================================
# LOAD FULL WLS ANIMAL POOL
# ============================================================

if not WLS_PATH.exists():
    raise FileNotFoundError(
        f"WLS file not found: {WLS_PATH}"
    )

wls = pd.read_csv(
    WLS_PATH
)

required_wls_columns = [
    "subject_id",
    "recording_id",
    "wave",
    "items_text",
    "total_items"
]

missing_wls_columns = [
    column
    for column in required_wls_columns
    if column not in wls.columns
]

if missing_wls_columns:
    raise ValueError(
        "WLS file is missing columns: "
        + ", ".join(missing_wls_columns)
    )


# ============================================================
# BUILD WLS FEATURES
# ============================================================

wls["wave"] = pd.to_numeric(
    wls["wave"],
    errors="coerce"
)

wls["clean_total_items"] = pd.to_numeric(
    wls["total_items"],
    errors="coerce"
)

wls["clean_unique_items"] = (
    wls["items_text"]
    .fillna("")
    .astype(str)
    .apply(
        lambda text: len(
            set(text.split())
        )
    )
)

wls["clean_repetition_count"] = (
    wls["clean_total_items"]
    - wls["clean_unique_items"]
)

wls["clean_type_token_ratio"] = np.where(
    wls["clean_total_items"] > 0,
    (
        wls["clean_unique_items"]
        / wls["clean_total_items"]
    ),
    np.nan
)

wls["binary_label"] = 0
wls["dataset"] = "WLS"
wls["diagnosis_group"] = "Control"

wls = wls[
    wls["wave"].isin([2003, 2011])
].copy()

wls = wls[
    wls["clean_total_items"] >= 3
].copy()


# ============================================================
# ONE ROW PER SUBJECT PER WAVE
# ============================================================

wls = (
    wls.sort_values(
        [
            "subject_id",
            "wave",
            "classification_confidence",
            "known_fraction"
        ],
        ascending=[
            True,
            True,
            False,
            False
        ]
    )
    .drop_duplicates(
        subset=[
            "subject_id",
            "wave"
        ],
        keep="first"
    )
    .reset_index(drop=True)
)


# ============================================================
# NUMERIC CONVERSION
# ============================================================

all_feature_columns = sorted(
    {
        feature
        for feature_list in FEATURE_SETS.values()
        for feature in feature_list
    }
)

for column in all_feature_columns:
    if column not in pitt.columns:
        raise ValueError(
            f"Pitt file is missing feature: {column}"
        )

    if column not in wls.columns:
        raise ValueError(
            f"WLS file is missing feature: {column}"
        )

    pitt[column] = pd.to_numeric(
        pitt[column],
        errors="coerce"
    )

    wls[column] = pd.to_numeric(
        wls[column],
        errors="coerce"
    )


# ============================================================
# SUBJECT-LEVEL WAVE SEPARATION
# ============================================================

wls_2003 = wls[
    wls["wave"] == 2003
].copy()

wls_2011 = wls[
    wls["wave"] == 2011
].copy()

print("=" * 100)
print("WAVE-HELD-OUT VALIDATION SETUP")
print("=" * 100)

print(
    "Pitt AD subjects:",
    pitt["subject_id"].nunique()
)

print(
    "WLS 2003 animal-control rows:",
    len(wls_2003)
)

print(
    "WLS 2003 unique subjects:",
    wls_2003["subject_id"].nunique()
)

print(
    "WLS 2011 animal-control rows:",
    len(wls_2011)
)

print(
    "WLS 2011 unique subjects:",
    wls_2011["subject_id"].nunique()
)

print(
    "Repetitions per direction:",
    N_REPETITIONS
)

print(
    "AD external-test fraction:",
    AD_TEST_FRACTION
)


# ============================================================
# MODEL
# ============================================================

def make_model():
    return Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "classifier",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=5000,
                    random_state=RANDOM_SEED
                )
            )
        ]
    )


# ============================================================
# METRICS
# ============================================================

def calculate_metrics(
    true_labels,
    probabilities,
    threshold
):
    true_labels = np.asarray(
        true_labels,
        dtype=int
    )

    probabilities = np.asarray(
        probabilities,
        dtype=float
    )

    predictions = (
        probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        true_labels,
        predictions,
        labels=[0, 1]
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    return {
        "accuracy": accuracy_score(
            true_labels,
            predictions
        ),

        "balanced_accuracy": balanced_accuracy_score(
            true_labels,
            predictions
        ),

        "precision": precision_score(
            true_labels,
            predictions,
            zero_division=0
        ),

        "recall": recall_score(
            true_labels,
            predictions,
            zero_division=0
        ),

        "specificity": specificity,

        "f1": f1_score(
            true_labels,
            predictions,
            zero_division=0
        ),

        "roc_auc": roc_auc_score(
            true_labels,
            probabilities
        ),

        "average_precision": average_precision_score(
            true_labels,
            probabilities
        ),

        "mcc": matthews_corrcoef(
            true_labels,
            predictions
        ),

        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp)
    }


# ============================================================
# TRAINING-ONLY THRESHOLD SELECTION
# ============================================================

def choose_threshold(
    X_train,
    y_train,
    random_state
):
    X_train = X_train.reset_index(
        drop=True
    )

    y_train = y_train.reset_index(
        drop=True
    )

    inner_cv = StratifiedKFold(
        n_splits=N_INNER_FOLDS,
        shuffle=True,
        random_state=random_state
    )

    inner_probabilities = np.zeros(
        len(y_train),
        dtype=float
    )

    for (
        inner_train_indices,
        inner_validation_indices
    ) in inner_cv.split(
        X_train,
        y_train
    ):
        model = make_model()

        model.fit(
            X_train.iloc[
                inner_train_indices
            ],
            y_train.iloc[
                inner_train_indices
            ]
        )

        inner_probabilities[
            inner_validation_indices
        ] = model.predict_proba(
            X_train.iloc[
                inner_validation_indices
            ]
        )[:, 1]

    threshold_rows = []

    for threshold in THRESHOLD_GRID:
        metrics = calculate_metrics(
            y_train,
            inner_probabilities,
            threshold
        )

        threshold_rows.append({
            "threshold": threshold,
            "balanced_accuracy": metrics[
                "balanced_accuracy"
            ],
            "mcc": metrics["mcc"],
            "f1": metrics["f1"],
            "recall": metrics["recall"],
            "specificity": metrics[
                "specificity"
            ]
        })

    threshold_table = pd.DataFrame(
        threshold_rows
    )

    threshold_table[
        "selection_score"
    ] = (
        threshold_table[
            "balanced_accuracy"
        ]
        + threshold_table["mcc"]
    )

    best_row = threshold_table.sort_values(
        [
            "selection_score",
            "balanced_accuracy",
            "mcc",
            "threshold"
        ],
        ascending=[
            False,
            False,
            False,
            True
        ]
    ).iloc[0]

    return float(
        best_row["threshold"]
    )


# ============================================================
# SAMPLE CONTROLS
# ============================================================

def sample_controls(
    control_pool,
    number_needed,
    random_state
):
    if len(control_pool) < number_needed:
        raise RuntimeError(
            "Control pool is too small. "
            f"Needed {number_needed}, found {len(control_pool)}."
        )

    return control_pool.sample(
        n=number_needed,
        replace=False,
        random_state=random_state
    ).copy()


# ============================================================
# RUN ONE WAVE-HELD-OUT EXPERIMENT
# ============================================================

def run_one_experiment(
    train_control_pool,
    test_control_pool,
    train_wave,
    test_wave,
    feature_set_name,
    feature_columns,
    repetition
):
    split_seed = (
        RANDOM_SEED
        + repetition
        + train_wave
        + test_wave
    )

    pitt_train, pitt_test = train_test_split(
        pitt,
        test_size=AD_TEST_FRACTION,
        random_state=split_seed,
        shuffle=True
    )

    pitt_train = pitt_train.reset_index(
        drop=True
    )

    pitt_test = pitt_test.reset_index(
        drop=True
    )

    number_train_ad = len(
        pitt_train
    )

    number_test_ad = len(
        pitt_test
    )

    train_controls = sample_controls(
        train_control_pool,
        number_train_ad,
        random_state=(
            split_seed + 1000
        )
    )

    test_controls = sample_controls(
        test_control_pool,
        number_test_ad,
        random_state=(
            split_seed + 2000
        )
    )

    training_data = pd.concat(
        [
            pitt_train,
            train_controls
        ],
        ignore_index=True
    )

    test_data = pd.concat(
        [
            pitt_test,
            test_controls
        ],
        ignore_index=True
    )

    training_data = training_data.sample(
        frac=1,
        random_state=(
            split_seed + 3000
        )
    ).reset_index(drop=True)

    test_data = test_data.sample(
        frac=1,
        random_state=(
            split_seed + 4000
        )
    ).reset_index(drop=True)

    X_train = training_data[
        feature_columns
    ].copy()

    y_train = training_data[
        "binary_label"
    ].astype(int)

    X_test = test_data[
        feature_columns
    ].copy()

    y_test = test_data[
        "binary_label"
    ].astype(int)

    threshold = choose_threshold(
        X_train,
        y_train,
        random_state=(
            split_seed + 5000
        )
    )

    model = make_model()

    model.fit(
        X_train,
        y_train
    )

    test_probabilities = model.predict_proba(
        X_test
    )[:, 1]

    metrics = calculate_metrics(
        y_test,
        test_probabilities,
        threshold
    )

    return {
        "direction": (
            f"Train {train_wave} - Test {test_wave}"
        ),

        "train_wave": train_wave,
        "test_wave": test_wave,

        "feature_set": feature_set_name,

        "repetition": repetition,

        "split_seed": split_seed,

        "train_AD_subjects": number_train_ad,
        "train_control_subjects": len(
            train_controls
        ),

        "test_AD_subjects": number_test_ad,
        "test_control_subjects": len(
            test_controls
        ),

        "threshold": threshold,

        **metrics
    }


# ============================================================
# RUN BOTH DIRECTIONS
# ============================================================

directions = [
    {
        "train_wave": 2003,
        "test_wave": 2011,
        "train_control_pool": wls_2003,
        "test_control_pool": wls_2011
    },

    {
        "train_wave": 2011,
        "test_wave": 2003,
        "train_control_pool": wls_2011,
        "test_control_pool": wls_2003
    }
]

all_results = []

total_experiments = (
    len(directions)
    * len(FEATURE_SETS)
    * N_REPETITIONS
)

completed = 0

for direction in directions:
    for (
        feature_set_name,
        feature_columns
    ) in FEATURE_SETS.items():

        print()
        print("=" * 100)
        print(
            f"TRAIN WLS {direction['train_wave']} "
            f"-> TEST WLS {direction['test_wave']}"
        )
        print(
            "FEATURE SET:",
            feature_set_name
        )
        print("=" * 100)

        for repetition in range(
            1,
            N_REPETITIONS + 1
        ):
            result = run_one_experiment(
                train_control_pool=direction[
                    "train_control_pool"
                ],

                test_control_pool=direction[
                    "test_control_pool"
                ],

                train_wave=direction[
                    "train_wave"
                ],

                test_wave=direction[
                    "test_wave"
                ],

                feature_set_name=feature_set_name,

                feature_columns=feature_columns,

                repetition=repetition
            )

            all_results.append(
                result
            )

            completed += 1

            if (
                repetition == 1
                or repetition % 10 == 0
                or repetition == N_REPETITIONS
            ):
                print(
                    f"Completed repetition "
                    f"{repetition}/{N_REPETITIONS} | "
                    f"accuracy="
                    f"{result['accuracy']:.4f} | "
                    f"ROC-AUC="
                    f"{result['roc_auc']:.4f} | "
                    f"threshold="
                    f"{result['threshold']:.2f}"
                )


results = pd.DataFrame(
    all_results
)


# ============================================================
# SUMMARY FUNCTION
# ============================================================

METRIC_COLUMNS = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "roc_auc",
    "average_precision",
    "mcc",
    "threshold"
]


summary_rows = []

for (
    direction,
    feature_set
), group in results.groupby(
    [
        "direction",
        "feature_set"
    ]
):
    for metric in METRIC_COLUMNS:
        values = group[
            metric
        ].dropna().to_numpy()

        summary_rows.append({
            "direction": direction,
            "feature_set": feature_set,
            "metric": metric,

            "mean": np.mean(
                values
            ),

            "standard_deviation": np.std(
                values,
                ddof=1
            ),

            "minimum": np.min(
                values
            ),

            "percentile_2_5": np.percentile(
                values,
                2.5
            ),

            "median": np.median(
                values
            ),

            "percentile_97_5": np.percentile(
                values,
                97.5
            ),

            "maximum": np.max(
                values
            )
        })


summary = pd.DataFrame(
    summary_rows
)


# ============================================================
# COMPACT SUMMARY
# ============================================================

compact_summary = (
    results.groupby(
        [
            "direction",
            "feature_set"
        ]
    )
    .agg(
        repetitions=(
            "repetition",
            "count"
        ),

        accuracy_mean=(
            "accuracy",
            "mean"
        ),

        accuracy_std=(
            "accuracy",
            "std"
        ),

        balanced_accuracy_mean=(
            "balanced_accuracy",
            "mean"
        ),

        precision_mean=(
            "precision",
            "mean"
        ),

        recall_mean=(
            "recall",
            "mean"
        ),

        specificity_mean=(
            "specificity",
            "mean"
        ),

        f1_mean=(
            "f1",
            "mean"
        ),

        roc_auc_mean=(
            "roc_auc",
            "mean"
        ),

        roc_auc_std=(
            "roc_auc",
            "std"
        ),

        average_precision_mean=(
            "average_precision",
            "mean"
        ),

        mcc_mean=(
            "mcc",
            "mean"
        ),

        threshold_median=(
            "threshold",
            "median"
        )
    )
    .reset_index()
)


# ============================================================
# OVERALL SUMMARY ACROSS BOTH DIRECTIONS
# ============================================================

overall_summary = (
    results.groupby(
        "feature_set"
    )
    .agg(
        experiments=(
            "repetition",
            "count"
        ),

        accuracy_mean=(
            "accuracy",
            "mean"
        ),

        accuracy_std=(
            "accuracy",
            "std"
        ),

        balanced_accuracy_mean=(
            "balanced_accuracy",
            "mean"
        ),

        precision_mean=(
            "precision",
            "mean"
        ),

        recall_mean=(
            "recall",
            "mean"
        ),

        specificity_mean=(
            "specificity",
            "mean"
        ),

        f1_mean=(
            "f1",
            "mean"
        ),

        roc_auc_mean=(
            "roc_auc",
            "mean"
        ),

        roc_auc_std=(
            "roc_auc",
            "std"
        ),

        average_precision_mean=(
            "average_precision",
            "mean"
        ),

        mcc_mean=(
            "mcc",
            "mean"
        ),

        threshold_median=(
            "threshold",
            "median"
        )
    )
    .reset_index()
)


# ============================================================
# STABILITY COUNTS
# ============================================================

stability_rows = []

for (
    direction,
    feature_set
), group in results.groupby(
    [
        "direction",
        "feature_set"
    ]
):
    stability_rows.append({
        "direction": direction,
        "feature_set": feature_set,

        "accuracy_at_least_0_80": (
            group["accuracy"] >= 0.80
        ).mean(),

        "accuracy_at_least_0_85": (
            group["accuracy"] >= 0.85
        ).mean(),

        "accuracy_at_least_0_90": (
            group["accuracy"] >= 0.90
        ).mean(),

        "roc_auc_at_least_0_90": (
            group["roc_auc"] >= 0.90
        ).mean(),

        "roc_auc_at_least_0_95": (
            group["roc_auc"] >= 0.95
        ).mean()
    })


stability = pd.DataFrame(
    stability_rows
)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print()
print("=" * 100)
print("WAVE-HELD-OUT COMPACT SUMMARY")
print("=" * 100)

display(
    compact_summary
)

print()
print("=" * 100)
print("OVERALL SUMMARY ACROSS BOTH DIRECTIONS")
print("=" * 100)

display(
    overall_summary
)

print()
print("=" * 100)
print("STABILITY RESULTS")
print("=" * 100)

display(
    stability
)


# ============================================================
# SAVE TABLES
# ============================================================

results.to_csv(
    OUTPUT_DIR
    / "wave_held_out_all_experiments.csv",
    index=False
)

summary.to_csv(
    OUTPUT_DIR
    / "wave_held_out_full_summary.csv",
    index=False
)

compact_summary.to_csv(
    OUTPUT_DIR
    / "wave_held_out_compact_summary.csv",
    index=False
)

overall_summary.to_csv(
    OUTPUT_DIR
    / "wave_held_out_overall_summary.csv",
    index=False
)

stability.to_csv(
    OUTPUT_DIR
    / "wave_held_out_stability.csv",
    index=False
)


# ============================================================
# PLOTS
# ============================================================

for metric in [
    "accuracy",
    "roc_auc",
    "mcc",
    "recall",
    "specificity"
]:
    for feature_set in FEATURE_SETS:
        plt.figure(
            figsize=(8, 5)
        )

        for direction_name, group in results[
            results["feature_set"]
            == feature_set
        ].groupby(
            "direction"
        ):
            plt.hist(
                group[metric].dropna(),
                bins=15,
                alpha=0.5,
                label=direction_name
            )

        plt.xlabel(
            metric
        )

        plt.ylabel(
            "Number of experiments"
        )

        plt.title(
            f"Wave-Held-Out {metric}\n"
            f"{feature_set}"
        )

        plt.legend()

        plt.tight_layout()

        safe_feature_name = (
            feature_set.lower()
            .replace(" ", "_")
            .replace("+", "plus")
        )

        plt.savefig(
            OUTPUT_DIR
            / (
                f"{safe_feature_name}_"
                f"{metric}_distribution.png"
            ),
            dpi=200,
            bbox_inches="tight"
        )

        plt.show()
        plt.close()


# ============================================================
# FINAL MESSAGE
# ============================================================

print()
print("=" * 100)
print("WAVE-HELD-OUT VALIDATION COMPLETE")
print("=" * 100)

print(
    "Total experiments:",
    len(results)
)

print(
    "Output folder:"
)

print(
    OUTPUT_DIR
)

print()
print(
    "Most important output:"
)

print(
    OUTPUT_DIR
    / "wave_held_out_compact_summary.csv"
)

print()
print(
    "Also inspect:"
)

print(
    OUTPUT_DIR
    / "wave_held_out_overall_summary.csv"
)

print(
    OUTPUT_DIR
    / "wave_held_out_stability.csv"
)

In [ ]:
# ============================================================
# CONSOLIDATE ENTIRE FLUENCY PROJECT IN GOOGLE DRIVE
# ============================================================

from pathlib import Path
import shutil
import json
import hashlib
from datetime import datetime

import pandas as pd


# ============================================================
# PATHS
# ============================================================

SOURCE_ROOT = Path(
    "/content/drive/MyDrive/Fluency_Processed"
)

FINAL_ROOT = Path(
    "/content/drive/MyDrive/"
    "Alzheimers_Project/Fluency_Category_Final"
)

BACKUP_ZIP_BASE = Path(
    "/content/drive/MyDrive/"
    "Alzheimers_Project/Fluency_Category_Final_Backup"
)

FINAL_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# IMPORTANT SOURCE FOLDERS
# ============================================================

folders_to_copy = [
    "Audit",
    "WLS_Category_Filtered",
    "Category_Final",
    "Category_Model_Baseline",
    "Category_Timing_Final",
    "Category_Timing_Model",
    "Category_Final_Model",
    "Category_Control_Resampling",
    "Category_Wave_Held_Out",
]

files_to_copy = [
    "fluency_all.csv",
    "category_fluency_all.csv",
    "letter_fluency_all.csv",
    "dataset_summary.csv",
    "integrity_report.csv",
]


# ============================================================
# COPY FOLDERS
# ============================================================

print("=" * 100)
print("COPYING FLUENCY PROJECT")
print("=" * 100)

for folder_name in folders_to_copy:
    source = SOURCE_ROOT / folder_name
    destination = FINAL_ROOT / folder_name

    if not source.exists():
        print(f"Missing folder, skipped: {source}")
        continue

    if destination.exists():
        shutil.rmtree(destination)

    shutil.copytree(
        source,
        destination
    )

    print(f"Copied folder: {folder_name}")


# ============================================================
# COPY ROOT FILES
# ============================================================

for filename in files_to_copy:
    source = SOURCE_ROOT / filename
    destination = FINAL_ROOT / filename

    if not source.exists():
        print(f"Missing file, skipped: {source}")
        continue

    shutil.copy2(
        source,
        destination
    )

    print(f"Copied file: {filename}")


# ============================================================
# CREATE PROJECT README
# ============================================================

readme_text = """
ALZHEIMER'S CATEGORY-FLUENCY MODEL
=================================

Project task
------------
Participants name as many animals as possible in approximately one minute.

Datasets
--------
Pitt:
- Alzheimer's disease, MCI, controls, and other/unknown diagnoses
- CHAT fluency transcripts
- Animal-fluency responses extracted from @G: animals
- One selected recording per subject for the final AD model

WLS:
- Healthy controls
- CogCat 2003 and CogCat 2011 spreadsheets
- Animal and food prompts were mixed in the original files
- High-confidence animal rows were separated from food and uncertain rows
- One selected animal-fluency recording per control subject

Final model
-----------
Model:
- Logistic regression
- Median imputation
- Standard scaling
- Balanced class weighting

Features:
1. clean_total_items
2. clean_unique_items
3. clean_repetition_count
4. clean_type_token_ratio

Deployment threshold:
- 0.52 in the original final nested evaluation
- Thresholds should be recalibrated for any new external population

Final nested out-of-fold result
-------------------------------
Subjects:
- 113 Pitt AD
- 113 WLS controls
- 226 total subjects

Accuracy: 0.8982
Balanced accuracy: 0.8982
Precision: 0.9167
Sensitivity: 0.8761
Specificity: 0.9204
F1: 0.8959
ROC-AUC: 0.9619
Average precision: 0.9672
MCC: 0.7972

Confusion matrix:
- TN: 104
- FP: 9
- FN: 14
- TP: 99

Bootstrap 95% confidence intervals
----------------------------------
Accuracy: 0.8584 to 0.9381
Balanced accuracy: 0.8582 to 0.9375
Precision: 0.8609 to 0.9646
Sensitivity: 0.8108 to 0.9333
Specificity: 0.8661 to 0.9658
F1: 0.8505 to 0.9358
ROC-AUC: 0.9368 to 0.9821
Average precision: 0.9453 to 0.9844
MCC: 0.7168 to 0.8753

Control-resampling stability
----------------------------
100 different WLS control samples:

Mean accuracy: 0.8904
Accuracy standard deviation: 0.0169
Mean ROC-AUC: 0.9612
ROC-AUC standard deviation: 0.0057
Mean MCC: 0.7818

97% of runs reached at least 0.85 accuracy.
100% of runs reached at least 0.90 ROC-AUC.
98% of runs reached at least 0.95 ROC-AUC.

Wave-held-out validation
------------------------
Four-feature model across both directions:

Mean accuracy: 0.8933
Mean balanced accuracy: 0.8933
Mean F1: 0.8923
Mean ROC-AUC: 0.9643
Mean MCC: 0.7913

Train WLS 2003, test WLS 2011:
- Accuracy: 0.8938
- ROC-AUC: 0.9620

Train WLS 2011, test WLS 2003:
- Accuracy: 0.8928
- ROC-AUC: 0.9667

Timing result
-------------
Timing features did not improve performance.

Best model:
- Four basic count features

Timing-only models:
- Approximately 0.77 to 0.79 accuracy

Important limitation
--------------------
All Alzheimer's subjects are from Pitt and almost all controls are from WLS.
Therefore, disease label and corpus source remain confounded.

This model is:
- A robust cross-corpus research classifier
- Not a clinically validated diagnostic test
- Not suitable for telling a user that they have Alzheimer's disease
- In need of validation on AD and control subjects collected under the same protocol

Recommended use
---------------
Use the category-fluency model as one component in a multi-task research system,
alongside Cookie Theft, recall, sentence construction, and possibly acoustic models.
"""

readme_path = FINAL_ROOT / "README.txt"

readme_path.write_text(
    readme_text.strip() + "\n",
    encoding="utf-8"
)

print("Created README.txt")


# ============================================================
# CREATE MACHINE-READABLE PROJECT METADATA
# ============================================================

metadata = {
    "project_name": "Alzheimer's Category Fluency Classifier",
    "created_at": datetime.now().isoformat(),
    "task": "Animal category verbal fluency",
    "model_type": "Logistic Regression",
    "features": [
        "clean_total_items",
        "clean_unique_items",
        "clean_repetition_count",
        "clean_type_token_ratio",
    ],
    "deployment_threshold": 0.52,
    "subjects": {
        "pitt_ad": 113,
        "wls_control": 113,
        "total": 226,
    },
    "nested_oof_metrics": {
        "accuracy": 0.8982,
        "balanced_accuracy": 0.8982,
        "precision": 0.9167,
        "recall": 0.8761,
        "specificity": 0.9204,
        "f1": 0.8959,
        "roc_auc": 0.9619,
        "average_precision": 0.9672,
        "mcc": 0.7972,
        "tn": 104,
        "fp": 9,
        "fn": 14,
        "tp": 99,
    },
    "control_resampling": {
        "runs": 100,
        "mean_accuracy": 0.890398,
        "accuracy_sd": 0.016921,
        "mean_roc_auc": 0.961233,
        "roc_auc_sd": 0.005738,
        "mean_mcc": 0.781771,
    },
    "wave_held_out": {
        "experiments": 100,
        "mean_accuracy": 0.893276,
        "mean_roc_auc": 0.96434,
        "mean_f1": 0.892286,
        "mean_mcc": 0.791329,
    },
    "limitation": (
        "AD samples are from Pitt and controls are primarily from WLS. "
        "Corpus source and diagnosis remain confounded."
    ),
}

metadata_path = FINAL_ROOT / "project_metadata.json"

metadata_path.write_text(
    json.dumps(
        metadata,
        indent=2
    ),
    encoding="utf-8"
)

print("Created project_metadata.json")


# ============================================================
# CREATE DATA DICTIONARY
# ============================================================

data_dictionary = pd.DataFrame([
    {
        "column": "subject_id",
        "description": "Dataset-prefixed participant identifier",
    },
    {
        "column": "recording_id",
        "description": "Unique recording/task identifier",
    },
    {
        "column": "dataset",
        "description": "Source corpus, Pitt or WLS",
    },
    {
        "column": "diagnosis_group",
        "description": "AD, Control, MCI, or Other_or_Unknown",
    },
    {
        "column": "binary_label",
        "description": "0 for control and 1 for Alzheimer's disease",
    },
    {
        "column": "clean_total_items",
        "description": "Total valid animal responses including repetitions",
    },
    {
        "column": "clean_unique_items",
        "description": "Number of distinct valid animal responses",
    },
    {
        "column": "clean_repetition_count",
        "description": "Total responses minus distinct responses",
    },
    {
        "column": "clean_type_token_ratio",
        "description": "Unique animal responses divided by total responses",
    },
    {
        "column": "predicted_probability_AD",
        "description": "Out-of-fold predicted probability for the AD class",
    },
    {
        "column": "predicted_label",
        "description": "Predicted class after applying the selected threshold",
    },
])

data_dictionary.to_csv(
    FINAL_ROOT / "data_dictionary.csv",
    index=False
)

print("Created data_dictionary.csv")


# ============================================================
# CREATE FILE MANIFEST WITH SHA-256 CHECKSUMS
# ============================================================

def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as file_handle:
        while True:
            chunk = file_handle.read(
                1024 * 1024
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


manifest_rows = []

for path in sorted(
    FINAL_ROOT.rglob("*")
):
    if not path.is_file():
        continue

    relative_path = path.relative_to(
        FINAL_ROOT
    )

    manifest_rows.append({
        "relative_path": str(relative_path),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    })

manifest = pd.DataFrame(
    manifest_rows
)

manifest.to_csv(
    FINAL_ROOT / "file_manifest.csv",
    index=False
)

print(
    "Created file_manifest.csv with",
    len(manifest),
    "files"
)


# ============================================================
# VERIFY CRITICAL FILES
# ============================================================

critical_files = [
    FINAL_ROOT
    / "Category_Final_Model"
    / "category_fluency_final_model.joblib",

    FINAL_ROOT
    / "Category_Final_Model"
    / "final_metrics.csv",

    FINAL_ROOT
    / "Category_Final_Model"
    / "bootstrap_confidence_intervals.csv",

    FINAL_ROOT
    / "Category_Final_Model"
    / "nested_out_of_fold_predictions.csv",

    FINAL_ROOT
    / "Category_Control_Resampling"
    / "control_resampling_summary.csv",

    FINAL_ROOT
    / "Category_Wave_Held_Out"
    / "wave_held_out_compact_summary.csv",

    FINAL_ROOT
    / "Category_Final"
    / "category_final_balanced.csv",

    FINAL_ROOT
    / "Category_Timing_Final"
    / "category_timing_balanced.csv",
]

missing_critical_files = [
    str(path)
    for path in critical_files
    if not path.exists()
]

if missing_critical_files:
    print("\nWARNING: Missing critical files:")

    for path in missing_critical_files:
        print(" -", path)
else:
    print(
        "All critical project files are present."
    )


# ============================================================
# CREATE ZIP BACKUP
# ============================================================

zip_path = shutil.make_archive(
    str(BACKUP_ZIP_BASE),
    "zip",
    root_dir=FINAL_ROOT.parent,
    base_dir=FINAL_ROOT.name
)

print("Created ZIP backup:")
print(zip_path)


# ============================================================
# FINAL SUMMARY
# ============================================================

total_files = sum(
    1
    for path in FINAL_ROOT.rglob("*")
    if path.is_file()
)

total_bytes = sum(
    path.stat().st_size
    for path in FINAL_ROOT.rglob("*")
    if path.is_file()
)

print()
print("=" * 100)
print("FLUENCY PROJECT SAVED")
print("=" * 100)

print("Final project folder:")
print(FINAL_ROOT)

print()
print("ZIP backup:")
print(zip_path)

print()
print("Files saved:", total_files)
print(
    "Total size:",
    f"{total_bytes / (1024 ** 2):.2f} MB"
)

print()
print("Main model:")
print(
    FINAL_ROOT
    / "Category_Final_Model"
    / "category_fluency_final_model.joblib"
)

print()
print("Main README:")
print(
    FINAL_ROOT
    / "README.txt"
)